In [2]:
# %% [markdown]
# CAMELOT-IDS v2 — PC-optimized full notebook/script (resume-safe)
# Target machine:
# - 64 GB RAM
# - RTX 3060 12 GB
# - Dual Xeon E5-2683 v4
#
# Added without changing model functionality:
# - preprocessing cache
# - epoch checkpoints + resume
# - cached test/cal logits
# - cached calibration/RAPS/stage outputs
# - cached stream inputs/results
#
# Notes:
# - Resumes from the last completed epoch, not mid-batch
# - Raw model/training/calibration/drift/export logic is unchanged

# %% [markdown]
# Cell 0 — Optional installs
# !pip -q install -U pyarrow river onnx onnxruntime joblib tqdm xgboost catboost

# %%
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import re
import gc
import json
import time
import math
import random
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional, Dict, Any, List, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    log_loss,
    balanced_accuracy_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import joblib

warnings.filterwarnings("ignore")


# %%
def set_seed(seed: int = 42, deterministic: bool = False):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = bool(deterministic)
        torch.backends.cudnn.benchmark = not bool(deterministic)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


@dataclass
class ExpConfig:
    # ---------------------
    # Paths
    # ---------------------
    DATA_ROOT: str = r"C:\Users\HaseebWajid\Desktop\Ayesha Code\CICIoT2023"
    OUT_DIR: str = "./results_camelot_ids_v2_pc"

    # ---------------------
    # Runtime / reproducibility
    # ---------------------
    SEED: int = 42
    DETERMINISTIC: bool = False          # True for paper reruns, False for faster runs
    USE_TORCH_COMPILE: bool = False      # safe fallback if compile fails
    TF32_MODE: bool = True               # good for Ampere (RTX 3060)

    # ---------------------
    # Resume / checkpointing
    # ---------------------
    RESUME: bool = True
    CACHE_PREPROCESSED: bool = True
    CACHE_LOGITS: bool = True
    CACHE_STREAM_INPUTS: bool = True
    SAVE_LAST_EVERY_EPOCH: bool = True

    # ---------------------
    # Data loading controls
    # ---------------------
    MAX_FILES: Optional[int] = 100       # start here on 64 GB RAM; use None after a successful run
    SAMPLE_FRAC: Optional[float] = None  # e.g. 0.05 for quick debug
    READ_ENGINE: str = "pyarrow"        # pyarrow | c | python | auto
    LOW_MEMORY_READ: bool = False

    LABEL_COL_CANDIDATES: Tuple[str, ...] = ("label", "Label", "Attack", "attack", "Class", "class", "category", "Category")
    TIME_COL_CANDIDATES:  Tuple[str, ...] = ("ts", "TS", "timestamp", "Timestamp", "time", "Time", "Datetime", "datetime")
    GROUP_COL_CANDIDATES: Tuple[str, ...] = ("Device", "device", "DeviceName", "device_name", "MAC", "mac")

    # Splits
    TRAIN_PCT: float = 0.80
    VAL_PCT: float   = 0.10
    CAL_PCT: float   = 0.05
    TEST_PCT: float  = 0.05
    SPLIT_POLICY: str = "file_holdout"  # auto | time | group | file_holdout | stratified
    N_GROUP_SPLITS: int = 20

    # Leakage-safe dropping
    DROP_FEATURES_NORM: Tuple[str, ...] = (
        "flowid", "timestamp", "ts", "srcip", "dstip", "sourceip", "destinationip",
        "srcport", "dstport", "device", "devicename", "mac",
        "label", "attack", "class", "category",
    )

    CLIP_INF: float = 1e9
    POST_SCALE_CLIP: float = 20.0

    # ---------------------
    # Training
    # ---------------------
    BATCH_SIZE: int = 1024
    EPOCHS: int = 70
    WARMUP_EPOCHS: float = 2.0
    LR: float = 1e-3
    WEIGHT_DECAY: float = 8e-3
    GRAD_CLIP: float = 1.0
    USE_AMP: bool = True
    NUM_WORKERS: int = 0
    PIN_MEMORY: bool = True
    PREFETCH_FACTOR: int = 2

    # ---------------------
    # Model
    # ---------------------
    D_MODEL: int = 256
    NHEAD: int = 8
    LOCAL_LAYERS: int = 2
    GLOBAL_LAYERS: int = 6
    FF_DIM: int = 1024
    DROPOUT: float = 0.10
    ATTN_DROPOUT: float = 0.05

    # Hierarchical multi-task
    USE_MULTITASK: bool = True
    W_COARSE: float = 0.30
    W_BIN: float = 0.20
    W_CONS_COARSE: float = 0.10
    W_CONS_BIN: float = 0.05

    # ---------------------
    # Imbalance: LDAM-DRW
    # ---------------------
    FINE_LOSS_MODE: str = "ldam_drw"    # ldam_drw | logit_adj | balanced_softmax
    DRW_EPOCHS: int = 3
    CB_BETA: float = 0.9999
    LDAM_MAX_M: float = 0.5
    LDAM_S: float = 30.0
    LOGIT_ADJ_TAU: float = 0.5

    # ---------------------
    # Calibration + Conformal
    # ---------------------
    TEMPERATURE_SCALE: bool = True
    N_BINS_ECE: int = 15
    ALPHAS: Tuple[float, ...] = (0.01, 0.05, 0.10)

    RAPS_KREG: int = 3
    RAPS_LAMBDA: float = 0.01
    RAPS_MONDRIAN: str = "pred_coarse"  # global | pred_coarse

    # ---------------------
    # Drift + Adaptation
    # ---------------------
    DO_STREAM_EVAL: bool = True
    STREAM_BATCH: int = 50_000           # logical stream chunk on CPU
    STREAM_MICRO_BATCH: int = 16_384     # actual GPU chunk to avoid OOM

    DRIFT_DETECTOR: str = "ADWIN+MART"  # ADWIN | KSWIN | MART | ADWIN+MART | KSWIN+MART
    DRIFT_DELTA: float = 0.002
    DRIFT_SIGNAL: str = "entropy"       # entropy | 1-maxprob | set_size
    MART_EPS: float = 0.5
    MART_THRESHOLD: float = 25.0

    ADAPT_METHOD: str = "TENT+HEAD"     # NONE | TENT | TENT+HEAD
    ADAPT_STEPS: int = 5
    ADAPT_LR: float = 1e-4
    ADAPT_BUFFER_BATCHES: int = 3

    ADAPT_ALPHA: float = 0.10
    HEAD_TUNE_LR: float = 5e-5
    HEAD_TUNE_STEPS: int = 10
    MIN_PSEUDO: int = 512

    # ---------------------
    # Baselines (optional, slow)
    # ---------------------
    RUN_BASELINES: bool = False
    BASELINE_MAX_TRAIN: int = 800_000
    BASELINE_MAX_TEST: int = 300_000

    # ---------------------
    # Export
    # ---------------------
    EXPORT_ONNX: bool = True
    EXPORT_TORCHSCRIPT: bool = True
    EXPORT_INT8_DYNAMIC: bool = True


cfg = ExpConfig()
Path(cfg.OUT_DIR).mkdir(parents=True, exist_ok=True)

RUN_DIR = Path(cfg.OUT_DIR)
CACHE_DIR = RUN_DIR / "cache"
ARRAY_CACHE_DIR = CACHE_DIR / "arrays"
LOGITS_CACHE_DIR = CACHE_DIR / "logits"
STREAM_CACHE_DIR = CACHE_DIR / "stream"
CKPT_DIR = RUN_DIR / "checkpoints"
STAGE_DIR = RUN_DIR / "stages"
export_dir = RUN_DIR / "export"

for d in [CACHE_DIR, ARRAY_CACHE_DIR, LOGITS_CACHE_DIR, STREAM_CACHE_DIR, CKPT_DIR, STAGE_DIR, export_dir]:
    d.mkdir(parents=True, exist_ok=True)

set_seed(cfg.SEED, deterministic=cfg.DETERMINISTIC)

if DEVICE == "cuda" and cfg.TF32_MODE:
    try:
        torch.backends.cuda.matmul.fp32_precision = "tf32"
        torch.backends.cudnn.fp32_precision = "tf32"
    except Exception:
        torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

with open(RUN_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(asdict(cfg), f, indent=2)

print("OUT_DIR:", cfg.OUT_DIR)


# %%
def save_json_atomic(path: Path, obj: Dict[str, Any]):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


def load_json(path: Path, default=None):
    path = Path(path)
    if not path.exists():
        return {} if default is None else default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def atomic_torch_save(obj: Any, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def atomic_save_npy(path: Path, arr: np.ndarray):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "wb") as f:
        np.save(f, arr, allow_pickle=False)
    os.replace(tmp, path)


def save_array_dict(folder: Path, arrays: Dict[str, np.ndarray]):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    for k, v in arrays.items():
        atomic_save_npy(folder / f"{k}.npy", v)
    save_json_atomic(folder / "_ok.json", {"keys": list(arrays.keys())})


def load_array_dict(folder: Path, keys: Optional[List[str]] = None) -> Dict[str, np.ndarray]:
    folder = Path(folder)
    meta = load_json(folder / "_ok.json", {})
    if keys is None:
        keys = meta.get("keys", [])
    return {k: np.load(folder / f"{k}.npy") for k in keys}


def folder_ready(folder: Path, keys: List[str]) -> bool:
    folder = Path(folder)
    return (folder / "_ok.json").exists() and all((folder / f"{k}.npy").exists() for k in keys)


def get_rng_state() -> Dict[str, Any]:
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["torch_cuda"] = torch.cuda.get_rng_state_all()
    return state


def set_rng_state(state: Optional[Dict[str, Any]]):
    if not state:
        return

    if "python" in state:
        random.setstate(state["python"])

    if "numpy" in state:
        np.random.set_state(state["numpy"])

    if "torch" in state:
        cpu_state = state["torch"]

        if isinstance(cpu_state, np.ndarray):
            cpu_state = torch.from_numpy(cpu_state)
        elif not isinstance(cpu_state, torch.Tensor):
            cpu_state = torch.tensor(cpu_state)

        cpu_state = cpu_state.detach().to(device="cpu", dtype=torch.uint8)
        torch.set_rng_state(cpu_state)

    if torch.cuda.is_available() and "torch_cuda" in state:
        cuda_states = []
        for s in state["torch_cuda"]:
            if isinstance(s, np.ndarray):
                s = torch.from_numpy(s)
            elif not isinstance(s, torch.Tensor):
                s = torch.tensor(s)

            s = s.detach().to(device="cpu", dtype=torch.uint8)
            cuda_states.append(s)

        torch.cuda.set_rng_state_all(cuda_states)
        
def stable_json_dumps(obj: Any) -> str:
    return json.dumps(obj, sort_keys=True, default=str)


def same_signature(a: Optional[Dict[str, Any]], b: Optional[Dict[str, Any]]) -> bool:
    return stable_json_dumps(a or {}) == stable_json_dumps(b or {})


def get_training_signature() -> Dict[str, Any]:
    sig = {}
    done_path = STAGE_DIR / "training_done.json"
    best_path = CKPT_DIR / "best_model.pt"
    if done_path.exists():
        done = load_json(done_path, {})
        sig["best_epoch"] = done.get("best_epoch")
        sig["best_score"] = done.get("best_score")
    if best_path.exists():
        sig["best_ckpt_mtime_ns"] = best_path.stat().st_mtime_ns
        sig["best_ckpt_size"] = best_path.stat().st_size
    return sig


ARRAY_KEYS = [
    "X_train", "y_train_f", "y_train_c", "y_train_b",
    "X_val", "y_val_f", "y_val_c", "y_val_b",
    "X_cal", "y_cal_f", "y_cal_c", "y_cal_b",
    "X_test", "y_test_f", "y_test_c", "y_test_b",
]
STREAM_KEYS = ["X_stream_test", "y_stream_f"]
LOGIT_KEYS = ["fine", "coarse", "bin", "yf", "yc", "yb"]


# %%
def norm_colname(c: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(c).strip().lower())


def detect_col(columns: List[str], candidates: Tuple[str, ...]) -> Optional[str]:
    colset = set(columns)
    for c in candidates:
        if c in colset:
            return c
    norm_map = {norm_colname(c): c for c in columns}
    for c in candidates:
        nc = norm_colname(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def sanitize_frame(df: pd.DataFrame, clip_inf: float = 1e9) -> pd.DataFrame:
    df = df.replace({"Infinity": np.inf, "inf": np.inf, "+inf": np.inf, "-inf": -np.inf,
                     "NaN": np.nan, "nan": np.nan, "": np.nan})
    df = df.replace([np.inf, -np.inf], np.nan)
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        df[num_cols] = df[num_cols].clip(-clip_inf, clip_inf)
    return df


def downcast_numeric(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.select_dtypes(include=["float64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="float", errors="ignore")
    for c in df.select_dtypes(include=["int64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="integer", errors="ignore")
    return df


def memory_mb(df: pd.DataFrame) -> float:
    try:
        return float(df.memory_usage(deep=True).sum()) / (1024 ** 2)
    except Exception:
        return float("nan")


def discover_csv_files(root: Path, max_files: Optional[int] = None) -> List[Path]:
    files = sorted([p for p in root.rglob("*.csv") if p.is_file()])
    if not files:
        raise FileNotFoundError(f"No .csv files found under: {root}")
    if max_files is not None:
        files = files[:max_files]
    return files


def running_on_kaggle():
    return Path("/kaggle/input").exists()


def find_dataset_root(hint: Optional[str] = None) -> Path:
    base = Path("/kaggle/input")
    if hint is not None and (base / hint).exists():
        return base / hint
    best, best_n = None, -1
    for d in base.iterdir():
        if not d.is_dir():
            continue
        n = len(list(d.rglob("*.csv")))
        if n > best_n:
            best_n = n
            best = d
    if best is None:
        raise FileNotFoundError("No dataset found in /kaggle/input")
    return best


def read_one_csv(path: Path, cfg: ExpConfig) -> pd.DataFrame:
    engines = []
    if cfg.READ_ENGINE == "auto":
        engines = ["pyarrow", "c", "python"]
    else:
        engines = [cfg.READ_ENGINE, "c", "python"]

    last_err = None
    for eng in engines:
        try:
            kwargs = dict(low_memory=cfg.LOW_MEMORY_READ)
            if eng == "pyarrow":
                kwargs.pop("low_memory", None)
                return pd.read_csv(path, engine="pyarrow")
            if eng == "c":
                return pd.read_csv(path, engine="c", **kwargs)
            if eng == "python":
                kwargs.pop("low_memory", None)
                return pd.read_csv(path, engine="python")
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"Failed to read {path}: {last_err}")


def maybe_compress_categoricals(df: pd.DataFrame, max_unique_ratio: float = 0.50) -> pd.DataFrame:
    for c in df.select_dtypes(include=["object"]).columns:
        nunique = df[c].nunique(dropna=False)
        ratio = nunique / max(len(df), 1)
        if ratio <= max_unique_ratio:
            try:
                df[c] = df[c].astype("category")
            except Exception:
                pass
    return df


def sort_stream_df_simple(d: pd.DataFrame, time_col: Optional[str]) -> pd.DataFrame:
    d = d.copy()
    if time_col is not None and time_col in d.columns:
        d[time_col] = pd.to_datetime(d[time_col], errors="coerce")
        d = d.sort_values(time_col, kind="mergesort")
    else:
        d = d.sort_values(["__src_file"], kind="mergesort")
    return d


# %%
DICT_34_CLASSES: Dict[str, int] = {
    "BenignTraffic": 0,
    "DDoS-RSTFINFlood": 1, "DDoS-PSHACK_Flood": 2, "DDoS-SYN_Flood": 3, "DDoS-UDP_Flood": 4,
    "DDoS-TCP_Flood": 5, "DDoS-ICMP_Flood": 6, "DDoS-SynonymousIP_Flood": 7,
    "DDoS-ACK_Fragmentation": 8, "DDoS-UDP_Fragmentation": 9, "DDoS-ICMP_Fragmentation": 10,
    "DDoS-SlowLoris": 11, "DDoS-HTTP_Flood": 12,
    "DoS-UDP_Flood": 13, "DoS-SYN_Flood": 14, "DoS-TCP_Flood": 15, "DoS-HTTP_Flood": 16,
    "Mirai-greeth_flood": 17, "Mirai-greip_flood": 18, "Mirai-udpplain": 19,
    "Recon-PingSweep": 20, "Recon-OSScan": 21, "Recon-PortScan": 22, "VulnerabilityScan": 23, "Recon-HostDiscovery": 24,
    "DNS_Spoofing": 25, "MITM-ArpSpoofing": 26,
    "BrowserHijacking": 27, "Backdoor_Malware": 28, "XSS": 29, "Uploading_Attack": 30, "SqlInjection": 31, "CommandInjection": 32,
    "DictionaryBruteForce": 33
}

DICT_8_FROM_FINEID: Dict[int, int] = {
    0: 0,
    1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1, 11: 1, 12: 1,
    17: 2, 18: 2, 19: 2,
    20: 3, 21: 3, 22: 3, 23: 3, 24: 3,
    25: 4, 26: 4,
    27: 5, 28: 5, 29: 5, 30: 5, 31: 5, 32: 5,
    33: 6,
    13: 7, 14: 7, 15: 7, 16: 7
}

COARSE_NAMES = ["Benign", "DDoS", "Mirai", "Recon", "Spoofing", "Web-based", "Brute Force", "DoS"]
FINE_CLASS_NAMES = [k for k, v in sorted(DICT_34_CLASSES.items(), key=lambda kv: kv[1])]
NUM_FINE = len(FINE_CLASS_NAMES)
NUM_COARSE = len(COARSE_NAMES)

fine_to_coarse = np.zeros(NUM_FINE, dtype=np.int64)
for fine_name, fine_id in DICT_34_CLASSES.items():
    fine_to_coarse[fine_id] = DICT_8_FROM_FINEID[fine_id]

benign_fine_id = DICT_34_CLASSES["BenignTraffic"]
assert benign_fine_id == 0


# %%
def build_feature_columns(df: pd.DataFrame, cfg: ExpConfig, label_col: str, time_col: Optional[str], group_col: Optional[str]) -> List[str]:
    drop_norm = set(cfg.DROP_FEATURES_NORM)
    cols = []
    for c in df.columns:
        if c in [label_col, "y_fine", "y_coarse", "y_bin", "__src_file"]:
            continue
        if group_col is not None and c == group_col:
            continue
        if time_col is not None and c == time_col:
            continue
        if norm_colname(c) in drop_norm:
            continue
        cols.append(c)
    return cols


def stratified_4way_split(df: pd.DataFrame, y_col: str, cfg: ExpConfig) -> Dict[str, pd.DataFrame]:
    train_df, rem_df = train_test_split(
        df, test_size=(1 - cfg.TRAIN_PCT), stratify=df[y_col], random_state=cfg.SEED
    )
    rem_frac = 1 - cfg.TRAIN_PCT
    val_frac = cfg.VAL_PCT / rem_frac
    val_df, temp_df = train_test_split(
        rem_df, test_size=(1 - val_frac), stratify=rem_df[y_col], random_state=cfg.SEED
    )
    cal_ratio = cfg.CAL_PCT / (cfg.CAL_PCT + cfg.TEST_PCT)
    cal_df, test_df = train_test_split(
        temp_df, test_size=(1 - cal_ratio), stratify=temp_df[y_col], random_state=cfg.SEED
    )
    return {"train": train_df, "val": val_df, "cal": cal_df, "test": test_df}


def time_split(df: pd.DataFrame, time_col: str, cfg: ExpConfig) -> Dict[str, pd.DataFrame]:
    d = df.copy()
    d[time_col] = pd.to_datetime(d[time_col], errors="coerce")
    d = d.sort_values(time_col, kind="mergesort")
    n = len(d)
    n_train = int(cfg.TRAIN_PCT * n)
    n_val = int(cfg.VAL_PCT * n)
    n_cal = int(cfg.CAL_PCT * n)
    return {
        "train": d.iloc[:n_train].copy(),
        "val": d.iloc[n_train:n_train + n_val].copy(),
        "cal": d.iloc[n_train + n_val:n_train + n_val + n_cal].copy(),
        "test": d.iloc[n_train + n_val + n_cal:].copy(),
    }


def stratified_group_4way(df: pd.DataFrame, y_col: str, group_col: str, cfg: ExpConfig) -> Optional[Dict[str, pd.DataFrame]]:
    y = df[y_col].values
    groups = df[group_col].astype(str).fillna("NA").values
    n_groups = pd.Series(groups).nunique()
    n_splits = min(cfg.N_GROUP_SPLITS, n_groups)
    if n_splits < 4:
        return None

    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=cfg.SEED)
    fold = np.empty(len(df), dtype=np.int16)
    Xdummy = np.zeros((len(df), 1), dtype=np.int8)

    for i, (_, te) in enumerate(sgkf.split(Xdummy, y, groups)):
        fold[te] = i

    rng = np.random.default_rng(cfg.SEED)
    perm = rng.permutation(n_splits)

    n_test = max(1, int(round(cfg.TEST_PCT * n_splits)))
    n_cal = max(1, int(round(cfg.CAL_PCT * n_splits)))
    n_val = max(1, int(round(cfg.VAL_PCT * n_splits)))
    total = n_test + n_cal + n_val
    if total >= n_splits:
        return None

    test_f = set(perm[:n_test])
    cal_f = set(perm[n_test:n_test + n_cal])
    val_f = set(perm[n_test + n_cal:n_test + n_cal + n_val])
    train_f = set(perm[n_test + n_cal + n_val:])

    splits = {
        "train": df.loc[np.isin(fold, list(train_f))].copy(),
        "val": df.loc[np.isin(fold, list(val_f))].copy(),
        "cal": df.loc[np.isin(fold, list(cal_f))].copy(),
        "test": df.loc[np.isin(fold, list(test_f))].copy(),
    }
    return splits


def infer_numeric_object_cols(df: pd.DataFrame, threshold: float = 0.95) -> List[str]:
    obj_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    numeric_like = []
    for c in obj_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        ratio = float(s.notna().mean()) if len(s) else 0.0
        if ratio >= threshold:
            numeric_like.append(c)
    return numeric_like


def fit_preprocessors(train_df: pd.DataFrame, feature_cols: List[str], cfg: ExpConfig):
    X = sanitize_frame(train_df[feature_cols].copy(), cfg.CLIP_INF)
    numeric_like_obj = infer_numeric_object_cols(X, threshold=0.95)
    for c in numeric_like_obj:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    non_num = [c for c in X.columns if c not in num_cols]
    if non_num:
        print("Dropping non-numeric columns:", non_num[:10], "...")
        X = X[num_cols]

    num_imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    num_imputer.fit(X[num_cols])
    Ximp = num_imputer.transform(X[num_cols]).astype(np.float32)
    scaler.fit(Ximp)

    meta = {"num_cols": num_cols, "numeric_like_obj": numeric_like_obj}
    return num_imputer, scaler, meta


def transform_df(
    df_in: pd.DataFrame,
    feature_cols: List[str],
    num_imputer,
    scaler,
    meta,
    cfg: ExpConfig
) -> np.ndarray:
    X = sanitize_frame(df_in[feature_cols].copy(), cfg.CLIP_INF)
    for c in meta["numeric_like_obj"]:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")

    num_cols = meta["num_cols"]
    X = X[num_cols]
    Xn = num_imputer.transform(X).astype(np.float32)
    Xn = scaler.transform(Xn).astype(np.float32)
    Xn = np.clip(Xn, -cfg.POST_SCALE_CLIP, cfg.POST_SCALE_CLIP).astype(np.float32)
    Xn = np.nan_to_num(
        Xn,
        nan=0.0,
        posinf=cfg.POST_SCALE_CLIP,
        neginf=-cfg.POST_SCALE_CLIP
    ).astype(np.float32)
    return Xn


# %%
preprocess_cache_ready = (
    cfg.RESUME
    and cfg.CACHE_PREPROCESSED
    and folder_ready(ARRAY_CACHE_DIR, ARRAY_KEYS)
    and (export_dir / "preprocess_and_meta.joblib").exists()
    and (RUN_DIR / "split_mode.json").exists()
    and ((not cfg.DO_STREAM_EVAL) or folder_ready(STREAM_CACHE_DIR, STREAM_KEYS))
)

if preprocess_cache_ready:
    print("Loading cached preprocessed arrays...")

    arrs = load_array_dict(ARRAY_CACHE_DIR, ARRAY_KEYS)
    X_train = arrs["X_train"]
    y_train_f = arrs["y_train_f"]
    y_train_c = arrs["y_train_c"]
    y_train_b = arrs["y_train_b"]

    X_val = arrs["X_val"]
    y_val_f = arrs["y_val_f"]
    y_val_c = arrs["y_val_c"]
    y_val_b = arrs["y_val_b"]

    X_cal = arrs["X_cal"]
    y_cal_f = arrs["y_cal_f"]
    y_cal_c = arrs["y_cal_c"]
    y_cal_b = arrs["y_cal_b"]

    X_test = arrs["X_test"]
    y_test_f = arrs["y_test_f"]
    y_test_c = arrs["y_test_c"]
    y_test_b = arrs["y_test_b"]

    if cfg.DO_STREAM_EVAL:
        sarrs = load_array_dict(STREAM_CACHE_DIR, STREAM_KEYS)
        X_stream_test = sarrs["X_stream_test"]
        y_stream_f = sarrs["y_stream_f"]
    else:
        X_stream_test, y_stream_f = None, None

    meta_obj = joblib.load(export_dir / "preprocess_and_meta.joblib")
    FEATURE_COLS = meta_obj["FEATURE_COLS"]
    meta = meta_obj["meta"]
    num_imp = meta_obj["num_imputer"]
    scaler = meta_obj["scaler"]

    split_meta = load_json(RUN_DIR / "split_mode.json", {})
    split_mode = split_meta.get("split_mode", "unknown")
    LABEL_COL = split_meta.get("LABEL_COL", None)
    TIME_COL = split_meta.get("TIME_COL", None)
    GROUP_COL = split_meta.get("GROUP_COL", None)

    num_features = X_train.shape[1]
    n_total_samples = int(len(X_train) + len(X_val) + len(X_cal) + len(X_test))

    print("Loaded cached arrays.")
    print("Shapes:", X_train.shape, X_val.shape, X_cal.shape, X_test.shape)
    print("num_features:", num_features, "| num_fine:", NUM_FINE, "| num_coarse:", NUM_COARSE)
else:
    if running_on_kaggle():
        DATA_ROOT = find_dataset_root(getattr(cfg, "DATASET_HINT", None))
    else:
        DATA_ROOT = Path(cfg.DATA_ROOT).expanduser().resolve()

    print("DATA_ROOT:", DATA_ROOT)

    files = discover_csv_files(DATA_ROOT, cfg.MAX_FILES)
    print("CSV files found:", len(files), "| example:", files[0].name)

    dfs = []
    for p in tqdm(files, desc="Reading CSVs"):
        d = read_one_csv(p, cfg)
        d["__src_file"] = p.name
        dfs.append(d)

    df = pd.concat(dfs, ignore_index=True)
    del dfs
    gc.collect()

    df = sanitize_frame(df, cfg.CLIP_INF)
    df = downcast_numeric(df)
    df = maybe_compress_categoricals(df)

    if cfg.SAMPLE_FRAC is not None:
        df = df.sample(frac=cfg.SAMPLE_FRAC, random_state=cfg.SEED).reset_index(drop=True)

    print("df shape:", df.shape, "| memory(MB):", round(memory_mb(df), 2))

    LABEL_COL = detect_col(df.columns.tolist(), cfg.LABEL_COL_CANDIDATES)
    TIME_COL = detect_col(df.columns.tolist(), cfg.TIME_COL_CANDIDATES)
    GROUP_COL = detect_col(df.columns.tolist(), cfg.GROUP_COL_CANDIDATES)

    print("Detected LABEL_COL:", LABEL_COL)
    print("Detected TIME_COL :", TIME_COL)
    print("Detected GROUP_COL:", GROUP_COL)

    if LABEL_COL is None:
        raise ValueError("Could not detect label column.")

    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()
    unknown = set(df[LABEL_COL].unique()) - set(DICT_34_CLASSES.keys())
    if unknown:
        print("WARNING: unknown labels found (dropping):", list(sorted(unknown))[:20])
        df = df.loc[~df[LABEL_COL].isin(unknown)].copy()

    df["y_fine"] = df[LABEL_COL].map(DICT_34_CLASSES).astype(np.int64)
    df["y_coarse"] = df["y_fine"].map(lambda i: int(fine_to_coarse[i])).astype(np.int64)
    df["y_bin"] = (df["y_fine"] != benign_fine_id).astype(np.int64)

    df["__src_file"] = df["__src_file"].astype("category")
    try:
        df[LABEL_COL] = df[LABEL_COL].astype("category")
    except Exception:
        pass

    print("After label filter:", df.shape, "| fine classes:", df["y_fine"].nunique(), "| coarse:", df["y_coarse"].nunique())

    n_total_samples = int(len(df))

    FEATURE_COLS = build_feature_columns(df, cfg, LABEL_COL, TIME_COL, GROUP_COL)
    print("Num features:", len(FEATURE_COLS))
    print("First 25 features:", FEATURE_COLS[:25])

    split_mode, splits = None, None

    if cfg.SPLIT_POLICY in ("time", "auto") and TIME_COL is not None:
        splits = time_split(df, TIME_COL, cfg)
        split_mode = f"time_order({TIME_COL})"

    if splits is None and cfg.SPLIT_POLICY in ("group", "auto") and GROUP_COL is not None:
        splits = stratified_group_4way(df, "y_fine", GROUP_COL, cfg)
        if splits is not None:
            split_mode = f"stratified_group({GROUP_COL})"

    if splits is None and cfg.SPLIT_POLICY in ("file_holdout", "auto"):
        splits = stratified_group_4way(df, "y_fine", "__src_file", cfg)
        if splits is not None:
            split_mode = "stratified_group(__src_file)"

    if splits is None:
        splits = stratified_4way_split(df, "y_fine", cfg)
        split_mode = "stratified_random"

    print("Split mode:", split_mode)
    for k, v in splits.items():
        print(k, v.shape, "| fine classes:", v["y_fine"].nunique(), "| coarse:", v["y_coarse"].nunique())

    with open(RUN_DIR / "split_mode.json", "w", encoding="utf-8") as f:
        json.dump({
            "split_mode": split_mode,
            "LABEL_COL": LABEL_COL,
            "TIME_COL": TIME_COL,
            "GROUP_COL": GROUP_COL
        }, f, indent=2)

    train_df, val_df, cal_df, test_df = splits["train"], splits["val"], splits["cal"], splits["test"]

    num_imp, scaler, meta = fit_preprocessors(train_df, FEATURE_COLS, cfg)

    X_train = transform_df(train_df, FEATURE_COLS, num_imp, scaler, meta, cfg)
    y_train_f = train_df["y_fine"].values.astype(np.int64)
    y_train_c = train_df["y_coarse"].values.astype(np.int64)
    y_train_b = train_df["y_bin"].values.astype(np.int64)
    del train_df
    gc.collect()

    X_val = transform_df(val_df, FEATURE_COLS, num_imp, scaler, meta, cfg)
    y_val_f = val_df["y_fine"].values.astype(np.int64)
    y_val_c = val_df["y_coarse"].values.astype(np.int64)
    y_val_b = val_df["y_bin"].values.astype(np.int64)
    del val_df
    gc.collect()

    X_cal = transform_df(cal_df, FEATURE_COLS, num_imp, scaler, meta, cfg)
    y_cal_f = cal_df["y_fine"].values.astype(np.int64)
    y_cal_c = cal_df["y_coarse"].values.astype(np.int64)
    y_cal_b = cal_df["y_bin"].values.astype(np.int64)
    gc.collect()

    X_test = transform_df(test_df, FEATURE_COLS, num_imp, scaler, meta, cfg)
    y_test_f = test_df["y_fine"].values.astype(np.int64)
    y_test_c = test_df["y_coarse"].values.astype(np.int64)
    y_test_b = test_df["y_bin"].values.astype(np.int64)

    if cfg.DO_STREAM_EVAL:
        s_test = sort_stream_df_simple(test_df, TIME_COL)
        X_stream_test = transform_df(s_test, FEATURE_COLS, num_imp, scaler, meta, cfg)
        y_stream_f = s_test["y_fine"].values.astype(np.int64)
    else:
        X_stream_test, y_stream_f = None, None

    num_features = X_train.shape[1]
    print("Shapes:", X_train.shape, X_val.shape, X_cal.shape, X_test.shape)
    print("num_features:", num_features, "| num_fine:", NUM_FINE, "| num_coarse:", NUM_COARSE)

    joblib.dump({
        "FEATURE_COLS": FEATURE_COLS,
        "meta": meta,
        "num_imputer": num_imp,
        "scaler": scaler,
        "cfg": asdict(cfg),
        "fine_class_names": FINE_CLASS_NAMES,
        "coarse_names": COARSE_NAMES,
        "fine_to_coarse": fine_to_coarse.tolist(),
        "benign_fine_id": int(benign_fine_id),
    }, export_dir / "preprocess_and_meta.joblib")

    if cfg.CACHE_PREPROCESSED:
        print("Saving preprocessed cache...")
        save_array_dict(ARRAY_CACHE_DIR, {
            "X_train": X_train, "y_train_f": y_train_f, "y_train_c": y_train_c, "y_train_b": y_train_b,
            "X_val": X_val, "y_val_f": y_val_f, "y_val_c": y_val_c, "y_val_b": y_val_b,
            "X_cal": X_cal, "y_cal_f": y_cal_f, "y_cal_c": y_cal_c, "y_cal_b": y_cal_b,
            "X_test": X_test, "y_test_f": y_test_f, "y_test_c": y_test_c, "y_test_b": y_test_b,
        })
        if cfg.DO_STREAM_EVAL and cfg.CACHE_STREAM_INPUTS:
            save_array_dict(STREAM_CACHE_DIR, {
                "X_stream_test": X_stream_test,
                "y_stream_f": y_stream_f,
            })
        print("Preprocessed cache saved.")

    del cal_df, test_df
    del df
    gc.collect()


# %%
PROTOCOL_HINTS = set([norm_colname(x) for x in [
    "HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC", "TCP", "UDP", "DHCP", "ARP", "ICMP", "IPv", "LLC"
]])
STAT_KEYS = ["tot", "sum", "min", "max", "avg", "std", "size", "iat", "number", "magnitude", "magnitue", "radius", "covariance", "variance", "weight"]
RATE_KEYS = ["rate", "srate", "drate"]


def build_feature_groups(feature_cols: List[str]) -> Tuple[List[str], List[List[int]]]:
    groups: Dict[str, List[int]] = {"rates": [], "flags": [], "protocols": [], "stats": [], "other": []}
    for idx, name in enumerate(feature_cols):
        n = norm_colname(name)
        if any(k in n for k in RATE_KEYS):
            groups["rates"].append(idx)
        elif "flag" in n or n.endswith("count") or "count" in n:
            groups["flags"].append(idx)
        elif n in PROTOCOL_HINTS:
            groups["protocols"].append(idx)
        elif any(k in n for k in STAT_KEYS):
            groups["stats"].append(idx)
        else:
            groups["other"].append(idx)

    group_names = [k for k, v in groups.items() if len(v) > 0]
    group_idxs = [groups[k] for k in group_names]
    return group_names, group_idxs


GROUP_NAMES, GROUP_IDXS = build_feature_groups(meta["num_cols"])
print("Groups:", GROUP_NAMES)
for gn, gi in zip(GROUP_NAMES, GROUP_IDXS):
    print(f" - {gn}: {len(gi)} features")

NUM_GROUPS = len(GROUP_NAMES)


# %%
class FlowDataset(Dataset):
    def __init__(self, X: np.ndarray, y_f: np.ndarray, y_c: np.ndarray, y_b: np.ndarray):
        self.X = torch.from_numpy(X.astype(np.float32, copy=False))
        self.yf = torch.from_numpy(y_f.astype(np.int64, copy=False))
        self.yc = torch.from_numpy(y_c.astype(np.int64, copy=False))
        self.yb = torch.from_numpy(y_b.astype(np.int64, copy=False))

    def __len__(self):
        return int(self.X.shape[0])

    def __getitem__(self, idx):
        return self.X[idx], self.yf[idx], self.yc[idx], self.yb[idx]


pin = (cfg.PIN_MEMORY and DEVICE == "cuda")
persistent = (cfg.NUM_WORKERS > 0)

loader_kwargs = dict(
    batch_size=cfg.BATCH_SIZE,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=pin,
    persistent_workers=persistent,
)

if persistent:
    loader_kwargs["prefetch_factor"] = cfg.PREFETCH_FACTOR

ds_train = FlowDataset(X_train, y_train_f, y_train_c, y_train_b)
ds_val = FlowDataset(X_val, y_val_f, y_val_c, y_val_b)
ds_cal = FlowDataset(X_cal, y_cal_f, y_cal_c, y_cal_b)
ds_test = FlowDataset(X_test, y_test_f, y_test_c, y_test_b)

train_loader = DataLoader(ds_train, shuffle=True, **loader_kwargs)
val_loader = DataLoader(ds_val, shuffle=False, **loader_kwargs)
cal_loader = DataLoader(ds_cal, shuffle=False, **loader_kwargs)
test_loader = DataLoader(ds_test, shuffle=False, **loader_kwargs)

print("Train batches:", len(train_loader), "| Val batches:", len(val_loader))


# %%
def class_balanced_weights(y: np.ndarray, num_classes: int, beta: float = 0.9999) -> torch.Tensor:
    counts = np.bincount(y, minlength=num_classes).astype(np.float64)
    effective = 1.0 - np.power(beta, counts)
    w = (1.0 - beta) / np.clip(effective, 1e-12, None)
    w = w / (w.mean() + 1e-12)
    return torch.tensor(w, dtype=torch.float32), counts


class BalancedSoftmaxLoss(nn.Module):
    def __init__(self, class_counts: np.ndarray):
        super().__init__()
        cc = torch.tensor(class_counts.astype(np.float32), dtype=torch.float32)
        self.register_buffer("log_counts", torch.log(cc + 1e-12))

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        adj = logits + self.log_counts.unsqueeze(0)
        return F.cross_entropy(adj, target)


class LogitAdjustedCELoss(nn.Module):
    def __init__(self, class_counts: np.ndarray, tau: float = 0.5):
        super().__init__()
        pi = class_counts / (class_counts.sum() + 1e-12)
        adj = tau * np.log(pi + 1e-12)
        self.register_buffer("adj", torch.tensor(adj, dtype=torch.float32))

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        return F.cross_entropy(logits + self.adj.unsqueeze(0), target)


class LDAMLoss(nn.Module):
    def __init__(self, class_counts: np.ndarray, max_m: float = 0.5, s: float = 30.0, weight: Optional[torch.Tensor] = None):
        super().__init__()
        counts = torch.tensor(class_counts, dtype=torch.float32)
        m = 1.0 / torch.sqrt(torch.sqrt(counts + 1e-12))
        m = m * (max_m / m.max())
        self.register_buffer("m_list", m)
        self.s = float(s)
        self.weight = weight

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        idx = torch.arange(logits.size(0), device=logits.device)
        margins = self.m_list[target]
        logits_m = logits.clone()
        logits_m[idx, target] -= margins
        return F.cross_entropy(self.s * logits_m, target, weight=self.weight)


w_fine_cb, counts_fine = class_balanced_weights(y_train_f, NUM_FINE, cfg.CB_BETA)
w_coarse_cb, counts_coarse = class_balanced_weights(y_train_c, NUM_COARSE, cfg.CB_BETA)
w_bin_cb, counts_bin = class_balanced_weights(y_train_b, 2, cfg.CB_BETA)

print("Fine class count stats:", int(counts_fine.min()), int(counts_fine.max()))


def make_fine_loss(epoch: int) -> nn.Module:
    warm = (epoch <= cfg.DRW_EPOCHS)
    if cfg.FINE_LOSS_MODE == "balanced_softmax":
        return BalancedSoftmaxLoss(counts_fine).to(DEVICE)

    if cfg.FINE_LOSS_MODE == "logit_adj":
        return LogitAdjustedCELoss(counts_fine, tau=cfg.LOGIT_ADJ_TAU).to(DEVICE)

    ww = None if warm else w_fine_cb.to(DEVICE)
    return LDAMLoss(counts_fine, max_m=cfg.LDAM_MAX_M, s=cfg.LDAM_S, weight=ww).to(DEVICE)


def make_aux_losses(epoch: int):
    warm = (epoch <= cfg.DRW_EPOCHS)
    wc = None if warm else w_coarse_cb.to(DEVICE)
    wb = None if warm else w_bin_cb.to(DEVICE)
    loss_coarse = nn.CrossEntropyLoss(weight=wc).to(DEVICE)
    loss_bin = nn.CrossEntropyLoss(weight=wb).to(DEVICE)
    return loss_coarse, loss_bin


fine_to_coarse_t = torch.tensor(fine_to_coarse, device=DEVICE, dtype=torch.long)


# %%
class FeatureTokenizer(nn.Module):
    def __init__(self, n_features: int, d_model: int):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_features, d_model))
        self.b = nn.Parameter(torch.empty(n_features, d_model))
        nn.init.trunc_normal_(self.W, std=0.02)
        nn.init.trunc_normal_(self.b, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x.unsqueeze(-1) * self.W.unsqueeze(0) + self.b.unsqueeze(0)


def make_encoder_layer(d_model, nhead, ff_dim, dropout, attn_dropout):
    return nn.TransformerEncoderLayer(
        d_model=d_model,
        nhead=nhead,
        dim_feedforward=ff_dim,
        dropout=dropout,
        activation="gelu",
        batch_first=True,
        norm_first=True
    )


class CamelotIDSv2(nn.Module):
    def __init__(self, n_features: int, n_fine: int, n_coarse: int, group_idxs: List[List[int]], cfg: ExpConfig):
        super().__init__()
        self.cfg = cfg
        self.n_features = n_features
        self.n_fine = n_fine
        self.n_coarse = n_coarse
        self.group_idxs = group_idxs
        self.num_groups = len(group_idxs)

        self.tokenizer = FeatureTokenizer(n_features, cfg.D_MODEL)

        self.group_token = nn.Parameter(torch.zeros(self.num_groups, cfg.D_MODEL))
        nn.init.trunc_normal_(self.group_token, std=0.02)
        self.group_type = nn.Embedding(self.num_groups, cfg.D_MODEL)

        for gi, idxs in enumerate(group_idxs):
            self.register_buffer(f"group_idx_{gi}", torch.tensor(idxs, dtype=torch.long))

        local_layer = make_encoder_layer(cfg.D_MODEL, cfg.NHEAD, cfg.FF_DIM, cfg.DROPOUT, cfg.ATTN_DROPOUT)
        self.local_encoder = nn.TransformerEncoder(local_layer, num_layers=cfg.LOCAL_LAYERS)

        self.cls = nn.Parameter(torch.zeros(1, 1, cfg.D_MODEL))
        nn.init.trunc_normal_(self.cls, std=0.02)

        global_layer = make_encoder_layer(cfg.D_MODEL, cfg.NHEAD, cfg.FF_DIM, cfg.DROPOUT, cfg.ATTN_DROPOUT)
        self.global_encoder = nn.TransformerEncoder(global_layer, num_layers=cfg.GLOBAL_LAYERS)

        self.pos_global = nn.Parameter(torch.zeros(1, 1 + self.num_groups, cfg.D_MODEL))
        nn.init.trunc_normal_(self.pos_global, std=0.02)

        self.head_fine = nn.Sequential(nn.LayerNorm(cfg.D_MODEL), nn.Dropout(cfg.DROPOUT), nn.Linear(cfg.D_MODEL, n_fine))
        self.head_coarse = nn.Sequential(nn.LayerNorm(cfg.D_MODEL), nn.Dropout(cfg.DROPOUT), nn.Linear(cfg.D_MODEL, n_coarse))
        self.head_bin = nn.Sequential(nn.LayerNorm(cfg.D_MODEL), nn.Dropout(cfg.DROPOUT), nn.Linear(cfg.D_MODEL, 2))

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        B = x.size(0)
        tok = self.tokenizer(x)

        group_reps = []
        for g in range(self.num_groups):
            idxs_t = getattr(self, f"group_idx_{g}")
            t = tok.index_select(dim=1, index=idxs_t)
            t = t + self.group_type.weight[g].view(1, 1, -1)

            gt = self.group_token[g].view(1, 1, -1).expand(B, 1, -1)
            z = torch.cat([gt, t], dim=1)
            z = self.local_encoder(z)
            group_reps.append(z[:, 0])

        G = torch.stack(group_reps, dim=1)
        cls = self.cls.expand(B, -1, -1)
        Z = torch.cat([cls, G], dim=1)
        Z = Z + self.pos_global
        Z = self.global_encoder(Z)
        h = Z[:, 0]

        return {
            "logits_fine": self.head_fine(h),
            "logits_coarse": self.head_coarse(h),
            "logits_bin": self.head_bin(h),
        }


model = CamelotIDSv2(
    n_features=num_features,
    n_fine=NUM_FINE,
    n_coarse=NUM_COARSE,
    group_idxs=GROUP_IDXS,
    cfg=cfg
).to(DEVICE)

if cfg.USE_TORCH_COMPILE and hasattr(torch, "compile"):
    try:
        model = torch.compile(model, mode="reduce-overhead")
        print("torch.compile enabled")
    except Exception as e:
        print("torch.compile skipped:", e)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print("Model params (M):", round(n_params, 3))


# %%
def softmax_np(x: np.ndarray) -> np.ndarray:
    x = x - x.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / np.clip(ex.sum(axis=1, keepdims=True), 1e-12, None)


def metrics_mc(y_true: np.ndarray, probs: np.ndarray) -> Dict[str, float]:
    y_pred = probs.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
    }


class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = float(decay)
        self.shadow = {}
        self.backup = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[name] = p.detach().clone()

    @torch.no_grad()
    def update(self, model: nn.Module):
        for name, p in model.named_parameters():
            if name in self.shadow:
                self.shadow[name].mul_(self.decay).add_(p.detach(), alpha=(1.0 - self.decay))

    def apply(self, model: nn.Module):
        self.backup = {}
        sd = model.state_dict()
        for name in self.shadow:
            self.backup[name] = sd[name].detach().clone()
            sd[name].copy_(self.shadow[name])

    def restore(self, model: nn.Module):
        sd = model.state_dict()
        for name, p in self.backup.items():
            sd[name].copy_(p)
        self.backup = {}

    def state_dict(self):
        return {
            "decay": self.decay,
            "shadow": self.shadow,
        }

    def load_state_dict(self, state):
        self.decay = float(state["decay"])
        self.shadow = {k: v.clone() for k, v in state["shadow"].items()}
        self.backup = {}


@torch.inference_mode()
def predict_logits(model: nn.Module, loader: DataLoader) -> Dict[str, np.ndarray]:
    model.eval()
    out = {"fine": [], "coarse": [], "bin": [], "yf": [], "yc": [], "yb": []}
    use_amp = (cfg.USE_AMP and DEVICE == "cuda")

    for xb, yf, yc, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", enabled=use_amp):
            r = model(xb)
        out["fine"].append(r["logits_fine"].detach().cpu().numpy())
        out["coarse"].append(r["logits_coarse"].detach().cpu().numpy())
        out["bin"].append(r["logits_bin"].detach().cpu().numpy())
        out["yf"].append(yf.numpy())
        out["yc"].append(yc.numpy())
        out["yb"].append(yb.numpy())

    for k in ["fine", "coarse", "bin", "yf", "yc", "yb"]:
        out[k] = np.concatenate(out[k], axis=0)
    return out


def predict_logits_cached(name: str, model: nn.Module, loader: DataLoader, signature: Optional[Dict[str, Any]] = None) -> Dict[str, np.ndarray]:
    folder = LOGITS_CACHE_DIR / name
    meta_path = folder / "_meta.json"

    if cfg.RESUME and cfg.CACHE_LOGITS and folder_ready(folder, LOGIT_KEYS):
        cache_meta = load_json(meta_path, {})
        if same_signature(cache_meta.get("signature", {}), signature or {}):
            print(f"Loading cached logits: {name}")
            return load_array_dict(folder, LOGIT_KEYS)

    out = predict_logits(model, loader)

    if cfg.CACHE_LOGITS:
        save_array_dict(folder, out)
        save_json_atomic(meta_path, {
            "name": name,
            "signature": signature or {}
        })
    return out


def hierarchical_consistency_loss(logits_fine, logits_coarse, logits_bin):
    p_f = torch.softmax(logits_fine, dim=1)
    p_c = torch.softmax(logits_coarse, dim=1)
    p_b = torch.softmax(logits_bin, dim=1)

    B = p_f.size(0)
    idx = fine_to_coarse_t.unsqueeze(0).expand(B, -1)
    p_from_f = torch.zeros(B, NUM_COARSE, device=p_f.device)
    p_from_f.scatter_add_(1, idx, p_f)
    cons_c = F.kl_div(torch.log(p_c + 1e-8), p_from_f, reduction="batchmean")

    p_from_f_bin = torch.stack([p_f[:, benign_fine_id], 1.0 - p_f[:, benign_fine_id]], dim=1)
    cons_b = F.kl_div(torch.log(p_b + 1e-8), p_from_f_bin, reduction="batchmean")
    return cons_c, cons_b


def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, cfg: ExpConfig):
    last_ckpt_path = CKPT_DIR / "last_checkpoint.pt"
    best_ckpt_path = CKPT_DIR / "best_model.pt"
    done_path = STAGE_DIR / "training_done.json"
    history_csv_path = RUN_DIR / "epoch_history.csv"

    def make_empty_history():
        return {
            "epoch": [],
            "train_loss": [],

            "val_fine_accuracy": [],
            "val_fine_balanced_accuracy": [],
            "val_fine_macro_f1": [],
            "val_fine_weighted_f1": [],

            "val_coarse_accuracy": [],
            "val_coarse_balanced_accuracy": [],
            "val_coarse_macro_f1": [],
            "val_coarse_weighted_f1": [],

            "val_bin_f1": [],

            "best_score_so_far": [],
            "best_epoch_so_far": [],
        }

    def upgrade_history(history_obj):
        base = make_empty_history()

        if not isinstance(history_obj, dict):
            return base

        # copy existing keys
        for k in base.keys():
            if k in history_obj and isinstance(history_obj[k], list):
                base[k] = history_obj[k]

        # backward compatibility with your older history format
        if len(base["epoch"]) == 0:
            n_old = 0
            for k in history_obj.keys():
                if isinstance(history_obj[k], list):
                    n_old = max(n_old, len(history_obj[k]))
            if n_old > 0:
                base["epoch"] = list(range(1, n_old + 1))

        if len(base["val_fine_macro_f1"]) == 0 and "val_macro_f1_fine" in history_obj:
            base["val_fine_macro_f1"] = history_obj["val_macro_f1_fine"]

        if len(base["val_coarse_macro_f1"]) == 0 and "val_macro_f1_coarse" in history_obj:
            base["val_coarse_macro_f1"] = history_obj["val_macro_f1_coarse"]

        if len(base["val_bin_f1"]) == 0 and "val_f1_bin" in history_obj:
            base["val_bin_f1"] = history_obj["val_f1_bin"]

        if len(base["train_loss"]) == 0 and "train_loss" in history_obj:
            base["train_loss"] = history_obj["train_loss"]

        # make all lists same length
        max_len = max(len(v) for v in base.values()) if len(base) > 0 else 0
        for k in base.keys():
            if len(base[k]) < max_len:
                if k == "epoch" and len(base[k]) == 0:
                    base[k] = list(range(1, max_len + 1))
                else:
                    base[k] = base[k] + [float("nan")] * (max_len - len(base[k]))

        return base

    def save_history_csv(history_obj, csv_path):
        df_hist = pd.DataFrame(history_obj)
        df_hist.to_csv(csv_path, index=False)

    def print_history(history_obj):
        if len(history_obj["epoch"]) == 0:
            print("No completed epochs found in saved history.")
            return

        print("\nSaved completed epochs:")
        for i in range(len(history_obj["epoch"])):
            ep = history_obj["epoch"][i]
            tr_loss = history_obj["train_loss"][i]
            ff1 = history_obj["val_fine_macro_f1"][i]
            cf1 = history_obj["val_coarse_macro_f1"][i]
            bf1 = history_obj["val_bin_f1"][i]

            tr_loss_s = f"{tr_loss:.4f}" if pd.notna(tr_loss) else "nan"
            ff1_s = f"{ff1:.4f}" if pd.notna(ff1) else "nan"
            cf1_s = f"{cf1:.4f}" if pd.notna(cf1) else "nan"
            bf1_s = f"{bf1:.4f}" if pd.notna(bf1) else "nan"

            print(
                f"[Completed epoch {ep}] "
                f"loss={tr_loss_s} | "
                f"val_fine_macroF1={ff1_s} | "
                f"val_coarse_macroF1={cf1_s} | "
                f"val_binF1={bf1_s}"
            )

    empty_history = make_empty_history()

    if cfg.RESUME and done_path.exists() and best_ckpt_path.exists():
        done_meta = load_json(done_path, {})
        best_blob = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(best_blob["model_state"])
        history = upgrade_history(done_meta.get("history", empty_history))
        save_history_csv(history, history_csv_path)
        print(f"Training already complete. Loaded best model from epoch {done_meta.get('best_epoch', '?')}.")
        print_history(history)
        return history

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

    total_steps = max(1, cfg.EPOCHS * len(train_loader))
    warmup_steps = max(1, int(cfg.WARMUP_EPOCHS * len(train_loader)))

    def lr_lambda(step):
        if step < warmup_steps:
            return max(1e-6, step / max(1, warmup_steps))
        t = (step - warmup_steps) / max(1, (total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * t))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    use_amp = (cfg.USE_AMP and DEVICE == "cuda")
    scaler_obj = torch.amp.GradScaler("cuda", enabled=use_amp) if DEVICE == "cuda" else None
    ema = EMA(model, decay=0.999)

    best_score = -1.0
    best_epoch = 0
    patience, bad = 7, 0
    start_epoch = 1
    history = make_empty_history()

    if cfg.RESUME and last_ckpt_path.exists():
        print("Resuming from last checkpoint...")
        ckpt = torch.load(last_ckpt_path, map_location=DEVICE, weights_only=False)

        model.load_state_dict(ckpt["model_state"])
        opt.load_state_dict(ckpt["optimizer_state"])
        sched.load_state_dict(ckpt["scheduler_state"])

        if scaler_obj is not None and ckpt.get("scaler_state") is not None:
            scaler_obj.load_state_dict(ckpt["scaler_state"])

        if ckpt.get("ema_state") is not None:
            ema.load_state_dict(ckpt["ema_state"])

        history = upgrade_history(ckpt.get("history", history))
        best_score = float(ckpt.get("best_score", -1.0))
        best_epoch = int(ckpt.get("best_epoch", 0))
        bad = int(ckpt.get("bad", 0))
        start_epoch = int(ckpt["epoch"]) + 1

        set_rng_state(ckpt.get("rng_state", None))
        print(f"Resumed at epoch {start_epoch}/{cfg.EPOCHS}")

        print_history(history)
        save_history_csv(history, history_csv_path)

    for epoch in range(start_epoch, cfg.EPOCHS + 1):
        model.train()
        loss_fine = make_fine_loss(epoch)
        loss_coarse, loss_bin = make_aux_losses(epoch)

        total_loss, n = 0.0, 0

        for xb, yf, yc, yb in tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.EPOCHS}", leave=False):
            xb = xb.to(DEVICE, non_blocking=True)
            yf = yf.to(DEVICE, non_blocking=True)
            yc = yc.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", enabled=use_amp):
                out = model(xb)
                lf = loss_fine(out["logits_fine"], yf)
                lc = loss_coarse(out["logits_coarse"], yc)
                lb = loss_bin(out["logits_bin"], yb)
                cons_c, cons_b = hierarchical_consistency_loss(
                    out["logits_fine"], out["logits_coarse"], out["logits_bin"]
                )
                loss = (
                    lf
                    + cfg.W_COARSE * lc
                    + cfg.W_BIN * lb
                    + cfg.W_CONS_COARSE * cons_c
                    + cfg.W_CONS_BIN * cons_b
                )

            if use_amp:
                scaler_obj.scale(loss).backward()
                scaler_obj.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
                scaler_obj.step(opt)
                scaler_obj.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
                opt.step()

            sched.step()
            ema.update(model)

            total_loss += float(loss.item()) * int(xb.size(0))
            n += int(xb.size(0))

        train_loss = total_loss / max(n, 1)

        ema.apply(model)
        val_out = predict_logits(model, val_loader)
        ema.restore(model)

        val_probs_f = softmax_np(val_out["fine"])
        val_probs_c = softmax_np(val_out["coarse"])
        val_probs_b = softmax_np(val_out["bin"])

        m_f = metrics_mc(val_out["yf"], val_probs_f)
        m_c = metrics_mc(val_out["yc"], val_probs_c)
        yb_pred = val_probs_b.argmax(axis=1)
        f1_bin = float(f1_score(val_out["yb"], yb_pred, average="binary", pos_label=1))

        score = m_f["macro_f1"]
        if score > best_score:
            best_score = score
            best_epoch = epoch
            bad = 0

            atomic_torch_save({
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_score": float(best_score),
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            }, best_ckpt_path)
        else:
            bad += 1

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)

        history["val_fine_accuracy"].append(m_f["accuracy"])
        history["val_fine_balanced_accuracy"].append(m_f["balanced_accuracy"])
        history["val_fine_macro_f1"].append(m_f["macro_f1"])
        history["val_fine_weighted_f1"].append(m_f["weighted_f1"])

        history["val_coarse_accuracy"].append(m_c["accuracy"])
        history["val_coarse_balanced_accuracy"].append(m_c["balanced_accuracy"])
        history["val_coarse_macro_f1"].append(m_c["macro_f1"])
        history["val_coarse_weighted_f1"].append(m_c["weighted_f1"])

        history["val_bin_f1"].append(f1_bin)

        history["best_score_so_far"].append(float(best_score))
        history["best_epoch_so_far"].append(int(best_epoch))

        print(
            f"[Epoch {epoch}] loss={train_loss:.4f} | "
            f"val_fine_acc={m_f['accuracy']:.4f} | "
            f"val_fine_macroF1={m_f['macro_f1']:.4f} | "
            f"val_coarse_acc={m_c['accuracy']:.4f} | "
            f"val_coarse_macroF1={m_c['macro_f1']:.4f} | "
            f"val_binF1={f1_bin:.4f}"
        )

        save_history_csv(history, history_csv_path)

        if cfg.SAVE_LAST_EVERY_EPOCH:
            atomic_torch_save({
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_score": float(best_score),
                "bad": int(bad),
                "model_state": model.state_dict(),
                "optimizer_state": opt.state_dict(),
                "scheduler_state": sched.state_dict(),
                "scaler_state": scaler_obj.state_dict() if scaler_obj is not None else None,
                "ema_state": ema.state_dict(),
                "history": history,
                "rng_state": get_rng_state(),
            }, last_ckpt_path)

        if bad >= patience:
            print("Early stopping.")
            break

    if best_ckpt_path.exists():
        best_blob = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(best_blob["model_state"])

    save_json_atomic(done_path, {
        "best_epoch": int(best_epoch),
        "best_score": float(best_score),
        "history": history,
    })

    save_history_csv(history, history_csv_path)
    return history
    
history = train_model(model, train_loader, val_loader, cfg)
training_signature = get_training_signature()


# %%
test_out = predict_logits_cached("test", model, test_loader, signature=training_signature)

test_probs_f = softmax_np(test_out["fine"])
test_probs_c = softmax_np(test_out["coarse"])
test_probs_b = softmax_np(test_out["bin"])

test_metrics_f = metrics_mc(test_out["yf"], test_probs_f)
test_metrics_c = metrics_mc(test_out["yc"], test_probs_c)

yb_pred = test_probs_b.argmax(axis=1)
test_f1_bin = float(f1_score(test_out["yb"], yb_pred, average="binary", pos_label=1))

print("TEST fine:", test_metrics_f)
print("TEST coarse:", test_metrics_c)
print("TEST binF1(malicious):", test_f1_bin)

with open(RUN_DIR / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump({"fine": test_metrics_f, "coarse": test_metrics_c, "binF1_mal": test_f1_bin}, f, indent=2)

y_pred_f = test_probs_f.argmax(axis=1)
prec, rec, f1c, sup = precision_recall_fscore_support(
    test_out["yf"], y_pred_f, labels=np.arange(NUM_FINE), zero_division=0
)
per_class = [
    {
        "class": FINE_CLASS_NAMES[i],
        "support": int(sup[i]),
        "precision": float(prec[i]),
        "recall": float(rec[i]),
        "f1": float(f1c[i]),
    }
    for i in range(NUM_FINE)
]
per_class_sorted = sorted(per_class, key=lambda d: d["f1"])

with open(RUN_DIR / "per_class_metrics_sorted_by_f1.json", "w", encoding="utf-8") as f:
    json.dump(per_class_sorted, f, indent=2)

print("Worst-10 fine classes by F1:")
for r in per_class_sorted[:10]:
    print(r)

cmc = confusion_matrix(test_out["yc"], test_probs_c.argmax(axis=1), labels=np.arange(NUM_COARSE))
plt.figure(figsize=(8, 6))
plt.imshow(cmc, aspect="auto")
plt.title("Confusion Matrix (Coarse 8-class)")
plt.xlabel("Pred")
plt.ylabel("True")
plt.colorbar()
plt.tight_layout()
plt.savefig(str(RUN_DIR / "confusion_matrix_coarse.png"), dpi=200)
plt.close()


# %%
def expected_calibration_error(y_true: np.ndarray, probs: np.ndarray, n_bins: int = 15) -> float:
    y_pred = probs.argmax(axis=1)
    conf = probs[np.arange(len(probs)), y_pred]
    acc = (y_pred == y_true).astype(np.float32)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (conf >= lo) & (conf < hi) if i < n_bins - 1 else (conf >= lo) & (conf <= hi)
        if m.any():
            ece += (m.mean()) * abs(acc[m].mean() - conf[m].mean())
    return float(ece)


def brier_multi(y_true: np.ndarray, probs: np.ndarray, n_classes: int) -> float:
    y_onehot = np.eye(n_classes)[y_true]
    return float(np.mean(np.sum((probs - y_onehot) ** 2, axis=1)))


class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_t = nn.Parameter(torch.zeros(()))

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        t = torch.exp(self.log_t).clamp(1e-3, 100.0)
        return logits / t


def fit_temperature_np(logits_np: np.ndarray, y_np: np.ndarray) -> float:
    logits = torch.from_numpy(logits_np).to(DEVICE)
    y = torch.from_numpy(y_np).to(DEVICE)
    ts = TemperatureScaler().to(DEVICE)
    opt = torch.optim.LBFGS(ts.parameters(), lr=0.5, max_iter=50)
    nll = nn.CrossEntropyLoss()

    def closure():
        opt.zero_grad(set_to_none=True)
        loss = nll(ts(logits), y)
        loss.backward()
        return loss

    opt.step(closure)
    return float(torch.exp(ts.log_t).detach().cpu().item())


calibration_done_path = STAGE_DIR / "calibration_done.json"
cal_out = predict_logits_cached("cal", model, cal_loader, signature=training_signature)

if cfg.RESUME and calibration_done_path.exists() and (RUN_DIR / "calibration_summary.json").exists():
    calibration_done = load_json(calibration_done_path, {})
    if same_signature(calibration_done.get("signature", {}), training_signature):
        calibration_summary = load_json(RUN_DIR / "calibration_summary.json", {})
        temp_f = float(calibration_summary["temperature"]["fine"])
        temp_c = float(calibration_summary["temperature"]["coarse"])
        temp_b = float(calibration_summary["temperature"]["bin"])
        print("Loaded cached calibration summary.")
    else:
        calibration_summary = None
        temp_f, temp_c, temp_b = 1.0, 1.0, 1.0
else:
    calibration_summary = None
    temp_f, temp_c, temp_b = 1.0, 1.0, 1.0

if calibration_summary is None:
    if cfg.TEMPERATURE_SCALE:
        temp_f = fit_temperature_np(cal_out["fine"], cal_out["yf"])
        temp_c = fit_temperature_np(cal_out["coarse"], cal_out["yc"])
        temp_b = fit_temperature_np(cal_out["bin"], cal_out["yb"])

    print("Temperatures:", {"fine": temp_f, "coarse": temp_c, "bin": temp_b})

    test_probs_f_cal = softmax_np(test_out["fine"] / max(temp_f, 1e-6))
    test_probs_c_cal = softmax_np(test_out["coarse"] / max(temp_c, 1e-6))
    test_probs_b_cal = softmax_np(test_out["bin"] / max(temp_b, 1e-6))

    nll_before = float(log_loss(test_out["yf"], test_probs_f, labels=list(range(NUM_FINE))))
    nll_after = float(log_loss(test_out["yf"], test_probs_f_cal, labels=list(range(NUM_FINE))))
    ece_before = expected_calibration_error(test_out["yf"], test_probs_f, cfg.N_BINS_ECE)
    ece_after = expected_calibration_error(test_out["yf"], test_probs_f_cal, cfg.N_BINS_ECE)
    brier_before = brier_multi(test_out["yf"], test_probs_f, NUM_FINE)
    brier_after = brier_multi(test_out["yf"], test_probs_f_cal, NUM_FINE)

    calibration_summary = {
        "temperature": {"fine": temp_f, "coarse": temp_c, "bin": temp_b},
        "fine_before": {"nll": nll_before, "ece": ece_before, "brier": brier_before},
        "fine_after": {"nll": nll_after, "ece": ece_after, "brier": brier_after},
    }
    with open(RUN_DIR / "calibration_summary.json", "w", encoding="utf-8") as f:
        json.dump(calibration_summary, f, indent=2)
    save_json_atomic(calibration_done_path, {"signature": training_signature})
else:
    test_probs_f_cal = softmax_np(test_out["fine"] / max(temp_f, 1e-6))
    test_probs_c_cal = softmax_np(test_out["coarse"] / max(temp_c, 1e-6))
    test_probs_b_cal = softmax_np(test_out["bin"] / max(temp_b, 1e-6))

print("Calibration summary:", calibration_summary)

meta_obj = joblib.load(export_dir / "preprocess_and_meta.joblib")
meta_obj["temperature"] = {"fine": float(temp_f), "coarse": float(temp_c), "bin": float(temp_b)}
joblib.dump(meta_obj, export_dir / "preprocess_and_meta.joblib")


# %%
def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    n = len(scores)
    if n == 0:
        return 1.0
    q_level = math.ceil((n + 1) * (1 - alpha)) / n
    return float(np.quantile(scores, min(q_level, 1.0), method="higher"))


def raps_scores_true(probs: np.ndarray, y_true: np.ndarray, k_reg: int, lam: float) -> np.ndarray:
    n, C = probs.shape
    idx_sorted = np.argsort(-probs, axis=1)
    probs_sorted = np.take_along_axis(probs, idx_sorted, axis=1)
    cumsum = np.cumsum(probs_sorted, axis=1)

    inv = np.empty_like(idx_sorted)
    rows = np.arange(n)[:, None]
    inv[rows, idx_sorted] = np.arange(C)[None, :]
    rank0 = inv[rows[:, 0], y_true]
    rank1 = rank0 + 1

    score = cumsum[rows[:, 0], rank0] + lam * np.maximum(rank1 - k_reg, 0)
    return score.astype(np.float32)


def raps_predict_k(probs: np.ndarray, q: np.ndarray, k_reg: int, lam: float) -> np.ndarray:
    n, C = probs.shape
    idx_sorted = np.argsort(-probs, axis=1)
    probs_sorted = np.take_along_axis(probs, idx_sorted, axis=1)
    cumsum = np.cumsum(probs_sorted, axis=1)
    ranks = np.arange(1, C + 1, dtype=np.float32)[None, :]
    reg = lam * np.maximum(ranks - float(k_reg), 0.0)
    score_k = cumsum + reg
    ok = score_k <= q[:, None]
    k_star = ok.sum(axis=1).astype(np.int64)
    return np.maximum(k_star, 1)


def raps_eval(probs: np.ndarray, y_true: np.ndarray, q_row: np.ndarray, k_reg: int, lam: float) -> Dict[str, float]:
    n, C = probs.shape
    idx_sorted = np.argsort(-probs, axis=1)

    inv = np.empty_like(idx_sorted)
    rows = np.arange(n)[:, None]
    inv[rows, idx_sorted] = np.arange(C)[None, :]
    rank0 = inv[rows[:, 0], y_true]

    k_star = raps_predict_k(probs, q_row, k_reg, lam)
    covered = (rank0 < k_star).astype(np.float32)
    return {"coverage": float(covered.mean()), "avg_set_size": float(k_star.mean())}


raps_done_path = STAGE_DIR / "raps_done.json"
if cfg.RESUME and raps_done_path.exists() and (RUN_DIR / "raps_calibration.json").exists() and (RUN_DIR / "raps_results.json").exists():
    raps_done = load_json(raps_done_path, {})
    if same_signature(raps_done.get("signature", {}), training_signature):
        raps_calib = load_json(RUN_DIR / "raps_calibration.json", {})
        raps_results = load_json(RUN_DIR / "raps_results.json", {})
        print("Loaded cached RAPS outputs.")
    else:
        raps_calib = None
        raps_results = None
else:
    raps_calib = None
    raps_results = None

if raps_calib is None or raps_results is None:
    cal_probs_f = softmax_np(cal_out["fine"] / max(temp_f, 1e-6))
    cal_probs_c = softmax_np(cal_out["coarse"] / max(temp_c, 1e-6))

    scores_global = raps_scores_true(cal_probs_f, cal_out["yf"], cfg.RAPS_KREG, cfg.RAPS_LAMBDA)
    g_hat = cal_probs_c.argmax(axis=1)
    scores_by_g = {g: scores_global[g_hat == g] for g in range(NUM_COARSE)}

    raps_calib = {
        "mode": cfg.RAPS_MONDRIAN,
        "k_reg": int(cfg.RAPS_KREG),
        "lambda": float(cfg.RAPS_LAMBDA),
        "alphas": [float(a) for a in cfg.ALPHAS],
        "q_global": {},
        "q_by_pred_coarse": {},
        "min_group_n": 200
    }

    for a in cfg.ALPHAS:
        qg = conformal_quantile(scores_global, a)
        raps_calib["q_global"][str(a)] = float(qg)

        q_by = []
        for g in range(NUM_COARSE):
            sg = scores_by_g.get(g, np.array([], dtype=np.float32))
            if len(sg) >= raps_calib["min_group_n"]:
                q_by.append(conformal_quantile(sg, a))
            else:
                q_by.append(qg)
        raps_calib["q_by_pred_coarse"][str(a)] = [float(x) for x in q_by]

    with open(RUN_DIR / "raps_calibration.json", "w", encoding="utf-8") as f:
        json.dump(raps_calib, f, indent=2)

    g_hat_test = test_probs_c_cal.argmax(axis=1)
    raps_results = {}
    for a in cfg.ALPHAS:
        q_by = np.array(raps_calib["q_by_pred_coarse"][str(a)], dtype=np.float32)
        q_row = q_by[g_hat_test]
        r = raps_eval(test_probs_f_cal, test_out["yf"], q_row, cfg.RAPS_KREG, cfg.RAPS_LAMBDA)
        raps_results[str(a)] = r

    with open(RUN_DIR / "raps_results.json", "w", encoding="utf-8") as f:
        json.dump(raps_results, f, indent=2)

    save_json_atomic(raps_done_path, {"signature": training_signature})

print("RAPS results:", raps_results)


def singleton_risk_coverage(y_true, probs_f, probs_c, raps_calib, alpha, k_reg, lam, benign_id=0):
    q_by = np.array(raps_calib["q_by_pred_coarse"][str(alpha)], dtype=np.float32)
    g_hat = probs_c.argmax(axis=1)
    q_row = q_by[g_hat]
    k_star = raps_predict_k(probs_f, q_row, k_reg, lam)

    single = (k_star == 1)
    y_pred = probs_f.argmax(axis=1)

    cov = float(single.mean())
    if single.any():
        acc = float((y_pred[single] == y_true[single]).mean())
        mf1 = float(f1_score(y_true[single], y_pred[single], average="macro"))
    else:
        acc, mf1 = float("nan"), float("nan")

    benign_mask = (y_true == benign_id) & single
    benign_fpr = float((y_pred[benign_mask] != benign_id).mean()) if benign_mask.any() else float("nan")

    return {
        "alpha": float(alpha),
        "coverage_singleton": cov,
        "risk_1_minus_acc": float(1.0 - acc),
        "macro_f1_on_decisions": mf1,
        "benign_FPR_on_decisions": benign_fpr,
        "avg_set_size": float(k_star.mean()),
    }


sel_rows = [
    singleton_risk_coverage(
        test_out["yf"], test_probs_f_cal, test_probs_c_cal,
        raps_calib, a, cfg.RAPS_KREG, cfg.RAPS_LAMBDA, benign_fine_id
    )
    for a in cfg.ALPHAS
]
df_sel = pd.DataFrame(sel_rows)
df_sel.to_csv(RUN_DIR / "risk_coverage_table_raps.csv", index=False)
print(df_sel)

meta_obj = joblib.load(export_dir / "preprocess_and_meta.joblib")
meta_obj["raps_calib"] = raps_calib
joblib.dump(meta_obj, export_dir / "preprocess_and_meta.joblib")


# %%
def make_drift_detector(cfg: ExpConfig):
    try:
        from river import drift
        if "KSWIN" in cfg.DRIFT_DETECTOR.upper():
            return drift.KSWIN(alpha=cfg.DRIFT_DELTA, window_size=100, stat_size=30)
        if "ADWIN" in cfg.DRIFT_DETECTOR.upper():
            return drift.ADWIN(delta=cfg.DRIFT_DELTA)
    except Exception:
        pass
    return None


class ConformalMartingale:
    def __init__(self, ref_values: np.ndarray, eps: float = 0.5, threshold: float = 25.0):
        self.ref = np.asarray(ref_values, dtype=np.float32)
        self.eps = float(eps)
        self.threshold = float(threshold)
        self.M = 1.0

    def p_value(self, x: float) -> float:
        ref = self.ref
        return float((1.0 + np.sum(ref >= x)) / (len(ref) + 1.0))

    def update(self, x: float) -> Dict[str, float]:
        p = self.p_value(x)
        self.M *= self.eps * (p ** (self.eps - 1.0))
        drift = float(self.M > self.threshold)
        if drift:
            self.M = 1.0
        return {"p": float(p), "M": float(self.M), "drift": float(drift)}


class TentAdapter:
    def __init__(self, model: nn.Module, lr: float = 1e-4, steps: int = 5):
        self.model = model
        self.steps = int(steps)

        for p in self.model.parameters():
            p.requires_grad = False

        self.params = []
        for m in self.model.modules():
            if isinstance(m, (nn.LayerNorm, nn.BatchNorm1d, nn.GroupNorm)):
                for p in m.parameters():
                    p.requires_grad = True
                    self.params.append(p)

        self.opt = torch.optim.Adam(self.params, lr=lr)

        for m in self.model.modules():
            if isinstance(m, nn.Dropout):
                m.eval()

    def adapt(self, xb_cpu: torch.Tensor, micro_bs: int):
        if len(self.params) == 0:
            return
        self.model.train()
        for _ in range(self.steps):
            for j in range(0, xb_cpu.size(0), micro_bs):
                xb = xb_cpu[j:j + micro_bs].to(DEVICE, non_blocking=True)
                out = self.model(xb)
                p = torch.softmax(out["logits_fine"], dim=1)
                ent = -(p * torch.log(p + 1e-8)).sum(dim=1).mean()
                self.opt.zero_grad(set_to_none=True)
                ent.backward()
                self.opt.step()
                del xb, out, p, ent


class HeadTuner:
    def __init__(self, model: nn.Module, lr: float = 5e-5, steps: int = 10):
        self.model = model
        self.steps = int(steps)

        for p in self.model.parameters():
            p.requires_grad = False

        params = []
        for head in [self.model.head_fine, self.model.head_coarse, self.model.head_bin]:
            for p in head.parameters():
                p.requires_grad = True
                params.append(p)

        self.opt = torch.optim.Adam(params, lr=lr)
        self.model.eval()

    def tune(self, xb_cpu: torch.Tensor, y_f: torch.Tensor, y_c: torch.Tensor, y_b: torch.Tensor, micro_bs: int):
        self.model.train()
        y_f = y_f.cpu()
        y_c = y_c.cpu()
        y_b = y_b.cpu()

        for _ in range(self.steps):
            for j in range(0, xb_cpu.size(0), micro_bs):
                xb = xb_cpu[j:j + micro_bs].to(DEVICE, non_blocking=True)
                yf = y_f[j:j + micro_bs].to(DEVICE, non_blocking=True)
                yc = y_c[j:j + micro_bs].to(DEVICE, non_blocking=True)
                yb = y_b[j:j + micro_bs].to(DEVICE, non_blocking=True)

                out = self.model(xb)
                lf = F.cross_entropy(out["logits_fine"], yf)
                lc = F.cross_entropy(out["logits_coarse"], yc)
                lb = F.cross_entropy(out["logits_bin"], yb)
                loss = lf + 0.3 * lc + 0.2 * lb
                self.opt.zero_grad(set_to_none=True)
                loss.backward()
                self.opt.step()

                del xb, yf, yc, yb, out, lf, lc, lb, loss


@torch.inference_mode()
def batch_predict_probs(
    model: nn.Module,
    xb_np: np.ndarray,
    temp_f: float,
    temp_c: float,
    micro_bs: int = 16_384
):
    model.eval()
    pf_parts, pc_parts = [], []
    use_amp = (cfg.USE_AMP and DEVICE == "cuda")

    for j in range(0, len(xb_np), micro_bs):
        xb = torch.from_numpy(xb_np[j:j + micro_bs].astype(np.float32, copy=False)).to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", enabled=use_amp):
            out = model(xb)
            pf = torch.softmax(out["logits_fine"] / max(temp_f, 1e-6), dim=1).cpu()
            pc = torch.softmax(out["logits_coarse"] / max(temp_c, 1e-6), dim=1).cpu()

        pf_parts.append(pf)
        pc_parts.append(pc)
        del xb, out, pf, pc

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return torch.cat(pf_parts, dim=0).numpy(), torch.cat(pc_parts, dim=0).numpy()


def entropy_from_probs(p: np.ndarray) -> float:
    return float(np.mean(-np.sum(p * np.log(p + 1e-12), axis=1)))


def signal_from_probs(pf: np.ndarray, pc: np.ndarray, cfg: ExpConfig, raps_calib: Dict[str, Any]) -> float:
    if cfg.DRIFT_SIGNAL == "1-maxprob":
        return float(np.mean(1.0 - pf.max(axis=1)))
    if cfg.DRIFT_SIGNAL == "set_size":
        alpha = float(cfg.ADAPT_ALPHA)
        q_by = np.array(raps_calib["q_by_pred_coarse"][str(alpha)], dtype=np.float32)
        g_hat = pc.argmax(axis=1)
        q_row = q_by[g_hat]
        k_star = raps_predict_k(pf, q_row, cfg.RAPS_KREG, cfg.RAPS_LAMBDA)
        return float(k_star.mean())
    return entropy_from_probs(pf)


def stream_eval(cfg: ExpConfig, Xs: np.ndarray, ys_f: np.ndarray):
    ref_probs_f = softmax_np(cal_out["fine"] / max(temp_f, 1e-6))
    ref_ent = -np.sum(ref_probs_f * np.log(ref_probs_f + 1e-12), axis=1)
    ref_ent = ref_ent[:min(len(ref_ent), 200_000)]

    det = make_drift_detector(cfg)
    mart = ConformalMartingale(ref_ent, eps=cfg.MART_EPS, threshold=cfg.MART_THRESHOLD) if "MART" in cfg.DRIFT_DETECTOR.upper() else None

    tent = TentAdapter(model, lr=cfg.ADAPT_LR, steps=cfg.ADAPT_STEPS) if "TENT" in cfg.ADAPT_METHOD.upper() else None
    head_tuner = HeadTuner(model, lr=cfg.HEAD_TUNE_LR, steps=cfg.HEAD_TUNE_STEPS) if "HEAD" in cfg.ADAPT_METHOD.upper() else None

    B = cfg.STREAM_BATCH
    MB = cfg.STREAM_MICRO_BATCH
    acc_hist, sig_hist, drift_points = [], [], []
    p_hist, m_hist = [], []
    buffer = []

    for bi, i in enumerate(tqdm(range(0, len(Xs), B), desc="Stream eval")):
        xb_np = Xs[i:i + B]
        yb = ys_f[i:i + B]
        if len(xb_np) == 0:
            break

        pf, pc = batch_predict_probs(model, xb_np, temp_f, temp_c, micro_bs=MB)
        pred = pf.argmax(axis=1)
        acc = float(np.mean(pred == yb))
        acc_hist.append(acc)

        sgn = signal_from_probs(pf, pc, cfg, raps_calib)
        sig_hist.append(sgn)

        drifted = False

        if det is not None and ("ADWIN" in cfg.DRIFT_DETECTOR.upper() or "KSWIN" in cfg.DRIFT_DETECTOR.upper()):
            det.update(sgn)
            drifted = drifted or bool(getattr(det, "drift_detected", False))

        if mart is not None:
            upd = mart.update(sgn)
            p_hist.append(upd["p"])
            m_hist.append(upd["M"])
            drifted = drifted or bool(upd["drift"] > 0)

        xb_cpu = torch.from_numpy(xb_np.astype(np.float32, copy=False))
        buffer.append((xb_cpu, pf, pc))
        if len(buffer) > cfg.ADAPT_BUFFER_BATCHES:
            buffer.pop(0)

        if drifted:
            drift_points.append(bi)

            if tent is not None:
                for xb_buf, _, _ in buffer:
                    tent.adapt(xb_buf, micro_bs=MB)

            if head_tuner is not None:
                alpha = float(cfg.ADAPT_ALPHA)
                q_by = np.array(raps_calib["q_by_pred_coarse"][str(alpha)], dtype=np.float32)

                xb_buf, pf_buf, pc_buf = buffer[-1]
                g_hat = pc_buf.argmax(axis=1)
                q_row = q_by[g_hat]
                k_star = raps_predict_k(pf_buf, q_row, cfg.RAPS_KREG, cfg.RAPS_LAMBDA)
                single = (k_star == 1)

                if int(single.sum()) >= cfg.MIN_PSEUDO:
                    y_pseudo_f = pf_buf.argmax(axis=1)[single]
                    y_pseudo_c = fine_to_coarse[y_pseudo_f]
                    y_pseudo_b = (y_pseudo_f != benign_fine_id).astype(np.int64)

                    xb_sel = xb_buf[single]
                    yf_t = torch.from_numpy(y_pseudo_f.astype(np.int64))
                    yc_t = torch.from_numpy(y_pseudo_c.astype(np.int64))
                    yb_t = torch.from_numpy(y_pseudo_b.astype(np.int64))
                    head_tuner.tune(xb_sel, yf_t, yc_t, yb_t, micro_bs=MB)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(acc_hist, label="fine accuracy (labels only for analysis)")
    for p in drift_points:
        ax.axvline(p, linestyle=":", alpha=0.7)
    ax.set_title("Streaming accuracy with drift points")
    ax.set_xlabel("batch idx")
    ax.set_ylabel("accuracy")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.savefig(str(RUN_DIR / "stream_accuracy.png"), dpi=200)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(sig_hist, label=f"signal={cfg.DRIFT_SIGNAL}")
    for p in drift_points:
        ax.axvline(p, linestyle=":", alpha=0.7)
    ax.set_title("Streaming drift signal")
    ax.set_xlabel("batch idx")
    ax.set_ylabel(cfg.DRIFT_SIGNAL)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.savefig(str(RUN_DIR / "stream_signal.png"), dpi=200)
    plt.close(fig)

    if mart is not None and len(p_hist) == len(sig_hist):
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(m_hist, label="martingale M")
        ax.axhline(cfg.MART_THRESHOLD, linestyle="--", alpha=0.7, label="threshold")
        ax.set_title("Conformal martingale")
        ax.set_xlabel("batch idx")
        ax.set_ylabel("M")
        ax.grid(True, alpha=0.3)
        ax.legend()
        fig.savefig(str(RUN_DIR / "stream_martingale.png"), dpi=200)
        plt.close(fig)

    out = {"signal": cfg.DRIFT_SIGNAL, "acc_hist": acc_hist, "sig_hist": sig_hist, "drift_points": drift_points}
    with open(RUN_DIR / "stream_results.json", "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2)
    return out


stream_results = None
stream_done_path = STAGE_DIR / "stream_done.json"
stream_json = RUN_DIR / "stream_results.json"

if cfg.DO_STREAM_EVAL:
    cached_ok = False
    if cfg.RESUME and stream_done_path.exists() and stream_json.exists():
        stream_done = load_json(stream_done_path, {})
        cached_ok = same_signature(stream_done.get("signature", {}), training_signature)

    if cached_ok:
        stream_results = load_json(stream_json, None)
        print("Loaded cached stream results.")
    else:
        stream_results = stream_eval(cfg, X_stream_test, y_stream_f)
        save_json_atomic(stream_done_path, {"signature": training_signature})

    print("Drift points:", stream_results["drift_points"], "total:", len(stream_results["drift_points"]))


# %%
def sample_subset(X: np.ndarray, y: np.ndarray, max_n: int, seed: int = 42):
    if len(X) <= max_n:
        return X, y
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=max_n, replace=False)
    return X[idx], y[idx]


baseline_results = {}

if cfg.RUN_BASELINES:
    Xtr_b, ytr_b = sample_subset(X_train, y_train_f, cfg.BASELINE_MAX_TRAIN, cfg.SEED)
    Xte_b, yte_b = sample_subset(X_test, y_test_f, cfg.BASELINE_MAX_TEST, cfg.SEED)

    try:
        from catboost import CatBoostClassifier
        cb = CatBoostClassifier(
            loss_function="MultiClass",
            iterations=2500,
            depth=10,
            learning_rate=0.08,
            auto_class_weights="Balanced",
            verbose=250,
            task_type="GPU" if DEVICE == "cuda" else "CPU"
        )
        cb.fit(Xtr_b, ytr_b, eval_set=(Xte_b[:min(200000, len(Xte_b))], yte_b[:min(200000, len(yte_b))]))
        p = cb.predict_proba(Xte_b)
        baseline_results["CatBoost_fine34"] = metrics_mc(yte_b, p)
    except Exception as e:
        baseline_results["CatBoost_fine34"] = {"error": str(e)}

    try:
        import xgboost as xgb
        xgbm = xgb.XGBClassifier(
            n_estimators=1600,
            max_depth=10,
            learning_rate=0.06,
            subsample=0.85,
            colsample_bytree=0.85,
            tree_method="hist",
            n_jobs=-1
        )
        xgbm.fit(Xtr_b, ytr_b)
        p = xgbm.predict_proba(Xte_b)
        baseline_results["XGBoost_fine34"] = metrics_mc(yte_b, p)
    except Exception as e:
        baseline_results["XGBoost_fine34"] = {"error": str(e)}

with open(RUN_DIR / "baseline_results.json", "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, indent=2)

print("Baselines:", baseline_results)


# %%
@torch.inference_mode()
def measure_throughput(model: nn.Module, X_np: np.ndarray, batch_size: int = 4096, device: str = DEVICE):
    model.eval()
    n = len(X_np)
    if n == 0:
        return {"samples_per_sec": float("nan"), "ms_per_sample": float("nan")}

    warm = min(batch_size, n)
    xb = torch.from_numpy(X_np[:warm].astype(np.float32, copy=False)).to(device)
    for _ in range(5):
        _ = model(xb)

    if device == "cuda":
        torch.cuda.synchronize()

    t0 = time.time()
    seen = 0
    for i in range(0, n, batch_size):
        xb = torch.from_numpy(X_np[i:i + batch_size].astype(np.float32, copy=False)).to(device)
        _ = model(xb)
        seen += xb.size(0)

    if device == "cuda":
        torch.cuda.synchronize()

    dt = time.time() - t0
    sps = seen / max(dt, 1e-9)
    return {"samples_per_sec": float(sps), "ms_per_sample": float(1000.0 / sps)}


throughput_gpu = measure_throughput(model, X_test[:min(len(X_test), 200_000)], batch_size=4096, device=DEVICE)

model_cpu_for_bench = model.to("cpu")
throughput_cpu = measure_throughput(model_cpu_for_bench, X_test[:min(len(X_test), 200_000)], batch_size=4096, device="cpu")
model = model_cpu_for_bench.to(DEVICE)

print("Throughput GPU:", throughput_gpu)
print("Throughput CPU:", throughput_cpu)

torch.save({"state_dict": model.state_dict()}, export_dir / "camelot_ids_v2.pt")


class LogitsWrapper(nn.Module):
    def __init__(self, base: nn.Module):
        super().__init__()
        self.base = base

    def forward(self, x: torch.Tensor):
        o = self.base(x)
        return o["logits_fine"], o["logits_coarse"], o["logits_bin"]


wrapper = LogitsWrapper(model).to(DEVICE).eval()

if cfg.EXPORT_TORCHSCRIPT:
    try:
        example = torch.randn(1, num_features, device=DEVICE)
        ts = torch.jit.trace(wrapper, example)
        ts.save(str(export_dir / "camelot_ids_v2_logits_torchscript.pt"))
        print("TorchScript saved.")
    except Exception as e:
        print("TorchScript failed:", e)

if cfg.EXPORT_ONNX:
    try:
        dummy = torch.randn(1, num_features, device=DEVICE)
        torch.onnx.export(
            wrapper,
            dummy,
            str(export_dir / "camelot_ids_v2_logits.onnx"),
            input_names=["input"],
            output_names=["logits_fine", "logits_coarse", "logits_bin"],
            dynamic_axes={
                "input": {0: "batch"},
                "logits_fine": {0: "batch"},
                "logits_coarse": {0: "batch"},
                "logits_bin": {0: "batch"},
            },
            opset_version=13
        )
        print("ONNX saved.")
    except Exception as e:
        print("ONNX failed:", e)

if cfg.EXPORT_INT8_DYNAMIC:
    try:
        wrapper_cpu = LogitsWrapper(model.to("cpu")).eval()
        qmodel = torch.quantization.quantize_dynamic(wrapper_cpu, {nn.Linear}, dtype=torch.qint8)
        example = torch.randn(1, num_features)
        qts = torch.jit.trace(qmodel, example)
        qts.save(str(export_dir / "camelot_ids_v2_logits_torchscript_int8.pt"))
        print("Dynamic int8 TorchScript saved.")
        model = model.to(DEVICE)
    except Exception as e:
        print("Dynamic quantization failed:", e)
        model = model.to(DEVICE)

cli_code = r'''
import argparse
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import torch


def sanitize_frame(df: pd.DataFrame, clip_inf: float = 1e9) -> pd.DataFrame:
    df = df.replace({"Infinity": np.inf, "inf": np.inf, "+inf": np.inf, "-inf": -np.inf,
                     "NaN": np.nan, "nan": np.nan, "": np.nan})
    df = df.replace([np.inf, -np.inf], np.nan)
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        df[num_cols] = df[num_cols].clip(-clip_inf, clip_inf)
    return df


def transform_df(df_in: pd.DataFrame, feature_cols, meta_obj):
    cfg = meta_obj["cfg"]
    X = sanitize_frame(df_in[feature_cols].copy(), cfg["CLIP_INF"])
    for c in meta_obj["meta"]["numeric_like_obj"]:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")
    num_cols = meta_obj["meta"]["num_cols"]
    X = X[num_cols]
    Xn = meta_obj["num_imputer"].transform(X).astype(np.float32)
    Xn = meta_obj["scaler"].transform(Xn).astype(np.float32)
    Xn = np.clip(Xn, -cfg["POST_SCALE_CLIP"], cfg["POST_SCALE_CLIP"]).astype(np.float32)
    Xn = np.nan_to_num(Xn, nan=0.0, posinf=cfg["POST_SCALE_CLIP"], neginf=-cfg["POST_SCALE_CLIP"]).astype(np.float32)
    return Xn


def softmax_np(x):
    x = x - x.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / np.clip(ex.sum(axis=1, keepdims=True), 1e-12, None)


def raps_predict_set(probs_f, probs_c, raps_calib, alpha):
    k_reg = int(raps_calib["k_reg"])
    lam = float(raps_calib["lambda"])
    q_by = np.array(raps_calib["q_by_pred_coarse"][str(alpha)], dtype=np.float32)
    g_hat = probs_c.argmax(axis=1)
    q_row = q_by[g_hat]

    idx_sorted = np.argsort(-probs_f, axis=1)
    probs_sorted = np.take_along_axis(probs_f, idx_sorted, axis=1)
    cumsum = np.cumsum(probs_sorted, axis=1)
    C = probs_f.shape[1]
    ranks = np.arange(1, C + 1, dtype=np.float32)[None, :]
    reg = lam * np.maximum(ranks - float(k_reg), 0.0)
    score_k = cumsum + reg
    ok = score_k <= q_row[:, None]
    k_star = np.maximum(ok.sum(axis=1).astype(np.int64), 1)

    sets = []
    for i in range(len(probs_f)):
        sets.append(idx_sorted[i, :k_star[i]])
    return sets, k_star


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--export_dir", type=str, required=True)
    ap.add_argument("--input_csv", type=str, required=True)
    ap.add_argument("--out_csv", type=str, default="preds.csv")
    ap.add_argument("--device", type=str, default="cpu")
    ap.add_argument("--int8", action="store_true")
    ap.add_argument("--alpha", type=float, default=0.10)
    args = ap.parse_args()

    exp = Path(args.export_dir)
    meta = joblib.load(exp / "preprocess_and_meta.joblib")
    feature_cols = meta["FEATURE_COLS"]
    fine_names = meta["fine_class_names"]
    coarse_names = meta["coarse_names"]
    temp = meta.get("temperature", {"fine": 1.0, "coarse": 1.0, "bin": 1.0})
    raps_calib = meta.get("raps_calib", None)
    if raps_calib is None:
        raise RuntimeError("No raps_calib found in preprocess_and_meta.joblib")

    model_path = exp / ("camelot_ids_v2_logits_torchscript_int8.pt" if args.int8 else "camelot_ids_v2_logits_torchscript.pt")
    model = torch.jit.load(str(model_path), map_location=args.device)
    model.eval()

    df = pd.read_csv(args.input_csv, low_memory=False)
    X = transform_df(df, feature_cols, meta)
    xb = torch.from_numpy(X).to(args.device)

    with torch.inference_mode():
        logits_f, logits_c, logits_b = model(xb)
        logits_f = logits_f.cpu().numpy()
        logits_c = logits_c.cpu().numpy()
        logits_b = logits_b.cpu().numpy()

    pf = softmax_np(logits_f / max(float(temp["fine"]), 1e-6))
    pc = softmax_np(logits_c / max(float(temp["coarse"]), 1e-6))
    pb = softmax_np(logits_b / max(float(temp["bin"]), 1e-6))

    top_f = pf.argmax(axis=1)
    top_conf = pf[np.arange(len(pf)), top_f]
    top_c = pc.argmax(axis=1)
    top_b = pb.argmax(axis=1)

    sets, k_star = raps_predict_set(pf, pc, raps_calib, float(args.alpha))
    set_str = [",".join([fine_names[j] for j in s]) for s in sets]

    out = pd.DataFrame({
        "top1_fine": [fine_names[i] for i in top_f],
        "top1_conf": top_conf,
        "pred_coarse": [coarse_names[i] for i in top_c],
        "pred_bin": ["Malicious" if i == 1 else "Benign" for i in top_b],
        "raps_set_size": k_star,
        "raps_set": set_str,
    })
    out.to_csv(args.out_csv, index=False)
    print("Saved:", args.out_csv)


if __name__ == "__main__":
    main()
'''
(export_dir / "camelot_infer_v2.py").write_text(cli_code, encoding="utf-8")
print("CLI written:", export_dir / "camelot_infer_v2.py")


# %%
final_summary = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "split_mode": split_mode,
    "num_samples": n_total_samples,
    "num_features": int(num_features),
    "num_fine": int(NUM_FINE),
    "num_coarse": int(NUM_COARSE),
    "test_fine": test_metrics_f,
    "test_coarse": test_metrics_c,
    "test_binF1_mal": test_f1_bin,
    "calibration": calibration_summary,
    "raps": raps_results,
    "baselines": baseline_results,
    "throughput_gpu": throughput_gpu,
    "throughput_cpu": throughput_cpu,
    "stream_results": stream_results,
}
with open(RUN_DIR / "final_summary.json", "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=2)

print("Saved:", RUN_DIR / "final_summary.json")
print("All outputs in:", cfg.OUT_DIR)


DEVICE: cuda
OUT_DIR: ./results_camelot_ids_v2_pc
Loading cached preprocessed arrays...
Loaded cached arrays.
Shapes: (21705973, 46) (2631074, 46) (1389497, 46) (1372896, 46)
num_features: 46 | num_fine: 34 | num_coarse: 8
Groups: ['rates', 'flags', 'protocols', 'stats', 'other']
 - rates: 3 features
 - flags: 12 features
 - protocols: 14 features
 - stats: 13 features
 - other: 4 features
Train batches: 21198 | Val batches: 2570
Fine class count stats: 604 3347293
Model params (M): 6.359
Training already complete. Loaded best model from epoch 61.

Saved completed epochs:
[Completed epoch 1] loss=1.7803 | val_fine_macroF1=0.6532 | val_coarse_macroF1=0.6886 | val_binF1=0.9968
[Completed epoch 2] loss=0.3281 | val_fine_macroF1=0.6563 | val_coarse_macroF1=0.6771 | val_binF1=0.9965
[Completed epoch 3] loss=0.2321 | val_fine_macroF1=0.6635 | val_coarse_macroF1=0.6880 | val_binF1=0.9966
[Completed epoch 4] loss=0.2460 | val_fine_macroF1=0.6827 | val_coarse_macroF1=0.6935 | val_binF1=0.9968
[

W0320 03:58:31.772000 14328 site-packages\torch\onnx\_internal\exporter\_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `LogitsWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `LogitsWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 13).


[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 13 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "C:\Users\HaseebWajid\AppData\Local\anaconda3\envs\torch_env\Lib\site-packages\onnxscript\version_converter\__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\HaseebWajid\AppData\Local\anaconda3\envs\torch_env\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "C:\Users\HaseebWajid\AppData\Local\anaconda3\envs\torch_env\Lib\site-packages\onnxscript\version_converter\__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\HaseebWajid\AppData\Local\anaconda3\envs\torch_env\Lib\site-packages\onnx\version_converter.py", line 39, in convert_vers

Applied 26 of general pattern rewrite rules.
ONNX saved.
Dynamic quantization failed: 'function' object has no attribute 'device'
CLI written: results_camelot_ids_v2_pc\export\camelot_infer_v2.py
Saved: results_camelot_ids_v2_pc\final_summary.json
All outputs in: ./results_camelot_ids_v2_pc


In [5]:
# ============================================================
# BOOTSTRAP CELL FOR PUBLICATION EXTENSION
# Run this BEFORE the publication extension cell.
# It reconstructs the saved run context from disk after a restart.
# ============================================================

from pathlib import Path
from types import SimpleNamespace
import json
import math
import gc
import warnings

import numpy as np
import pandas as pd
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0) Basic runtime
# ------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# CHANGE THIS ONLY IF YOUR saved run folder is somewhere else
DEFAULT_OUT_DIR = Path("./results_camelot_ids_v2_pc")

# ------------------------------------------------------------
# 1) Locate saved run folder + config
# ------------------------------------------------------------
if "cfg" not in globals():
    config_path = DEFAULT_OUT_DIR / "config.json"
    if not config_path.exists():
        raise FileNotFoundError(
            f"Could not find config.json at: {config_path}\n"
            f"Edit DEFAULT_OUT_DIR in this bootstrap cell to your saved run folder."
        )

    with open(config_path, "r", encoding="utf-8") as f:
        cfg_dict = json.load(f)
    cfg = SimpleNamespace(**cfg_dict)
    print("Loaded cfg from:", config_path)

if "RUN_DIR" not in globals():
    RUN_DIR = Path(cfg.OUT_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = RUN_DIR / "cache"
ARRAY_CACHE_DIR = CACHE_DIR / "arrays"
LOGITS_CACHE_DIR = CACHE_DIR / "logits"
STREAM_CACHE_DIR = CACHE_DIR / "stream"
CKPT_DIR = RUN_DIR / "checkpoints"
STAGE_DIR = RUN_DIR / "stages"
export_dir = RUN_DIR / "export"

for d in [CACHE_DIR, ARRAY_CACHE_DIR, LOGITS_CACHE_DIR, STREAM_CACHE_DIR, CKPT_DIR, STAGE_DIR, export_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("RUN_DIR:", RUN_DIR)

# ------------------------------------------------------------
# 2) Utility helpers
# ------------------------------------------------------------
def load_json(path: Path, default=None):
    path = Path(path)
    if not path.exists():
        return {} if default is None else default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json_atomic(path: Path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    tmp.replace(path)

def atomic_save_npy(path: Path, arr: np.ndarray):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "wb") as f:
        np.save(f, arr, allow_pickle=False)
    tmp.replace(path)

def folder_ready(folder: Path, keys):
    folder = Path(folder)
    return (folder / "_ok.json").exists() and all((folder / f"{k}.npy").exists() for k in keys)

def load_array_dict(folder: Path, keys=None):
    folder = Path(folder)
    meta = load_json(folder / "_ok.json", {})
    if keys is None:
        keys = meta.get("keys", [])
    return {k: np.load(folder / f"{k}.npy") for k in keys}

def save_array_dict(folder: Path, arrays):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    for k, v in arrays.items():
        atomic_save_npy(folder / f"{k}.npy", v)
    save_json_atomic(folder / "_ok.json", {"keys": list(arrays.keys())})

def stable_json_dumps(obj):
    return json.dumps(obj, sort_keys=True, default=str)

def same_signature(a, b):
    return stable_json_dumps(a or {}) == stable_json_dumps(b or {})

def get_training_signature():
    sig = {}
    done_path = STAGE_DIR / "training_done.json"
    best_path = CKPT_DIR / "best_model.pt"
    if done_path.exists():
        done = load_json(done_path, {})
        sig["best_epoch"] = done.get("best_epoch")
        sig["best_score"] = done.get("best_score")
    if best_path.exists():
        sig["best_ckpt_mtime_ns"] = best_path.stat().st_mtime_ns
        sig["best_ckpt_size"] = best_path.stat().st_size
    return sig

def softmax_np(x: np.ndarray) -> np.ndarray:
    x = x - x.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / np.clip(ex.sum(axis=1, keepdims=True), 1e-12, None)

def metrics_mc(y_true: np.ndarray, probs: np.ndarray):
    y_pred = probs.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
    }

# ------------------------------------------------------------
# 3) Load metadata saved by main run
# ------------------------------------------------------------
meta_path = export_dir / "preprocess_and_meta.joblib"
if not meta_path.exists():
    raise FileNotFoundError(
        f"Missing saved metadata: {meta_path}\n"
        f"Your main notebook must finish preprocessing/model setup at least once."
    )

meta_obj = joblib.load(meta_path)

FEATURE_COLS = meta_obj["FEATURE_COLS"]
meta = meta_obj["meta"]
num_imp = meta_obj["num_imputer"]
scaler = meta_obj["scaler"]

FINE_CLASS_NAMES = meta_obj["fine_class_names"]
COARSE_NAMES = meta_obj["coarse_names"]
fine_to_coarse = np.array(meta_obj["fine_to_coarse"], dtype=np.int64)
benign_fine_id = int(meta_obj["benign_fine_id"])

NUM_FINE = len(FINE_CLASS_NAMES)
NUM_COARSE = len(COARSE_NAMES)

# ------------------------------------------------------------
# 4) Rebuild feature groups
# ------------------------------------------------------------
import re

def norm_colname(c: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(c).strip().lower())

PROTOCOL_HINTS = set([norm_colname(x) for x in [
    "HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC", "TCP", "UDP", "DHCP", "ARP", "ICMP", "IPv", "LLC"
]])
STAT_KEYS = ["tot", "sum", "min", "max", "avg", "std", "size", "iat", "number", "magnitude", "magnitue", "radius", "covariance", "variance", "weight"]
RATE_KEYS = ["rate", "srate", "drate"]

def build_feature_groups(feature_cols):
    groups = {"rates": [], "flags": [], "protocols": [], "stats": [], "other": []}
    for idx, name in enumerate(feature_cols):
        n = norm_colname(name)
        if any(k in n for k in RATE_KEYS):
            groups["rates"].append(idx)
        elif "flag" in n or n.endswith("count") or "count" in n:
            groups["flags"].append(idx)
        elif n in PROTOCOL_HINTS:
            groups["protocols"].append(idx)
        elif any(k in n for k in STAT_KEYS):
            groups["stats"].append(idx)
        else:
            groups["other"].append(idx)

    group_names = [k for k, v in groups.items() if len(v) > 0]
    group_idxs = [groups[k] for k in group_names]
    return group_names, group_idxs

GROUP_NAMES, GROUP_IDXS = build_feature_groups(meta["num_cols"])
num_features = len(meta["num_cols"])

print("num_features:", num_features)
print("GROUP_NAMES:", GROUP_NAMES)

# ------------------------------------------------------------
# 5) Load cached arrays + rebuild loaders
# ------------------------------------------------------------
ARRAY_KEYS = [
    "X_train", "y_train_f", "y_train_c", "y_train_b",
    "X_val", "y_val_f", "y_val_c", "y_val_b",
    "X_cal", "y_cal_f", "y_cal_c", "y_cal_b",
    "X_test", "y_test_f", "y_test_c", "y_test_b",
]

if not folder_ready(ARRAY_CACHE_DIR, ARRAY_KEYS):
    raise FileNotFoundError(
        f"Cached preprocessed arrays not found in: {ARRAY_CACHE_DIR}\n"
        f"Your main notebook must have saved preprocessing cache."
    )

arrs = load_array_dict(ARRAY_CACHE_DIR, ARRAY_KEYS)

X_train = arrs["X_train"]
y_train_f = arrs["y_train_f"]
y_train_c = arrs["y_train_c"]
y_train_b = arrs["y_train_b"]

X_val = arrs["X_val"]
y_val_f = arrs["y_val_f"]
y_val_c = arrs["y_val_c"]
y_val_b = arrs["y_val_b"]

X_cal = arrs["X_cal"]
y_cal_f = arrs["y_cal_f"]
y_cal_c = arrs["y_cal_c"]
y_cal_b = arrs["y_cal_b"]

X_test = arrs["X_test"]
y_test_f = arrs["y_test_f"]
y_test_c = arrs["y_test_c"]
y_test_b = arrs["y_test_b"]

class FlowDataset(Dataset):
    def __init__(self, X, y_f, y_c, y_b):
        self.X = torch.from_numpy(X.astype(np.float32, copy=False))
        self.yf = torch.from_numpy(y_f.astype(np.int64, copy=False))
        self.yc = torch.from_numpy(y_c.astype(np.int64, copy=False))
        self.yb = torch.from_numpy(y_b.astype(np.int64, copy=False))

    def __len__(self):
        return int(self.X.shape[0])

    def __getitem__(self, idx):
        return self.X[idx], self.yf[idx], self.yc[idx], self.yb[idx]

pin = (DEVICE == "cuda")
loader_kwargs = dict(
    batch_size=getattr(cfg, "BATCH_SIZE", 1024),
    num_workers=0,
    pin_memory=pin,
    persistent_workers=False,
)

ds_val = FlowDataset(X_val, y_val_f, y_val_c, y_val_b)
ds_cal = FlowDataset(X_cal, y_cal_f, y_cal_c, y_cal_b)
ds_test = FlowDataset(X_test, y_test_f, y_test_c, y_test_b)

val_loader = DataLoader(ds_val, shuffle=False, **loader_kwargs)
cal_loader = DataLoader(ds_cal, shuffle=False, **loader_kwargs)
test_loader = DataLoader(ds_test, shuffle=False, **loader_kwargs)

print("Cached arrays loaded:")
print("  X_val :", X_val.shape)
print("  X_cal :", X_cal.shape)
print("  X_test:", X_test.shape)

# ------------------------------------------------------------
# 6) Recreate model definition
# ------------------------------------------------------------
class FeatureTokenizer(nn.Module):
    def __init__(self, n_features: int, d_model: int):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_features, d_model))
        self.b = nn.Parameter(torch.empty(n_features, d_model))
        nn.init.trunc_normal_(self.W, std=0.02)
        nn.init.trunc_normal_(self.b, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x.unsqueeze(-1) * self.W.unsqueeze(0) + self.b.unsqueeze(0)

def make_encoder_layer(d_model, nhead, ff_dim, dropout, attn_dropout):
    return nn.TransformerEncoderLayer(
        d_model=d_model,
        nhead=nhead,
        dim_feedforward=ff_dim,
        dropout=dropout,
        activation="gelu",
        batch_first=True,
        norm_first=True
    )

class CamelotIDSv2(nn.Module):
    def __init__(self, n_features: int, n_fine: int, n_coarse: int, group_idxs, cfg):
        super().__init__()
        self.cfg = cfg
        self.n_features = n_features
        self.n_fine = n_fine
        self.n_coarse = n_coarse
        self.group_idxs = group_idxs
        self.num_groups = len(group_idxs)

        self.tokenizer = FeatureTokenizer(n_features, cfg.D_MODEL)

        self.group_token = nn.Parameter(torch.zeros(self.num_groups, cfg.D_MODEL))
        nn.init.trunc_normal_(self.group_token, std=0.02)
        self.group_type = nn.Embedding(self.num_groups, cfg.D_MODEL)

        for gi, idxs in enumerate(group_idxs):
            self.register_buffer(f"group_idx_{gi}", torch.tensor(idxs, dtype=torch.long))

        local_layer = make_encoder_layer(cfg.D_MODEL, cfg.NHEAD, cfg.FF_DIM, cfg.DROPOUT, cfg.ATTN_DROPOUT)
        self.local_encoder = nn.TransformerEncoder(local_layer, num_layers=cfg.LOCAL_LAYERS)

        self.cls = nn.Parameter(torch.zeros(1, 1, cfg.D_MODEL))
        nn.init.trunc_normal_(self.cls, std=0.02)

        global_layer = make_encoder_layer(cfg.D_MODEL, cfg.NHEAD, cfg.FF_DIM, cfg.DROPOUT, cfg.ATTN_DROPOUT)
        self.global_encoder = nn.TransformerEncoder(global_layer, num_layers=cfg.GLOBAL_LAYERS)

        self.pos_global = nn.Parameter(torch.zeros(1, 1 + self.num_groups, cfg.D_MODEL))
        nn.init.trunc_normal_(self.pos_global, std=0.02)

        self.head_fine = nn.Sequential(nn.LayerNorm(cfg.D_MODEL), nn.Dropout(cfg.DROPOUT), nn.Linear(cfg.D_MODEL, n_fine))
        self.head_coarse = nn.Sequential(nn.LayerNorm(cfg.D_MODEL), nn.Dropout(cfg.DROPOUT), nn.Linear(cfg.D_MODEL, n_coarse))
        self.head_bin = nn.Sequential(nn.LayerNorm(cfg.D_MODEL), nn.Dropout(cfg.DROPOUT), nn.Linear(cfg.D_MODEL, 2))

    def forward(self, x: torch.Tensor):
        B = x.size(0)
        tok = self.tokenizer(x)

        group_reps = []
        for g in range(self.num_groups):
            idxs_t = getattr(self, f"group_idx_{g}")
            t = tok.index_select(dim=1, index=idxs_t)
            t = t + self.group_type.weight[g].view(1, 1, -1)

            gt = self.group_token[g].view(1, 1, -1).expand(B, 1, -1)
            z = torch.cat([gt, t], dim=1)
            z = self.local_encoder(z)
            group_reps.append(z[:, 0])

        G = torch.stack(group_reps, dim=1)
        cls = self.cls.expand(B, -1, -1)
        Z = torch.cat([cls, G], dim=1)
        Z = Z + self.pos_global
        Z = self.global_encoder(Z)
        h = Z[:, 0]

        return {
            "logits_fine": self.head_fine(h),
            "logits_coarse": self.head_coarse(h),
            "logits_bin": self.head_bin(h),
        }

# ------------------------------------------------------------
# 7) Prediction + calibration helpers
# ------------------------------------------------------------
@torch.inference_mode()
def predict_logits(model: nn.Module, loader: DataLoader):
    model.eval()
    out = {"fine": [], "coarse": [], "bin": [], "yf": [], "yc": [], "yb": []}
    use_amp = (getattr(cfg, "USE_AMP", True) and DEVICE == "cuda")

    for xb, yf, yc, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", enabled=use_amp):
            r = model(xb)
        out["fine"].append(r["logits_fine"].detach().cpu().numpy())
        out["coarse"].append(r["logits_coarse"].detach().cpu().numpy())
        out["bin"].append(r["logits_bin"].detach().cpu().numpy())
        out["yf"].append(yf.numpy())
        out["yc"].append(yc.numpy())
        out["yb"].append(yb.numpy())

    for k in ["fine", "coarse", "bin", "yf", "yc", "yb"]:
        out[k] = np.concatenate(out[k], axis=0)
    return out

LOGIT_KEYS = ["fine", "coarse", "bin", "yf", "yc", "yb"]

def predict_logits_cached(name: str, model: nn.Module, loader: DataLoader, signature=None):
    folder = LOGITS_CACHE_DIR / name
    meta_path = folder / "_meta.json"

    if folder_ready(folder, LOGIT_KEYS):
        cache_meta = load_json(meta_path, {})
        if same_signature(cache_meta.get("signature", {}), signature or {}):
            print(f"Loading cached logits: {name}")
            return load_array_dict(folder, LOGIT_KEYS)

    out = predict_logits(model, loader)
    save_array_dict(folder, out)
    save_json_atomic(meta_path, {"name": name, "signature": signature or {}})
    return out

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_t = nn.Parameter(torch.zeros(()))

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        t = torch.exp(self.log_t).clamp(1e-3, 100.0)
        return logits / t

def fit_temperature_np(logits_np: np.ndarray, y_np: np.ndarray) -> float:
    logits = torch.from_numpy(logits_np).to(DEVICE)
    y = torch.from_numpy(y_np).to(DEVICE)
    ts = TemperatureScaler().to(DEVICE)
    opt = torch.optim.LBFGS(ts.parameters(), lr=0.5, max_iter=50)
    nll = nn.CrossEntropyLoss()

    def closure():
        opt.zero_grad(set_to_none=True)
        loss = nll(ts(logits), y)
        loss.backward()
        return loss

    opt.step(closure)
    return float(torch.exp(ts.log_t).detach().cpu().item())

# ------------------------------------------------------------
# 8) Optional throughput fallback
# ------------------------------------------------------------
if "throughput_gpu" not in globals():
    throughput_gpu = None
if "throughput_cpu" not in globals():
    throughput_cpu = None

print("\nBootstrap complete.")
print("You can now run the publication extension cell.")

DEVICE: cuda
Loaded cfg from: results_camelot_ids_v2_pc\config.json
RUN_DIR: results_camelot_ids_v2_pc
num_features: 46
GROUP_NAMES: ['rates', 'flags', 'protocols', 'stats', 'other']
Cached arrays loaded:
  X_val : (2631074, 46)
  X_cal : (1389497, 46)
  X_test: (1372896, 46)

Bootstrap complete.
You can now run the publication extension cell.


In [3]:
from pathlib import Path
import json
import math
import gc
import warnings
from copy import deepcopy
from sklearn.metrics import precision_recall_fscore_support
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    log_loss,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore")

# ---------------------------
# Directories
# ---------------------------
PUB_DIR = RUN_DIR / "publication_package"
PUB_FIG_DIR = PUB_DIR / "figures"
PUB_TAB_DIR = PUB_DIR / "tables"
PUB_JSON_DIR = PUB_DIR / "json"
PUB_EXPORT_DIR = PUB_DIR / "export"

for d in [PUB_DIR, PUB_FIG_DIR, PUB_TAB_DIR, PUB_JSON_DIR, PUB_EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Publication package dir:", PUB_DIR)

# ---------------------------
# Safety / config
# ---------------------------
PUB_BOOT_N = 120          # fast but useful CIs
PUB_BOOT_MAX_POOL = 150_000
PUB_RISK_POINTS = 31
PUB_RAPS_ALPHA_DEFAULT = 0.10
PUB_ONNX_OPSET = 18
PUB_VALIDATE_ONNX = True

# ---------------------------
# Helpers
# ---------------------------
def topk_acc_from_probs(y_true: np.ndarray, probs: np.ndarray, k: int) -> float:
    k = int(min(k, probs.shape[1]))
    topk = np.argpartition(probs, -k, axis=1)[:, -k:]
    return float((topk == y_true[:, None]).any(axis=1).mean())

def multiclass_extended_metrics(y_true: np.ndarray, probs: np.ndarray) -> dict:
    y_pred = probs.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
        "top3_accuracy": float(topk_acc_from_probs(y_true, probs, 3)),
        "top5_accuracy": float(topk_acc_from_probs(y_true, probs, 5)),
    }

def binary_extended_metrics(y_true: np.ndarray, probs_2: np.ndarray) -> dict:
    y_pred = probs_2.argmax(axis=1)
    p1 = probs_2[:, 1]
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "f1_malicious": float(f1_score(y_true, y_pred, average="binary", pos_label=1)),
        "precision_malicious": float(precision_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0)),
        "recall_malicious": float(recall_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
    }
    try:
        out["auroc"] = float(roc_auc_score(y_true, p1))
    except Exception:
        out["auroc"] = float("nan")
    try:
        out["auprc"] = float(average_precision_score(y_true, p1))
    except Exception:
        out["auprc"] = float("nan")
    return out

def calibration_stats(y_true: np.ndarray, probs: np.ndarray, n_bins: int = 15) -> dict:
    y_pred = probs.argmax(axis=1)
    conf = probs[np.arange(len(probs)), y_pred]
    acc = (y_pred == y_true).astype(np.float32)
    bins = np.linspace(0.0, 1.0, n_bins + 1)

    ece = 0.0
    mce = 0.0
    rows = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i < n_bins - 1:
            m = (conf >= lo) & (conf < hi)
        else:
            m = (conf >= lo) & (conf <= hi)

        if m.any():
            bin_acc = float(acc[m].mean())
            bin_conf = float(conf[m].mean())
            gap = abs(bin_acc - bin_conf)
            frac = float(m.mean())
            ece += frac * gap
            mce = max(mce, gap)
            rows.append({
                "bin_lo": float(lo),
                "bin_hi": float(hi),
                "count": int(m.sum()),
                "frac": frac,
                "acc": bin_acc,
                "conf": bin_conf,
                "gap": float(gap),
            })
        else:
            rows.append({
                "bin_lo": float(lo),
                "bin_hi": float(hi),
                "count": 0,
                "frac": 0.0,
                "acc": float("nan"),
                "conf": float("nan"),
                "gap": float("nan"),
            })

    return {
        "nll": float(log_loss(y_true, probs, labels=list(range(probs.shape[1])))),
        "ece": float(ece),
        "mce": float(mce),
        "brier": float(np.mean(np.sum((probs - np.eye(probs.shape[1])[y_true]) ** 2, axis=1))),
        "bins": rows,
    }

def plot_reliability(ax, y_true: np.ndarray, probs: np.ndarray, title: str, n_bins: int = 15):
    stats = calibration_stats(y_true, probs, n_bins=n_bins)
    bins = pd.DataFrame(stats["bins"])
    mids = (bins["bin_lo"].values + bins["bin_hi"].values) / 2.0
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    ax.plot(mids, bins["acc"].values, marker="o", label="accuracy")
    ax.plot(mids, bins["conf"].values, marker="s", label="confidence")
    ax.set_title(title)
    ax.set_xlabel("confidence bin")
    ax.set_ylabel("value")
    ax.grid(True, alpha=0.3)
    ax.legend()
    return stats

def stratified_pool_indices(y: np.ndarray, max_n: int, seed: int = 42) -> np.ndarray:
    if len(y) <= max_n:
        return np.arange(len(y))
    rng = np.random.default_rng(seed)
    idxs = []
    classes, counts = np.unique(y, return_counts=True)
    frac = max_n / len(y)
    for c, cnt in zip(classes, counts):
        class_idx = np.where(y == c)[0]
        take = max(1, int(round(cnt * frac)))
        take = min(take, len(class_idx))
        pick = rng.choice(class_idx, size=take, replace=False)
        idxs.append(pick)
    idxs = np.concatenate(idxs)
    if len(idxs) > max_n:
        idxs = rng.choice(idxs, size=max_n, replace=False)
    return np.sort(idxs)

def bootstrap_ci_classification(yf_true, yf_pred, yc_true, yc_pred, yb_true, yb_pred,
                                n_boot=120, max_pool=150_000, seed=42):
    pool = stratified_pool_indices(yf_true, max_pool, seed)
    yf_true_p = yf_true[pool]
    yf_pred_p = yf_pred[pool]
    yc_true_p = yc_true[pool]
    yc_pred_p = yc_pred[pool]
    yb_true_p = yb_true[pool]
    yb_pred_p = yb_pred[pool]

    n = len(pool)
    rng = np.random.default_rng(seed + 123)

    rows = []
    for _ in range(n_boot):
        b = rng.integers(0, n, size=n)
        rows.append({
            "fine_accuracy": float(accuracy_score(yf_true_p[b], yf_pred_p[b])),
            "fine_macro_f1": float(f1_score(yf_true_p[b], yf_pred_p[b], average="macro")),
            "coarse_accuracy": float(accuracy_score(yc_true_p[b], yc_pred_p[b])),
            "coarse_macro_f1": float(f1_score(yc_true_p[b], yc_pred_p[b], average="macro")),
            "bin_f1_malicious": float(f1_score(yb_true_p[b], yb_pred_p[b], average="binary", pos_label=1)),
        })

    dfb = pd.DataFrame(rows)

    out = {}
    for col in dfb.columns:
        vals = dfb[col].values
        out[col] = {
            "mean": float(np.mean(vals)),
            "ci95_lo": float(np.quantile(vals, 0.025)),
            "ci95_hi": float(np.quantile(vals, 0.975)),
        }
    return out, dfb

def save_df_many_formats(df: pd.DataFrame, stem: Path, index: bool = False):
    stem.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(str(stem.with_suffix(".csv")), index=index)
    try:
        df.to_latex(str(stem.with_suffix(".tex")), index=index, float_format="%.4f")
    except Exception:
        pass

def plot_cm(cm: np.ndarray, labels: list, title: str, out_path: Path, normalize: bool = False, figsize=(10, 8)):
    arr = cm.astype(np.float64)
    if normalize:
        denom = arr.sum(axis=1, keepdims=True)
        arr = np.divide(arr, np.clip(denom, 1e-12, None))
    plt.figure(figsize=figsize)
    plt.imshow(arr, aspect="auto")
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.colorbar()
    plt.xticks(np.arange(len(labels)), labels, rotation=90, fontsize=7)
    plt.yticks(np.arange(len(labels)), labels, fontsize=7)
    plt.tight_layout()
    plt.savefig(str(out_path), dpi=220, bbox_inches="tight")
    plt.close()

def build_fresh_model():
    m = CamelotIDSv2(
        n_features=num_features,
        n_fine=NUM_FINE,
        n_coarse=NUM_COARSE,
        group_idxs=GROUP_IDXS,
        cfg=cfg
    ).to(DEVICE)
    m.eval()
    return m

def load_state_into_fresh(state_dict: dict):
    m = build_fresh_model()
    m.load_state_dict(state_dict, strict=True)
    m.eval()
    return m

def apply_ema_shadow_(m: nn.Module, ema_state: dict):
    if ema_state is None:
        return m
    shadow = ema_state.get("shadow", {})
    sd = m.state_dict()
    for k, v in shadow.items():
        if k in sd:
            sd[k].copy_(v.detach().to(sd[k].device, dtype=sd[k].dtype))
    m.load_state_dict(sd, strict=False)
    m.eval()
    return m

@torch.inference_mode()
def eval_candidate_on_val(m: nn.Module, name: str):
    out = predict_logits(m, val_loader)
    probs_f = softmax_np(out["fine"])
    probs_c = softmax_np(out["coarse"])
    probs_b = softmax_np(out["bin"])
    fine_m = metrics_mc(out["yf"], probs_f)
    coarse_m = metrics_mc(out["yc"], probs_c)
    bin_f1 = float(f1_score(out["yb"], probs_b.argmax(axis=1), average="binary", pos_label=1))
    score = float(fine_m["macro_f1"])
    return {
        "name": name,
        "score_val_fine_macro_f1": score,
        "val_fine": fine_m,
        "val_coarse": coarse_m,
        "val_bin_f1_malicious": bin_f1,
    }

def write_markdown(path: Path, text: str):
    path.write_text(text, encoding="utf-8")

# ---------------------------
# 1) Candidate selection from existing checkpoints
# ---------------------------
candidate_rows = []
candidate_models = {}

best_ckpt_path = CKPT_DIR / "best_model.pt"
last_ckpt_path = CKPT_DIR / "last_checkpoint.pt"

if not best_ckpt_path.exists():
    raise FileNotFoundError(f"Missing: {best_ckpt_path}")
if not last_ckpt_path.exists():
    raise FileNotFoundError(f"Missing: {last_ckpt_path}")

best_blob = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
last_blob = torch.load(last_ckpt_path, map_location=DEVICE, weights_only=False)

# Candidate A: raw best checkpoint
cand_best_raw = load_state_into_fresh(best_blob["model_state"])
res_best_raw = eval_candidate_on_val(cand_best_raw, "best_raw_epoch_checkpoint")
candidate_rows.append(res_best_raw)
candidate_models["best_raw_epoch_checkpoint"] = cand_best_raw
print("Candidate:", res_best_raw["name"], "| val fine macro-F1 =", round(res_best_raw["score_val_fine_macro_f1"], 6))

# Candidate B: raw last checkpoint
cand_last_raw = load_state_into_fresh(last_blob["model_state"])
res_last_raw = eval_candidate_on_val(cand_last_raw, "last_raw_checkpoint")
candidate_rows.append(res_last_raw)
candidate_models["last_raw_checkpoint"] = cand_last_raw
print("Candidate:", res_last_raw["name"], "| val fine macro-F1 =", round(res_last_raw["score_val_fine_macro_f1"], 6))

# Candidate C: EMA shadow from last checkpoint applied to the last model
cand_last_ema = load_state_into_fresh(last_blob["model_state"])
cand_last_ema = apply_ema_shadow_(cand_last_ema, last_blob.get("ema_state", None))
res_last_ema = eval_candidate_on_val(cand_last_ema, "last_ema_shadow_checkpoint")
candidate_rows.append(res_last_ema)
candidate_models["last_ema_shadow_checkpoint"] = cand_last_ema
print("Candidate:", res_last_ema["name"], "| val fine macro-F1 =", round(res_last_ema["score_val_fine_macro_f1"], 6))

candidate_df = pd.DataFrame([
    {
        "candidate": r["name"],
        "val_fine_macro_f1": r["score_val_fine_macro_f1"],
        "val_fine_accuracy": r["val_fine"]["accuracy"],
        "val_coarse_macro_f1": r["val_coarse"]["macro_f1"],
        "val_bin_f1_malicious": r["val_bin_f1_malicious"],
    }
    for r in candidate_rows
]).sort_values("val_fine_macro_f1", ascending=False).reset_index(drop=True)

save_df_many_formats(candidate_df, PUB_TAB_DIR / "candidate_model_selection", index=False)
best_name = candidate_df.iloc[0]["candidate"]
pub_model = candidate_models[best_name]
pub_model.eval()

print("\nChosen publication model:", best_name)

# Save chosen model state for publication package
torch.save(
    {
        "candidate_name": best_name,
        "state_dict": {k: v.detach().cpu().clone() for k, v in pub_model.state_dict().items()},
        "num_features": int(num_features),
        "num_fine": int(NUM_FINE),
        "num_coarse": int(NUM_COARSE),
    },
    PUB_EXPORT_DIR / "publication_best_model.pt"
)

# ---------------------------
# 2) Fresh logits for chosen candidate
# ---------------------------
pub_signature = {
    "publication_candidate": best_name,
    "source_training_signature": get_training_signature(),
}

pub_test_out = predict_logits_cached("publication_test_" + best_name, pub_model, test_loader, signature=pub_signature)
pub_cal_out  = predict_logits_cached("publication_cal_" + best_name,  pub_model, cal_loader,  signature=pub_signature)

# ---------------------------
# 3) Fresh calibration (fast, no retraining)
# ---------------------------
pub_temp_f = fit_temperature_np(pub_cal_out["fine"], pub_cal_out["yf"]) if cfg.TEMPERATURE_SCALE else 1.0
pub_temp_c = fit_temperature_np(pub_cal_out["coarse"], pub_cal_out["yc"]) if cfg.TEMPERATURE_SCALE else 1.0
pub_temp_b = fit_temperature_np(pub_cal_out["bin"], pub_cal_out["yb"]) if cfg.TEMPERATURE_SCALE else 1.0

print("Refit temperatures:", {"fine": pub_temp_f, "coarse": pub_temp_c, "bin": pub_temp_b})

# Raw probs
pub_test_probs_f_raw = softmax_np(pub_test_out["fine"])
pub_test_probs_c_raw = softmax_np(pub_test_out["coarse"])
pub_test_probs_b_raw = softmax_np(pub_test_out["bin"])

# Calibrated probs
pub_test_probs_f = softmax_np(pub_test_out["fine"] / max(pub_temp_f, 1e-6))
pub_test_probs_c = softmax_np(pub_test_out["coarse"] / max(pub_temp_c, 1e-6))
pub_test_probs_b = softmax_np(pub_test_out["bin"] / max(pub_temp_b, 1e-6))

pub_yf = pub_test_out["yf"]
pub_yc = pub_test_out["yc"]
pub_yb = pub_test_out["yb"]

pub_pred_f = pub_test_probs_f.argmax(axis=1)
pub_pred_c = pub_test_probs_c.argmax(axis=1)
pub_pred_b = pub_test_probs_b.argmax(axis=1)

# ---------------------------
# 4) Core publication metrics
# ---------------------------
metrics_fine = multiclass_extended_metrics(pub_yf, pub_test_probs_f)
metrics_coarse = multiclass_extended_metrics(pub_yc, pub_test_probs_c)
metrics_bin = binary_extended_metrics(pub_yb, pub_test_probs_b)

cal_f_raw = calibration_stats(pub_yf, pub_test_probs_f_raw, n_bins=cfg.N_BINS_ECE)
cal_f_cal = calibration_stats(pub_yf, pub_test_probs_f, n_bins=cfg.N_BINS_ECE)

cal_c_raw = calibration_stats(pub_yc, pub_test_probs_c_raw, n_bins=cfg.N_BINS_ECE)
cal_c_cal = calibration_stats(pub_yc, pub_test_probs_c, n_bins=cfg.N_BINS_ECE)

cal_b_raw = calibration_stats(pub_yb, pub_test_probs_b_raw, n_bins=cfg.N_BINS_ECE)
cal_b_cal = calibration_stats(pub_yb, pub_test_probs_b, n_bins=cfg.N_BINS_ECE)

ci_summary, ci_boot_df = bootstrap_ci_classification(
    pub_yf, pub_pred_f,
    pub_yc, pub_pred_c,
    pub_yb, pub_pred_b,
    n_boot=PUB_BOOT_N,
    max_pool=PUB_BOOT_MAX_POOL,
    seed=cfg.SEED
)

with pd.ExcelWriter(PUB_TAB_DIR / "publication_tables.xlsx") as writer:
    candidate_df.to_excel(writer, sheet_name="candidate_selection", index=False)
    pd.DataFrame([metrics_fine]).to_excel(writer, sheet_name="fine_metrics", index=False)
    pd.DataFrame([metrics_coarse]).to_excel(writer, sheet_name="coarse_metrics", index=False)
    pd.DataFrame([metrics_bin]).to_excel(writer, sheet_name="binary_metrics", index=False)
    pd.DataFrame(ci_summary).T.reset_index().rename(columns={"index": "metric"}).to_excel(writer, sheet_name="bootstrap_ci", index=False)

# ---------------------------
# 5) Per-class and per-coarse tables
# ---------------------------
fine_prec, fine_rec, fine_f1, fine_sup = precision_recall_fscore_support(
    pub_yf, pub_pred_f, labels=np.arange(NUM_FINE), zero_division=0
)
fine_table = pd.DataFrame({
    "fine_class": FINE_CLASS_NAMES,
    "support": fine_sup.astype(int),
    "precision": fine_prec,
    "recall": fine_rec,
    "f1": fine_f1,
    "coarse_class": [COARSE_NAMES[fine_to_coarse[i]] for i in range(NUM_FINE)],
}).sort_values(["f1", "support"], ascending=[True, False]).reset_index(drop=True)

coarse_prec, coarse_rec, coarse_f1, coarse_sup = precision_recall_fscore_support(
    pub_yc, pub_pred_c, labels=np.arange(NUM_COARSE), zero_division=0
)
coarse_table = pd.DataFrame({
    "coarse_class": COARSE_NAMES,
    "support": coarse_sup.astype(int),
    "precision": coarse_prec,
    "recall": coarse_rec,
    "f1": coarse_f1,
}).sort_values("f1", ascending=True).reset_index(drop=True)

save_df_many_formats(fine_table, PUB_TAB_DIR / "fine_per_class_metrics", index=False)
save_df_many_formats(coarse_table, PUB_TAB_DIR / "coarse_per_class_metrics", index=False)
save_df_many_formats(ci_boot_df, PUB_TAB_DIR / "bootstrap_draws_key_metrics", index=False)

# ---------------------------
# 6) Confusion matrices
# ---------------------------
cm_fine = confusion_matrix(pub_yf, pub_pred_f, labels=np.arange(NUM_FINE))
cm_coarse = confusion_matrix(pub_yc, pub_pred_c, labels=np.arange(NUM_COARSE))

plot_cm(cm_fine, FINE_CLASS_NAMES, "Fine 34-class Confusion Matrix (raw counts)", PUB_FIG_DIR / "cm_fine_counts.png", normalize=False, figsize=(14, 12))
plot_cm(cm_fine, FINE_CLASS_NAMES, "Fine 34-class Confusion Matrix (row-normalized)", PUB_FIG_DIR / "cm_fine_normalized.png", normalize=True, figsize=(14, 12))
plot_cm(cm_coarse, COARSE_NAMES, "Coarse 8-class Confusion Matrix (raw counts)", PUB_FIG_DIR / "cm_coarse_counts.png", normalize=False, figsize=(8, 6))
plot_cm(cm_coarse, COARSE_NAMES, "Coarse 8-class Confusion Matrix (row-normalized)", PUB_FIG_DIR / "cm_coarse_normalized.png", normalize=True, figsize=(8, 6))

# ---------------------------
# 7) Reliability diagrams
# ---------------------------
fig, axs = plt.subplots(2, 3, figsize=(16, 9))

plot_reliability(axs[0, 0], pub_yf, pub_test_probs_f_raw, "Fine reliability (before)")
plot_reliability(axs[1, 0], pub_yf, pub_test_probs_f,     "Fine reliability (after)")

plot_reliability(axs[0, 1], pub_yc, pub_test_probs_c_raw, "Coarse reliability (before)")
plot_reliability(axs[1, 1], pub_yc, pub_test_probs_c,     "Coarse reliability (after)")

plot_reliability(axs[0, 2], pub_yb, pub_test_probs_b_raw, "Binary reliability (before)")
plot_reliability(axs[1, 2], pub_yb, pub_test_probs_b,     "Binary reliability (after)")

plt.tight_layout()
plt.savefig(str(PUB_FIG_DIR / "reliability_diagrams_all.png"), dpi=220, bbox_inches="tight")
plt.close()

# ---------------------------
# 8) ROC / PR for binary malicious-vs-benign
# ---------------------------
try:
    fpr, tpr, _ = roc_curve(pub_yb, pub_test_probs_b[:, 1])
    prec_curve, rec_curve, _ = precision_recall_curve(pub_yb, pub_test_probs_b[:, 1])

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUROC = {metrics_bin['auroc']:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Binary ROC (Malicious vs Benign)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(str(PUB_FIG_DIR / "binary_roc.png"), dpi=220, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(6, 5))
    plt.plot(rec_curve, prec_curve, label=f"AUPRC = {metrics_bin['auprc']:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Binary Precision-Recall (Malicious vs Benign)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(str(PUB_FIG_DIR / "binary_pr.png"), dpi=220, bbox_inches="tight")
    plt.close()
except Exception as e:
    print("ROC/PR plot skipped:", e)

# ---------------------------
# 9) Per-class figures
# ---------------------------
plt.figure(figsize=(14, 6))
tmp = fine_table.sort_values("f1", ascending=True)
plt.bar(np.arange(len(tmp)), tmp["f1"].values)
plt.xticks(np.arange(len(tmp)), tmp["fine_class"].values, rotation=90, fontsize=7)
plt.ylabel("F1")
plt.title("Per-class F1 (fine classes, sorted)")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(str(PUB_FIG_DIR / "per_class_f1_sorted.png"), dpi=220, bbox_inches="tight")
plt.close()

plt.figure(figsize=(8, 6))
plt.scatter(np.log10(np.maximum(fine_table["support"].values, 1)), fine_table["f1"].values)
for _, r in fine_table.nsmallest(10, "f1").iterrows():
    plt.annotate(r["fine_class"], (np.log10(max(r["support"], 1)), r["f1"]), fontsize=8)
plt.xlabel("log10(support)")
plt.ylabel("F1")
plt.title("Class support vs F1")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(PUB_FIG_DIR / "support_vs_f1.png"), dpi=220, bbox_inches="tight")
plt.close()

# ---------------------------
# 10) Training curves (from saved epoch history)
# ---------------------------
hist_path = RUN_DIR / "epoch_history.csv"
if hist_path.exists():
    hist_df = pd.read_csv(hist_path)

    plt.figure(figsize=(10, 5))
    plt.plot(hist_df["epoch"], hist_df["train_loss"], label="train_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training loss")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(str(PUB_FIG_DIR / "training_loss.png"), dpi=220, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(10, 5))
    plt.plot(hist_df["epoch"], hist_df["val_fine_macro_f1"], label="val fine macro-F1")
    plt.plot(hist_df["epoch"], hist_df["val_coarse_macro_f1"], label="val coarse macro-F1")
    plt.plot(hist_df["epoch"], hist_df["val_bin_f1"], label="val binary F1")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Validation curves")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(str(PUB_FIG_DIR / "validation_curves.png"), dpi=220, bbox_inches="tight")
    plt.close()

# ---------------------------
# 11) Selective prediction / risk-coverage
# ---------------------------
conf_f = pub_test_probs_f.max(axis=1)
thresholds = np.unique(np.quantile(conf_f, np.linspace(0.0, 0.995, PUB_RISK_POINTS)))

sel_rows = []
for th in thresholds:
    m = conf_f >= th
    if m.sum() == 0:
        continue
    acc = float((pub_pred_f[m] == pub_yf[m]).mean())
    mf1 = float(f1_score(pub_yf[m], pub_pred_f[m], average="macro"))
    sel_rows.append({
        "threshold": float(th),
        "coverage": float(m.mean()),
        "accuracy": float(acc),
        "risk_1_minus_accuracy": float(1.0 - acc),
        "macro_f1": float(mf1),
    })

sel_df = pd.DataFrame(sel_rows).sort_values("coverage")
save_df_many_formats(sel_df, PUB_TAB_DIR / "confidence_selective_prediction_curve", index=False)

plt.figure(figsize=(7, 5))
plt.plot(sel_df["coverage"], sel_df["risk_1_minus_accuracy"], marker="o")
plt.xlabel("Coverage")
plt.ylabel("Risk = 1 - accuracy")
plt.title("Selective prediction risk-coverage (fine top-1 confidence)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(PUB_FIG_DIR / "risk_coverage_confidence.png"), dpi=220, bbox_inches="tight")
plt.close()

# ---------------------------
# 12) Fresh RAPS using chosen model + calibrated probs
# ---------------------------
def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    n = len(scores)
    if n == 0:
        return 1.0
    q_level = math.ceil((n + 1) * (1 - alpha)) / n
    return float(np.quantile(scores, min(q_level, 1.0), method="higher"))

def raps_scores_true(probs: np.ndarray, y_true: np.ndarray, k_reg: int, lam: float) -> np.ndarray:
    n, C = probs.shape
    idx_sorted = np.argsort(-probs, axis=1)
    probs_sorted = np.take_along_axis(probs, idx_sorted, axis=1)
    cumsum = np.cumsum(probs_sorted, axis=1)

    inv = np.empty_like(idx_sorted)
    rows = np.arange(n)[:, None]
    inv[rows, idx_sorted] = np.arange(C)[None, :]
    rank0 = inv[rows[:, 0], y_true]
    rank1 = rank0 + 1

    score = cumsum[rows[:, 0], rank0] + lam * np.maximum(rank1 - k_reg, 0)
    return score.astype(np.float32)

def raps_predict_k(probs: np.ndarray, q: np.ndarray, k_reg: int, lam: float) -> np.ndarray:
    n, C = probs.shape
    idx_sorted = np.argsort(-probs, axis=1)
    probs_sorted = np.take_along_axis(probs, idx_sorted, axis=1)
    cumsum = np.cumsum(probs_sorted, axis=1)
    ranks = np.arange(1, C + 1, dtype=np.float32)[None, :]
    reg = lam * np.maximum(ranks - float(k_reg), 0.0)
    score_k = cumsum + reg
    ok = score_k <= q[:, None]
    k_star = ok.sum(axis=1).astype(np.int64)
    return np.maximum(k_star, 1)

def raps_eval(probs: np.ndarray, y_true: np.ndarray, q_row: np.ndarray, k_reg: int, lam: float) -> dict:
    n, C = probs.shape
    idx_sorted = np.argsort(-probs, axis=1)

    inv = np.empty_like(idx_sorted)
    rows = np.arange(n)[:, None]
    inv[rows, idx_sorted] = np.arange(C)[None, :]
    rank0 = inv[rows[:, 0], y_true]

    k_star = raps_predict_k(probs, q_row, k_reg, lam)
    covered = (rank0 < k_star).astype(np.float32)
    return {"coverage": float(covered.mean()), "avg_set_size": float(k_star.mean())}

pub_cal_probs_f = softmax_np(pub_cal_out["fine"] / max(pub_temp_f, 1e-6))
pub_cal_probs_c = softmax_np(pub_cal_out["coarse"] / max(pub_temp_c, 1e-6))

raps_calib_pub = {
    "mode": cfg.RAPS_MONDRIAN,
    "k_reg": int(cfg.RAPS_KREG),
    "lambda": float(cfg.RAPS_LAMBDA),
    "alphas": [float(a) for a in cfg.ALPHAS],
    "q_global": {},
    "q_by_pred_coarse": {},
    "min_group_n": 200
}

scores_global = raps_scores_true(pub_cal_probs_f, pub_cal_out["yf"], cfg.RAPS_KREG, cfg.RAPS_LAMBDA)
g_hat_cal = pub_cal_probs_c.argmax(axis=1)
scores_by_g = {g: scores_global[g_hat_cal == g] for g in range(NUM_COARSE)}

for a in cfg.ALPHAS:
    qg = conformal_quantile(scores_global, a)
    raps_calib_pub["q_global"][str(a)] = float(qg)
    q_by = []
    for g in range(NUM_COARSE):
        sg = scores_by_g.get(g, np.array([], dtype=np.float32))
        if len(sg) >= raps_calib_pub["min_group_n"]:
            q_by.append(conformal_quantile(sg, a))
        else:
            q_by.append(qg)
    raps_calib_pub["q_by_pred_coarse"][str(a)] = [float(x) for x in q_by]

raps_results_pub = {}
for a in cfg.ALPHAS:
    q_by = np.array(raps_calib_pub["q_by_pred_coarse"][str(a)], dtype=np.float32)
    q_row = q_by[pub_test_probs_c.argmax(axis=1)]
    raps_results_pub[str(a)] = raps_eval(pub_test_probs_f, pub_yf, q_row, cfg.RAPS_KREG, cfg.RAPS_LAMBDA)

# singleton risk / coverage table
def singleton_risk_coverage(y_true, probs_f, probs_c, raps_calib, alpha, k_reg, lam, benign_id=0):
    q_by = np.array(raps_calib["q_by_pred_coarse"][str(alpha)], dtype=np.float32)
    g_hat = probs_c.argmax(axis=1)
    q_row = q_by[g_hat]
    k_star = raps_predict_k(probs_f, q_row, k_reg, lam)

    single = (k_star == 1)
    y_pred = probs_f.argmax(axis=1)

    cov = float(single.mean())
    if single.any():
        acc = float((y_pred[single] == y_true[single]).mean())
        mf1 = float(f1_score(y_true[single], y_pred[single], average="macro"))
    else:
        acc, mf1 = float("nan"), float("nan")

    benign_mask = (y_true == benign_id) & single
    benign_fpr = float((y_pred[benign_mask] != benign_id).mean()) if benign_mask.any() else float("nan")

    return {
        "alpha": float(alpha),
        "coverage_singleton": cov,
        "risk_1_minus_acc": float(1.0 - acc),
        "macro_f1_on_decisions": mf1,
        "benign_FPR_on_decisions": benign_fpr,
        "avg_set_size": float(k_star.mean()),
    }

raps_sel_rows = [
    singleton_risk_coverage(
        pub_yf, pub_test_probs_f, pub_test_probs_c,
        raps_calib_pub, a, cfg.RAPS_KREG, cfg.RAPS_LAMBDA, benign_fine_id
    )
    for a in cfg.ALPHAS
]
raps_sel_df = pd.DataFrame(raps_sel_rows)

save_df_many_formats(pd.DataFrame(raps_results_pub).T.reset_index().rename(columns={"index": "alpha"}), PUB_TAB_DIR / "raps_results", index=False)
save_df_many_formats(raps_sel_df, PUB_TAB_DIR / "raps_singleton_risk_coverage", index=False)

# RAPS set size histogram for default alpha
alpha0 = float(PUB_RAPS_ALPHA_DEFAULT)
if str(alpha0) in raps_calib_pub["q_by_pred_coarse"]:
    q_by = np.array(raps_calib_pub["q_by_pred_coarse"][str(alpha0)], dtype=np.float32)
    q_row = q_by[pub_test_probs_c.argmax(axis=1)]
    k_star = raps_predict_k(pub_test_probs_f, q_row, cfg.RAPS_KREG, cfg.RAPS_LAMBDA)

    plt.figure(figsize=(7, 5))
    vals, counts = np.unique(k_star, return_counts=True)
    plt.bar(vals, counts / counts.sum())
    plt.xlabel("RAPS set size")
    plt.ylabel("Fraction")
    plt.title(f"RAPS set-size distribution (alpha={alpha0})")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(PUB_FIG_DIR / f"raps_set_size_alpha_{alpha0:.2f}.png"), dpi=220, bbox_inches="tight")
    plt.close()

# ---------------------------
# 13) Throughput summary reuse
# ---------------------------
pub_throughput = {
    "gpu": throughput_gpu if "throughput_gpu" in globals() else None,
    "cpu": throughput_cpu if "throughput_cpu" in globals() else None,
}

# ---------------------------
# 14) Safe export (fixes old export problems)
# ---------------------------
class PubLogitsWrapper(nn.Module):
    def __init__(self, base: nn.Module):
        super().__init__()
        self.base = base

    def forward(self, x: torch.Tensor):
        o = self.base(x)
        return o["logits_fine"], o["logits_coarse"], o["logits_bin"]

export_status = {
    "torchscript": {"ok": False, "path": None, "error": None},
    "onnx": {"ok": False, "path": None, "error": None, "validated": False},
}

pub_model_cpu = build_fresh_model().to("cpu").eval()
pub_model_cpu.load_state_dict({k: v.detach().cpu() for k, v in pub_model.state_dict().items()}, strict=True)
wrapper_cpu = PubLogitsWrapper(pub_model_cpu).eval()

# TorchScript via script (safer than trace here)
try:
    scripted = torch.jit.script(wrapper_cpu)
    ts_path = PUB_EXPORT_DIR / "camelot_ids_v2_publication_torchscript.pt"
    scripted.save(str(ts_path))
    export_status["torchscript"]["ok"] = True
    export_status["torchscript"]["path"] = str(ts_path)
    print("TorchScript export OK:", ts_path)
except Exception as e:
    export_status["torchscript"]["error"] = str(e)
    print("TorchScript export skipped:", e)

# ONNX opset 18
try:
    onnx_path = PUB_EXPORT_DIR / "camelot_ids_v2_publication.onnx"
    dummy = torch.randn(1, num_features, dtype=torch.float32)
    torch.onnx.export(
        wrapper_cpu,
        dummy,
        str(onnx_path),
        input_names=["input"],
        output_names=["logits_fine", "logits_coarse", "logits_bin"],
        dynamic_axes={
            "input": {0: "batch"},
            "logits_fine": {0: "batch"},
            "logits_coarse": {0: "batch"},
            "logits_bin": {0: "batch"},
        },
        opset_version=PUB_ONNX_OPSET,
    )
    export_status["onnx"]["ok"] = True
    export_status["onnx"]["path"] = str(onnx_path)
    print("ONNX export OK:", onnx_path)

    if PUB_VALIDATE_ONNX:
        try:
            import onnxruntime as ort
            sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
            x_small = X_test[:8].astype(np.float32, copy=False)

            with torch.inference_mode():
                pt_out = wrapper_cpu(torch.from_numpy(x_small))
                pt_f = pt_out[0].numpy()
                pt_c = pt_out[1].numpy()
                pt_b = pt_out[2].numpy()

            onnx_out = sess.run(None, {"input": x_small})
            ok_f = np.allclose(pt_f, onnx_out[0], atol=1e-4, rtol=1e-4)
            ok_c = np.allclose(pt_c, onnx_out[1], atol=1e-4, rtol=1e-4)
            ok_b = np.allclose(pt_b, onnx_out[2], atol=1e-4, rtol=1e-4)

            export_status["onnx"]["validated"] = bool(ok_f and ok_c and ok_b)
            export_status["onnx"]["validation_allclose"] = {
                "fine": bool(ok_f),
                "coarse": bool(ok_c),
                "bin": bool(ok_b),
            }
            print("ONNX validation:", export_status["onnx"]["validation_allclose"])
        except Exception as e:
            export_status["onnx"]["validated"] = False
            export_status["onnx"]["validation_error"] = str(e)
            print("ONNX validation skipped:", e)

except Exception as e:
    export_status["onnx"]["error"] = str(e)
    print("ONNX export failed:", e)

# Robust inference CLI (TorchScript first, ONNX fallback)
pub_cli = r'''
import argparse
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

def sanitize_frame(df: pd.DataFrame, clip_inf: float = 1e9) -> pd.DataFrame:
    df = df.replace({"Infinity": np.inf, "inf": np.inf, "+inf": np.inf, "-inf": -np.inf,
                     "NaN": np.nan, "nan": np.nan, "": np.nan})
    df = df.replace([np.inf, -np.inf], np.nan)
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        df[num_cols] = df[num_cols].clip(-clip_inf, clip_inf)
    return df

def transform_df(df_in: pd.DataFrame, feature_cols, meta_obj):
    cfg = meta_obj["cfg"]
    X = sanitize_frame(df_in[feature_cols].copy(), cfg["CLIP_INF"])
    for c in meta_obj["meta"]["numeric_like_obj"]:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")
    num_cols = meta_obj["meta"]["num_cols"]
    X = X[num_cols]
    Xn = meta_obj["num_imputer"].transform(X).astype(np.float32)
    Xn = meta_obj["scaler"].transform(Xn).astype(np.float32)
    Xn = np.clip(Xn, -cfg["POST_SCALE_CLIP"], cfg["POST_SCALE_CLIP"]).astype(np.float32)
    Xn = np.nan_to_num(Xn, nan=0.0, posinf=cfg["POST_SCALE_CLIP"], neginf=-cfg["POST_SCALE_CLIP"]).astype(np.float32)
    return Xn

def softmax_np(x):
    x = x - x.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / np.clip(ex.sum(axis=1, keepdims=True), 1e-12, None)

def raps_predict_set(probs_f, probs_c, raps_calib, alpha):
    k_reg = int(raps_calib["k_reg"])
    lam = float(raps_calib["lambda"])
    q_by = np.array(raps_calib["q_by_pred_coarse"][str(alpha)], dtype=np.float32)
    g_hat = probs_c.argmax(axis=1)
    q_row = q_by[g_hat]

    idx_sorted = np.argsort(-probs_f, axis=1)
    probs_sorted = np.take_along_axis(probs_f, idx_sorted, axis=1)
    cumsum = np.cumsum(probs_sorted, axis=1)
    C = probs_f.shape[1]
    ranks = np.arange(1, C + 1, dtype=np.float32)[None, :]
    reg = lam * np.maximum(ranks - float(k_reg), 0.0)
    score_k = cumsum + reg
    ok = score_k <= q_row[:, None]
    k_star = np.maximum(ok.sum(axis=1).astype(np.int64), 1)

    sets = []
    for i in range(len(probs_f)):
        sets.append(idx_sorted[i, :k_star[i]])
    return sets, k_star

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--meta_joblib", type=str, required=True)
    ap.add_argument("--input_csv", type=str, required=True)
    ap.add_argument("--out_csv", type=str, default="preds_publication.csv")
    ap.add_argument("--torchscript", type=str, default="")
    ap.add_argument("--onnx", type=str, default="")
    ap.add_argument("--device", type=str, default="cpu")
    ap.add_argument("--alpha", type=float, default=0.10)
    args = ap.parse_args()

    meta = joblib.load(args.meta_joblib)
    feature_cols = meta["FEATURE_COLS"]
    fine_names = meta["fine_class_names"]
    coarse_names = meta["coarse_names"]
    temp = meta.get("temperature", {"fine": 1.0, "coarse": 1.0, "bin": 1.0})
    raps_calib = meta.get("raps_calib_publication", meta.get("raps_calib", None))
    if raps_calib is None:
        raise RuntimeError("No RAPS calibration found in meta joblib.")

    df = pd.read_csv(args.input_csv, low_memory=False)
    X = transform_df(df, feature_cols, meta)

    logits_f = logits_c = logits_b = None

    if args.torchscript:
        import torch
        model = torch.jit.load(args.torchscript, map_location=args.device)
        model.eval()
        xb = torch.from_numpy(X).to(args.device)
        with torch.inference_mode():
            logits_f, logits_c, logits_b = model(xb)
            logits_f = logits_f.cpu().numpy()
            logits_c = logits_c.cpu().numpy()
            logits_b = logits_b.cpu().numpy()
    elif args.onnx:
        import onnxruntime as ort
        sess = ort.InferenceSession(args.onnx, providers=["CPUExecutionProvider"])
        logits_f, logits_c, logits_b = sess.run(None, {"input": X.astype(np.float32)})
    else:
        raise RuntimeError("Provide either --torchscript or --onnx")

    pf = softmax_np(logits_f / max(float(temp["fine"]), 1e-6))
    pc = softmax_np(logits_c / max(float(temp["coarse"]), 1e-6))
    pb = softmax_np(logits_b / max(float(temp["bin"]), 1e-6))

    top_f = pf.argmax(axis=1)
    top_conf = pf[np.arange(len(pf)), top_f]
    top_c = pc.argmax(axis=1)
    top_b = pb.argmax(axis=1)

    sets, k_star = raps_predict_set(pf, pc, raps_calib, float(args.alpha))
    set_str = [",".join([fine_names[j] for j in s]) for s in sets]

    out = pd.DataFrame({
        "top1_fine": [fine_names[i] for i in top_f],
        "top1_conf": top_conf,
        "pred_coarse": [coarse_names[i] for i in top_c],
        "pred_bin": ["Malicious" if i == 1 else "Benign" for i in top_b],
        "raps_set_size": k_star,
        "raps_set": set_str,
    })
    out.to_csv(args.out_csv, index=False)
    print("Saved:", args.out_csv)

if __name__ == "__main__":
    main()
'''
(PUB_EXPORT_DIR / "camelot_infer_publication.py").write_text(pub_cli, encoding="utf-8")

# ---------------------------
# 15) Save publication meta joblib
# ---------------------------
pub_meta = joblib.load(export_dir / "preprocess_and_meta.joblib")
pub_meta["temperature"] = {"fine": float(pub_temp_f), "coarse": float(pub_temp_c), "bin": float(pub_temp_b)}
pub_meta["raps_calib_publication"] = raps_calib_pub
joblib.dump(pub_meta, PUB_EXPORT_DIR / "preprocess_and_meta_publication.joblib")

# ---------------------------
# 16) Stream results reuse (fast)
# ---------------------------
stream_summary = None
stream_json = RUN_DIR / "stream_results.json"
if stream_json.exists():
    stream_summary = load_json(stream_json, None)

# ---------------------------
# 17) Publication summary files
# ---------------------------
publication_summary = {
    "chosen_publication_model": best_name,
    "candidate_selection": candidate_df.to_dict(orient="records"),
    "fine_metrics": metrics_fine,
    "coarse_metrics": metrics_coarse,
    "binary_metrics": metrics_bin,
    "fine_calibration_before": {k: v for k, v in cal_f_raw.items() if k != "bins"},
    "fine_calibration_after":  {k: v for k, v in cal_f_cal.items() if k != "bins"},
    "coarse_calibration_before": {k: v for k, v in cal_c_raw.items() if k != "bins"},
    "coarse_calibration_after":  {k: v for k, v in cal_c_cal.items() if k != "bins"},
    "binary_calibration_before": {k: v for k, v in cal_b_raw.items() if k != "bins"},
    "binary_calibration_after":  {k: v for k, v in cal_b_cal.items() if k != "bins"},
    "bootstrap_ci_key_metrics": ci_summary,
    "raps_results": raps_results_pub,
    "raps_singleton_risk_coverage": raps_sel_df.to_dict(orient="records"),
    "throughput": pub_throughput,
    "stream_results_reused": stream_summary,
    "export_status": export_status,
    "num_test_samples": int(len(pub_yf)),
    "num_features": int(num_features),
    "num_fine": int(NUM_FINE),
    "num_coarse": int(NUM_COARSE),
}

with open(PUB_JSON_DIR / "publication_summary.json", "w", encoding="utf-8") as f:
    json.dump(publication_summary, f, indent=2)

summary_md = f"""
# CAMELOT-IDS v2 — Publication Summary

## Chosen publication model
- Candidate: **{best_name}**

## Fine (34-class)
- Accuracy: **{metrics_fine['accuracy']:.6f}**
- Balanced accuracy: **{metrics_fine['balanced_accuracy']:.6f}**
- Macro-F1: **{metrics_fine['macro_f1']:.6f}**
- Weighted-F1: **{metrics_fine['weighted_f1']:.6f}**
- Macro-Precision: **{metrics_fine['macro_precision']:.6f}**
- Macro-Recall: **{metrics_fine['macro_recall']:.6f}**
- MCC: **{metrics_fine['mcc']:.6f}**
- Cohen's kappa: **{metrics_fine['kappa']:.6f}**
- Top-3 accuracy: **{metrics_fine['top3_accuracy']:.6f}**
- Top-5 accuracy: **{metrics_fine['top5_accuracy']:.6f}**

## Coarse (8-class)
- Accuracy: **{metrics_coarse['accuracy']:.6f}**
- Balanced accuracy: **{metrics_coarse['balanced_accuracy']:.6f}**
- Macro-F1: **{metrics_coarse['macro_f1']:.6f}**
- Weighted-F1: **{metrics_coarse['weighted_f1']:.6f}**
- MCC: **{metrics_coarse['mcc']:.6f}**
- Cohen's kappa: **{metrics_coarse['kappa']:.6f}**

## Binary (malicious vs benign)
- Accuracy: **{metrics_bin['accuracy']:.6f}**
- Balanced accuracy: **{metrics_bin['balanced_accuracy']:.6f}**
- F1 (malicious): **{metrics_bin['f1_malicious']:.6f}**
- Precision (malicious): **{metrics_bin['precision_malicious']:.6f}**
- Recall (malicious): **{metrics_bin['recall_malicious']:.6f}**
- AUROC: **{metrics_bin['auroc']:.6f}**
- AUPRC: **{metrics_bin['auprc']:.6f}**

## Calibration (fine)
- Before: NLL={cal_f_raw['nll']:.6f}, ECE={cal_f_raw['ece']:.6f}, Brier={cal_f_raw['brier']:.6f}
- After:  NLL={cal_f_cal['nll']:.6f}, ECE={cal_f_cal['ece']:.6f}, Brier={cal_f_cal['brier']:.6f}

## RAPS
{json.dumps(raps_results_pub, indent=2)}

## Export status
{json.dumps(export_status, indent=2)}

## Files generated
- Figures: `{PUB_FIG_DIR}`
- Tables: `{PUB_TAB_DIR}`
- JSON: `{PUB_JSON_DIR}`
- Export: `{PUB_EXPORT_DIR}`
"""
write_markdown(PUB_DIR / "publication_summary.md", summary_md)

print("\nSaved publication summary:", PUB_JSON_DIR / "publication_summary.json")
print("Saved markdown summary:", PUB_DIR / "publication_summary.md")
print("Publication package complete at:", PUB_DIR)

Publication package dir: results_camelot_ids_v2_pc\publication_package
Candidate: best_raw_epoch_checkpoint | val fine macro-F1 = 0.770156
Candidate: last_raw_checkpoint | val fine macro-F1 = 0.769814
Candidate: last_ema_shadow_checkpoint | val fine macro-F1 = 0.769492

Chosen publication model: best_raw_epoch_checkpoint
Loading cached logits: publication_test_best_raw_epoch_checkpoint
Loading cached logits: publication_cal_best_raw_epoch_checkpoint
Refit temperatures: {'fine': 1.0, 'coarse': 1.0, 'bin': 1.0}
TorchScript export skipped: 
getattr's second argument must be a string literal:
  File "C:\Users\HaseebWajid\AppData\Local\Temp\ipykernel_17984\859139402.py", line 352
        group_reps = []
        for g in range(self.num_groups):
            idxs_t = getattr(self, f"group_idx_{g}")
                     ~~~~~~~~~~~~~~~~~~~~~~~~~~~~ <--- HERE
            t = tok.index_select(dim=1, index=idxs_t)
            t = t + self.group_type.weight[g].view(1, 1, -1)

[torch.onnx] Obtain mo

In [6]:
# ============================================================
# EXPORT PATCH FOR PUBLICATION PACKAGE
# Run AFTER the publication extension cell.
# This fixes the remaining export issues without retraining.
# ============================================================

from pathlib import Path
import json
import numpy as np
import torch
import torch.nn as nn

PATCH_EXPORT_DIR = PUB_EXPORT_DIR
PATCH_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

class PubLogitsWrapper(nn.Module):
    def __init__(self, base: nn.Module):
        super().__init__()
        self.base = base

    def forward(self, x: torch.Tensor):
        o = self.base(x)
        return o["logits_fine"], o["logits_coarse"], o["logits_bin"]

# build CPU export wrapper from chosen publication model
pub_model_cpu = build_fresh_model().to("cpu").eval()
pub_model_cpu.load_state_dict({k: v.detach().cpu() for k, v in pub_model.state_dict().items()}, strict=True)
wrapper_cpu = PubLogitsWrapper(pub_model_cpu).eval()

export_patch_status = {
    "torchscript_trace": {"ok": False, "path": None, "error": None},
    "onnx_static_batch1": {"ok": False, "path": None, "validated": False, "error": None},
}

# ------------------------------------------------------------
# 1) TorchScript via TRACE, not SCRIPT
#    This avoids the getattr string-literal scripting error.
# ------------------------------------------------------------
try:
    example = torch.randn(1, num_features, dtype=torch.float32)
    traced = torch.jit.trace(wrapper_cpu, example, check_trace=False)
    ts_trace_path = PATCH_EXPORT_DIR / "camelot_ids_v2_publication_torchscript_trace.pt"
    traced.save(str(ts_trace_path))
    export_patch_status["torchscript_trace"]["ok"] = True
    export_patch_status["torchscript_trace"]["path"] = str(ts_trace_path)
    print("TorchScript TRACE export OK:", ts_trace_path)
except Exception as e:
    export_patch_status["torchscript_trace"]["error"] = str(e)
    print("TorchScript TRACE export failed:", e)

# ------------------------------------------------------------
# 2) ONNX STATIC BATCH=1
#    This avoids the bad dynamic reshape issue from the prior export.
# ------------------------------------------------------------
try:
    onnx_static_path = PATCH_EXPORT_DIR / "camelot_ids_v2_publication_static_b1.onnx"
    dummy = torch.randn(1, num_features, dtype=torch.float32)

    torch.onnx.export(
        wrapper_cpu,
        dummy,
        str(onnx_static_path),
        input_names=["input"],
        output_names=["logits_fine", "logits_coarse", "logits_bin"],
        opset_version=18,
        dynamic_axes=None,   # <-- important: static batch export
        do_constant_folding=True,
    )

    export_patch_status["onnx_static_batch1"]["ok"] = True
    export_patch_status["onnx_static_batch1"]["path"] = str(onnx_static_path)
    print("Static ONNX export OK:", onnx_static_path)

    # validate ONNX on batch size 1
    try:
        import onnxruntime as ort

        sess = ort.InferenceSession(str(onnx_static_path), providers=["CPUExecutionProvider"])

        x_small = X_test[:1].astype(np.float32, copy=False)
        with torch.inference_mode():
            pt_out = wrapper_cpu(torch.from_numpy(x_small))
            pt_f = pt_out[0].numpy()
            pt_c = pt_out[1].numpy()
            pt_b = pt_out[2].numpy()

        ort_out = sess.run(None, {"input": x_small})

        ok_f = np.allclose(pt_f, ort_out[0], atol=1e-4, rtol=1e-4)
        ok_c = np.allclose(pt_c, ort_out[1], atol=1e-4, rtol=1e-4)
        ok_b = np.allclose(pt_b, ort_out[2], atol=1e-4, rtol=1e-4)

        export_patch_status["onnx_static_batch1"]["validated"] = bool(ok_f and ok_c and ok_b)
        export_patch_status["onnx_static_batch1"]["validation_allclose"] = {
            "fine": bool(ok_f),
            "coarse": bool(ok_c),
            "bin": bool(ok_b),
        }
        print("Static ONNX validation:", export_patch_status["onnx_static_batch1"]["validation_allclose"])

    except Exception as e:
        export_patch_status["onnx_static_batch1"]["validated"] = False
        export_patch_status["onnx_static_batch1"]["validation_error"] = str(e)
        print("Static ONNX validation failed:", e)

except Exception as e:
    export_patch_status["onnx_static_batch1"]["error"] = str(e)
    print("Static ONNX export failed:", e)

# ------------------------------------------------------------
# 3) Write a static-batch ONNX inference CLI
#    It runs one row at a time, so it is slower, but robust.
# ------------------------------------------------------------
static_cli = r'''
import argparse
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import onnxruntime as ort

def sanitize_frame(df: pd.DataFrame, clip_inf: float = 1e9) -> pd.DataFrame:
    df = df.replace({"Infinity": np.inf, "inf": np.inf, "+inf": np.inf, "-inf": -np.inf,
                     "NaN": np.nan, "nan": np.nan, "": np.nan})
    df = df.replace([np.inf, -np.inf], np.nan)
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        df[num_cols] = df[num_cols].clip(-clip_inf, clip_inf)
    return df

def transform_df(df_in: pd.DataFrame, feature_cols, meta_obj):
    cfg = meta_obj["cfg"]
    X = sanitize_frame(df_in[feature_cols].copy(), cfg["CLIP_INF"])
    for c in meta_obj["meta"]["numeric_like_obj"]:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")
    num_cols = meta_obj["meta"]["num_cols"]
    X = X[num_cols]
    Xn = meta_obj["num_imputer"].transform(X).astype(np.float32)
    Xn = meta_obj["scaler"].transform(Xn).astype(np.float32)
    Xn = np.clip(Xn, -cfg["POST_SCALE_CLIP"], cfg["POST_SCALE_CLIP"]).astype(np.float32)
    Xn = np.nan_to_num(Xn, nan=0.0, posinf=cfg["POST_SCALE_CLIP"], neginf=-cfg["POST_SCALE_CLIP"]).astype(np.float32)
    return Xn

def softmax_np(x):
    x = x - x.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / np.clip(ex.sum(axis=1, keepdims=True), 1e-12, None)

def raps_predict_set(probs_f, probs_c, raps_calib, alpha):
    k_reg = int(raps_calib["k_reg"])
    lam = float(raps_calib["lambda"])
    q_by = np.array(raps_calib["q_by_pred_coarse"][str(alpha)], dtype=np.float32)
    g_hat = probs_c.argmax(axis=1)
    q_row = q_by[g_hat]

    idx_sorted = np.argsort(-probs_f, axis=1)
    probs_sorted = np.take_along_axis(probs_f, idx_sorted, axis=1)
    cumsum = np.cumsum(probs_sorted, axis=1)
    C = probs_f.shape[1]
    ranks = np.arange(1, C + 1, dtype=np.float32)[None, :]
    reg = lam * np.maximum(ranks - float(k_reg), 0.0)
    score_k = cumsum + reg
    ok = score_k <= q_row[:, None]
    k_star = np.maximum(ok.sum(axis=1).astype(np.int64), 1)

    sets = []
    for i in range(len(probs_f)):
        sets.append(idx_sorted[i, :k_star[i]])
    return sets, k_star

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--meta_joblib", type=str, required=True)
    ap.add_argument("--onnx", type=str, required=True)
    ap.add_argument("--input_csv", type=str, required=True)
    ap.add_argument("--out_csv", type=str, default="preds_publication_static_onnx.csv")
    ap.add_argument("--alpha", type=float, default=0.10)
    args = ap.parse_args()

    meta = joblib.load(args.meta_joblib)
    feature_cols = meta["FEATURE_COLS"]
    fine_names = meta["fine_class_names"]
    coarse_names = meta["coarse_names"]
    temp = meta.get("temperature", {"fine": 1.0, "coarse": 1.0, "bin": 1.0})
    raps_calib = meta.get("raps_calib_publication", meta.get("raps_calib", None))
    if raps_calib is None:
        raise RuntimeError("No RAPS calibration found in meta joblib.")

    df = pd.read_csv(args.input_csv, low_memory=False)
    X = transform_df(df, feature_cols, meta)

    sess = ort.InferenceSession(args.onnx, providers=["CPUExecutionProvider"])

    logits_f_list, logits_c_list, logits_b_list = [], [], []
    for i in range(len(X)):
        xi = X[i:i+1].astype(np.float32, copy=False)
        lf, lc, lb = sess.run(None, {"input": xi})
        logits_f_list.append(lf)
        logits_c_list.append(lc)
        logits_b_list.append(lb)

    logits_f = np.concatenate(logits_f_list, axis=0)
    logits_c = np.concatenate(logits_c_list, axis=0)
    logits_b = np.concatenate(logits_b_list, axis=0)

    pf = softmax_np(logits_f / max(float(temp["fine"]), 1e-6))
    pc = softmax_np(logits_c / max(float(temp["coarse"]), 1e-6))
    pb = softmax_np(logits_b / max(float(temp["bin"]), 1e-6))

    top_f = pf.argmax(axis=1)
    top_conf = pf[np.arange(len(pf)), top_f]
    top_c = pc.argmax(axis=1)
    top_b = pb.argmax(axis=1)

    sets, k_star = raps_predict_set(pf, pc, raps_calib, float(args.alpha))
    set_str = [",".join([fine_names[j] for j in s]) for s in sets]

    out = pd.DataFrame({
        "top1_fine": [fine_names[i] for i in top_f],
        "top1_conf": top_conf,
        "pred_coarse": [coarse_names[i] for i in top_c],
        "pred_bin": ["Malicious" if i == 1 else "Benign" for i in top_b],
        "raps_set_size": k_star,
        "raps_set": set_str,
    })
    out.to_csv(args.out_csv, index=False)
    print("Saved:", args.out_csv)

if __name__ == "__main__":
    main()
'''
(PATCH_EXPORT_DIR / "camelot_infer_publication_static_onnx.py").write_text(static_cli, encoding="utf-8")

# ------------------------------------------------------------
# 4) Save patch status
# ------------------------------------------------------------
with open(PATCH_EXPORT_DIR / "export_patch_status.json", "w", encoding="utf-8") as f:
    json.dump(export_patch_status, f, indent=2)

print("\nSaved:", PATCH_EXPORT_DIR / "export_patch_status.json")
print("Export patch complete.")

NameError: name 'PUB_EXPORT_DIR' is not defined

In [7]:
# ============================================================
# COMPATIBILITY PATCH FOR FAST EXTRA PUBLICATION BLOCK
# Run this BEFORE the fast extra publication block.
# ============================================================

import os
import random
import numpy as np
import torch
import torch.nn.functional as F

# ---------------------------
# Seed helper
# ---------------------------
def set_seed(seed: int = 42, deterministic: bool = False):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = bool(deterministic)
        torch.backends.cudnn.benchmark = not bool(deterministic)

# ---------------------------
# Hierarchical consistency loss
# ---------------------------
if "fine_to_coarse" not in globals():
    raise NameError("fine_to_coarse is not defined. Run the bootstrap/main context first.")

if "NUM_COARSE" not in globals():
    raise NameError("NUM_COARSE is not defined. Run the bootstrap/main context first.")

if "benign_fine_id" not in globals():
    raise NameError("benign_fine_id is not defined. Run the bootstrap/main context first.")

fine_to_coarse_t = torch.tensor(fine_to_coarse, device=DEVICE, dtype=torch.long)

def hierarchical_consistency_loss(logits_fine, logits_coarse, logits_bin):
    p_f = torch.softmax(logits_fine, dim=1)
    p_c = torch.softmax(logits_coarse, dim=1)
    p_b = torch.softmax(logits_bin, dim=1)

    B = p_f.size(0)
    idx = fine_to_coarse_t.unsqueeze(0).expand(B, -1)
    p_from_f = torch.zeros(B, NUM_COARSE, device=p_f.device)
    p_from_f.scatter_add_(1, idx, p_f)
    cons_c = F.kl_div(torch.log(p_c + 1e-8), p_from_f, reduction="batchmean")

    p_from_f_bin = torch.stack([p_f[:, benign_fine_id], 1.0 - p_f[:, benign_fine_id]], dim=1)
    cons_b = F.kl_div(torch.log(p_b + 1e-8), p_from_f_bin, reduction="batchmean")
    return cons_c, cons_b

print("Compatibility patch loaded: set_seed + hierarchical_consistency_loss")

Compatibility patch loaded: set_seed + hierarchical_consistency_loss


In [8]:
# ============================================================
# FAST EXTRA PUBLICATION RESULTS (SHORT-RUN VERSION)
# Run after the main notebook.
# This is designed to avoid multi-day runs.
# ============================================================

import os
import gc
import json
import math
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef,
    cohen_kappa_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

warnings.filterwarnings("ignore")

# ---------------------------
# Fast controls
# ---------------------------
FAST_PUB_DIR = RUN_DIR / "fast_extra_publication"
FAST_PUB_DIR.mkdir(parents=True, exist_ok=True)

FAST_CFG = {
    # bounded subset sizes for fast publication extras
    "TRAIN_CAP": 250_000,
    "VAL_CAP": 60_000,
    "TEST_CAP": 100_000,

    # repeatability
    "REPEAT_SEEDS": [11, 42, 123],
    "QUICK_EPOCHS": 8,
    "QUICK_PATIENCE": 3,
    "QUICK_BATCH_SIZE": 2048,
    "QUICK_LR": 1e-3,
    "QUICK_WD": 5e-3,
    "QUICK_GRAD_CLIP": 1.0,

    # baseline caps
    "BASELINE_TRAIN_CAP": 200_000,
    "BASELINE_TEST_CAP": 100_000,

    # toggles
    "RUN_REPEATABILITY": True,
    "RUN_BASELINES": True,
    "RUN_MULTITASK_ABLATION": True,
    "ABLATION_SEEDS": [42],      # keep to 1 seed for fast run
    "RUN_EXTERNAL": False,       # set True only if you have a second dataset
    "EXT_DATA_ROOT": None,       # set folder path if available
    "EXT_MAX_FILES": 25,

    # conformal default
    "RAPS_ALPHA": 0.10,
}

print("FAST publication dir:", FAST_PUB_DIR)
print(json.dumps(FAST_CFG, indent=2))

# ---------------------------
# Helpers
# ---------------------------
def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def save_df(df: pd.DataFrame, stem: Path, index=False):
    stem.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(str(stem.with_suffix(".csv")), index=index)
    try:
        df.to_latex(str(stem.with_suffix(".tex")), index=index, float_format="%.4f")
    except Exception:
        pass

def stratified_cap_indices(y: np.ndarray, cap: int, seed: int = 42) -> np.ndarray:
    if len(y) <= cap:
        return np.arange(len(y))
    rng = np.random.default_rng(seed)
    idxs = []
    classes, counts = np.unique(y, return_counts=True)
    frac = cap / len(y)
    for c, cnt in zip(classes, counts):
        c_idx = np.where(y == c)[0]
        take = max(1, int(round(cnt * frac)))
        take = min(take, len(c_idx))
        pick = rng.choice(c_idx, size=take, replace=False)
        idxs.append(pick)
    idxs = np.concatenate(idxs)
    if len(idxs) > cap:
        idxs = rng.choice(idxs, size=cap, replace=False)
    return np.sort(idxs)

def subset_arrays(X, yf, yc, yb, cap, seed):
    idx = stratified_cap_indices(yf, cap, seed)
    return X[idx], yf[idx], yc[idx], yb[idx], idx

def quick_metrics_mc(y_true: np.ndarray, probs: np.ndarray):
    y_pred = probs.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
    }

def quick_metrics_bin(y_true: np.ndarray, probs_2: np.ndarray):
    y_pred = probs_2.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "f1_malicious": float(f1_score(y_true, y_pred, average="binary", pos_label=1)),
        "precision_malicious": float(precision_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0)),
        "recall_malicious": float(recall_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
    }

def cb_weights_from_counts(y: np.ndarray, n_classes: int):
    counts = np.bincount(y, minlength=n_classes).astype(np.float64)
    w = counts.sum() / np.clip(counts, 1.0, None)
    w = w / np.mean(w)
    return torch.tensor(w, dtype=torch.float32)

def build_quick_loaders(seed=42):
    Xtr, ytr_f, ytr_c, ytr_b, _ = subset_arrays(X_train, y_train_f, y_train_c, y_train_b, FAST_CFG["TRAIN_CAP"], seed)
    Xva, yva_f, yva_c, yva_b, _ = subset_arrays(X_val,   y_val_f,   y_val_c,   y_val_b,   FAST_CFG["VAL_CAP"], seed)
    Xte, yte_f, yte_c, yte_b, _ = subset_arrays(X_test,  y_test_f,  y_test_c,  y_test_b,  FAST_CFG["TEST_CAP"], seed)

    ds_tr = FlowDataset(Xtr, ytr_f, ytr_c, ytr_b)
    ds_va = FlowDataset(Xva, yva_f, yva_c, yva_b)
    ds_te = FlowDataset(Xte, yte_f, yte_c, yte_b)

    loader_kwargs_local = dict(
        batch_size=FAST_CFG["QUICK_BATCH_SIZE"],
        num_workers=0,
        pin_memory=(DEVICE == "cuda"),
        persistent_workers=False,
    )

    tr_loader = DataLoader(ds_tr, shuffle=True, **loader_kwargs_local)
    va_loader = DataLoader(ds_va, shuffle=False, **loader_kwargs_local)
    te_loader = DataLoader(ds_te, shuffle=False, **loader_kwargs_local)

    return {
        "train": (Xtr, ytr_f, ytr_c, ytr_b, tr_loader),
        "val":   (Xva, yva_f, yva_c, yva_b, va_loader),
        "test":  (Xte, yte_f, yte_c, yte_b, te_loader),
    }

def build_fresh_model_local():
    m = CamelotIDSv2(
        n_features=num_features,
        n_fine=NUM_FINE,
        n_coarse=NUM_COARSE,
        group_idxs=GROUP_IDXS,
        cfg=cfg
    ).to(DEVICE)
    return m

@torch.inference_mode()
def eval_model_on_loader(model_local, loader):
    model_local.eval()
    out = predict_logits(model_local, loader)
    pf = softmax_np(out["fine"])
    pc = softmax_np(out["coarse"])
    pb = softmax_np(out["bin"])
    return {
        "fine": quick_metrics_mc(out["yf"], pf),
        "coarse": quick_metrics_mc(out["yc"], pc),
        "binary": quick_metrics_bin(out["yb"], pb),
        "raw": out,
        "pf": pf,
        "pc": pc,
        "pb": pb,
    }

def train_quick_model(seed=42, use_multitask=True):
    set_seed(seed, deterministic=False)
    bundles = build_quick_loaders(seed)

    Xtr, ytr_f, ytr_c, ytr_b, tr_loader = bundles["train"]
    Xva, yva_f, yva_c, yva_b, va_loader = bundles["val"]
    Xte, yte_f, yte_c, yte_b, te_loader = bundles["test"]

    model_local = build_fresh_model_local()
    opt = torch.optim.AdamW(model_local.parameters(), lr=FAST_CFG["QUICK_LR"], weight_decay=FAST_CFG["QUICK_WD"])
    use_amp_local = (cfg.USE_AMP and DEVICE == "cuda")
    scaler_local = torch.amp.GradScaler("cuda", enabled=use_amp_local) if DEVICE == "cuda" else None

    w_f = cb_weights_from_counts(ytr_f, NUM_FINE).to(DEVICE)
    w_c = cb_weights_from_counts(ytr_c, NUM_COARSE).to(DEVICE)
    w_b = cb_weights_from_counts(ytr_b, 2).to(DEVICE)

    fine_loss = nn.CrossEntropyLoss(weight=w_f).to(DEVICE)
    coarse_loss = nn.CrossEntropyLoss(weight=w_c).to(DEVICE)
    bin_loss = nn.CrossEntropyLoss(weight=w_b).to(DEVICE)

    best_state = None
    best_val = -1.0
    bad = 0
    hist_rows = []

    for epoch in range(1, FAST_CFG["QUICK_EPOCHS"] + 1):
        model_local.train()
        total_loss = 0.0
        n_seen = 0

        for xb, yf, yc, yb in tr_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yf = yf.to(DEVICE, non_blocking=True)
            yc = yc.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", enabled=use_amp_local):
                out = model_local(xb)
                lf = fine_loss(out["logits_fine"], yf)

                if use_multitask:
                    lc = coarse_loss(out["logits_coarse"], yc)
                    lb = bin_loss(out["logits_bin"], yb)
                    cons_c, cons_b = hierarchical_consistency_loss(
                        out["logits_fine"], out["logits_coarse"], out["logits_bin"]
                    )
                    loss = (
                        lf
                        + cfg.W_COARSE * lc
                        + cfg.W_BIN * lb
                        + cfg.W_CONS_COARSE * cons_c
                        + cfg.W_CONS_BIN * cons_b
                    )
                else:
                    loss = lf

            if use_amp_local:
                scaler_local.scale(loss).backward()
                scaler_local.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model_local.parameters(), FAST_CFG["QUICK_GRAD_CLIP"])
                scaler_local.step(opt)
                scaler_local.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model_local.parameters(), FAST_CFG["QUICK_GRAD_CLIP"])
                opt.step()

            total_loss += float(loss.item()) * int(xb.size(0))
            n_seen += int(xb.size(0))

        train_loss = total_loss / max(n_seen, 1)

        val_eval = eval_model_on_loader(model_local, va_loader)
        val_macro_f1 = val_eval["fine"]["macro_f1"]

        hist_rows.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_fine_macro_f1": val_macro_f1,
            "val_fine_accuracy": val_eval["fine"]["accuracy"],
            "val_coarse_macro_f1": val_eval["coarse"]["macro_f1"],
            "val_bin_f1": val_eval["binary"]["f1_malicious"],
        })

        if val_macro_f1 > best_val:
            best_val = val_macro_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model_local.state_dict().items()}
            bad = 0
        else:
            bad += 1

        print(f"[seed={seed} | multitask={use_multitask}] epoch={epoch} train_loss={train_loss:.4f} val_macroF1={val_macro_f1:.4f}")

        if bad >= FAST_CFG["QUICK_PATIENCE"]:
            break

    if best_state is not None:
        model_local.load_state_dict(best_state, strict=True)

    test_eval = eval_model_on_loader(model_local, te_loader)

    history_df = pd.DataFrame(hist_rows)
    return {
        "seed": seed,
        "use_multitask": bool(use_multitask),
        "history": history_df,
        "test_fine": test_eval["fine"],
        "test_coarse": test_eval["coarse"],
        "test_binary": test_eval["binary"],
    }

# ============================================================
# A) 3-seed repeatability (fast subset version)
# ============================================================
repeat_rows = []
if FAST_CFG["RUN_REPEATABILITY"]:
    print("\n=== FAST 3-SEED REPEATABILITY (subset) ===")
    repeat_dir = FAST_PUB_DIR / "repeatability"
    repeat_dir.mkdir(parents=True, exist_ok=True)

    for seed in FAST_CFG["REPEAT_SEEDS"]:
        res = train_quick_model(seed=seed, use_multitask=True)

        res["history"].to_csv(repeat_dir / f"history_seed_{seed}.csv", index=False)

        repeat_rows.append({
            "seed": seed,
            "fine_accuracy": res["test_fine"]["accuracy"],
            "fine_macro_f1": res["test_fine"]["macro_f1"],
            "coarse_accuracy": res["test_coarse"]["accuracy"],
            "coarse_macro_f1": res["test_coarse"]["macro_f1"],
            "bin_f1_malicious": res["test_binary"]["f1_malicious"],
        })

        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    repeat_df = pd.DataFrame(repeat_rows)
    repeat_summary = {
        "fine_accuracy_mean": float(repeat_df["fine_accuracy"].mean()),
        "fine_accuracy_std": float(repeat_df["fine_accuracy"].std(ddof=1)) if len(repeat_df) > 1 else 0.0,
        "fine_macro_f1_mean": float(repeat_df["fine_macro_f1"].mean()),
        "fine_macro_f1_std": float(repeat_df["fine_macro_f1"].std(ddof=1)) if len(repeat_df) > 1 else 0.0,
        "coarse_accuracy_mean": float(repeat_df["coarse_accuracy"].mean()),
        "coarse_accuracy_std": float(repeat_df["coarse_accuracy"].std(ddof=1)) if len(repeat_df) > 1 else 0.0,
        "coarse_macro_f1_mean": float(repeat_df["coarse_macro_f1"].mean()),
        "coarse_macro_f1_std": float(repeat_df["coarse_macro_f1"].std(ddof=1)) if len(repeat_df) > 1 else 0.0,
        "bin_f1_mean": float(repeat_df["bin_f1_malicious"].mean()),
        "bin_f1_std": float(repeat_df["bin_f1_malicious"].std(ddof=1)) if len(repeat_df) > 1 else 0.0,
        "note": "Fast repeatability on capped stratified subset, not full-data reruns.",
    }

    save_df(repeat_df, repeat_dir / "repeatability_results", index=False)
    save_json(repeat_dir / "repeatability_summary.json", repeat_summary)
    print("Repeatability summary:", repeat_summary)
else:
    repeat_df = None
    repeat_summary = None

# ============================================================
# B) Fast baseline comparisons
# ============================================================
baseline_rows = []
if FAST_CFG["RUN_BASELINES"]:
    print("\n=== FAST BASELINE COMPARISONS ===")
    base_dir = FAST_PUB_DIR / "baselines"
    base_dir.mkdir(parents=True, exist_ok=True)

    Xtr_b, ytr_f_b, ytr_c_b, ytr_b_b, _ = subset_arrays(
        X_train, y_train_f, y_train_c, y_train_b,
        FAST_CFG["BASELINE_TRAIN_CAP"], cfg.SEED
    )
    Xte_b, yte_f_b, yte_c_b, yte_b_b, _ = subset_arrays(
        X_test, y_test_f, y_test_c, y_test_b,
        FAST_CFG["BASELINE_TEST_CAP"], cfg.SEED
    )

    def eval_baseline_preds(model_name, y_pred_f):
        y_pred_f = np.asarray(y_pred_f).astype(np.int64)
        y_pred_c = fine_to_coarse[y_pred_f]
        y_pred_b = (y_pred_f != benign_fine_id).astype(np.int64)

        return {
            "model": model_name,
            "fine_accuracy": float(accuracy_score(yte_f_b, y_pred_f)),
            "fine_macro_f1": float(f1_score(yte_f_b, y_pred_f, average="macro")),
            "fine_weighted_f1": float(f1_score(yte_f_b, y_pred_f, average="weighted")),
            "coarse_accuracy": float(accuracy_score(yte_c_b, y_pred_c)),
            "coarse_macro_f1": float(f1_score(yte_c_b, y_pred_c, average="macro")),
            "bin_f1_malicious": float(f1_score(yte_b_b, y_pred_b, average="binary", pos_label=1)),
        }

    # 1) Logistic Regression
    try:
        lr = LogisticRegression(
            max_iter=120,
            solver="saga",
            multi_class="multinomial",
            n_jobs=-1,
            class_weight="balanced",
            verbose=0,
        )
        lr.fit(Xtr_b, ytr_f_b)
        pred = lr.predict(Xte_b)
        baseline_rows.append(eval_baseline_preds("LogisticRegression", pred))
        print("LogisticRegression done")
    except Exception as e:
        baseline_rows.append({"model": "LogisticRegression", "error": str(e)})
        print("LogisticRegression failed:", e)

    # 2) HistGradientBoosting
    try:
        hgb = HistGradientBoostingClassifier(
            max_iter=220,
            learning_rate=0.08,
            max_depth=10,
            random_state=cfg.SEED,
        )
        hgb.fit(Xtr_b, ytr_f_b)
        pred = hgb.predict(Xte_b)
        baseline_rows.append(eval_baseline_preds("HistGradientBoosting", pred))
        print("HistGradientBoosting done")
    except Exception as e:
        baseline_rows.append({"model": "HistGradientBoosting", "error": str(e)})
        print("HistGradientBoosting failed:", e)

    # 3) XGBoost
    try:
        import xgboost as xgb
        xgbm = xgb.XGBClassifier(
            n_estimators=500,
            max_depth=8,
            learning_rate=0.08,
            subsample=0.85,
            colsample_bytree=0.85,
            tree_method="hist",
            n_jobs=-1,
            objective="multi:softmax",
            num_class=NUM_FINE,
            random_state=cfg.SEED,
        )
        xgbm.fit(Xtr_b, ytr_f_b)
        pred = xgbm.predict(Xte_b)
        baseline_rows.append(eval_baseline_preds("XGBoost", pred))
        print("XGBoost done")
    except Exception as e:
        baseline_rows.append({"model": "XGBoost", "error": str(e)})
        print("XGBoost failed:", e)

    # 4) CatBoost
    try:
        from catboost import CatBoostClassifier
        cb = CatBoostClassifier(
            loss_function="MultiClass",
            iterations=500,
            depth=8,
            learning_rate=0.08,
            auto_class_weights="Balanced",
            verbose=False,
            task_type="GPU" if DEVICE == "cuda" else "CPU",
        )
        cb.fit(Xtr_b, ytr_f_b)
        pred = cb.predict(Xte_b).reshape(-1).astype(int)
        baseline_rows.append(eval_baseline_preds("CatBoost", pred))
        print("CatBoost done")
    except Exception as e:
        baseline_rows.append({"model": "CatBoost", "error": str(e)})
        print("CatBoost failed:", e)

    baseline_df = pd.DataFrame(baseline_rows)
    save_df(baseline_df, base_dir / "baseline_comparison_fast", index=False)
else:
    baseline_df = None

# ============================================================
# C) Fast ablation study
#    1. zero-cost inference ablations (already-trained model)
#    2. quick multitask on/off subset retraining
# ============================================================
print("\n=== FAST ABLATION STUDY ===")
ablation_dir = FAST_PUB_DIR / "ablations"
ablation_dir.mkdir(parents=True, exist_ok=True)

ablation_rows = []

# zero-cost calibration ablation from existing logits
if "test_out" in globals():
    probs_f_raw = softmax_np(test_out["fine"])
    probs_c_raw = softmax_np(test_out["coarse"])
    probs_b_raw = softmax_np(test_out["bin"])

    probs_f_temp = softmax_np(test_out["fine"] / max(float(temp_f), 1e-6))
    probs_c_temp = softmax_np(test_out["coarse"] / max(float(temp_c), 1e-6))
    probs_b_temp = softmax_np(test_out["bin"] / max(float(temp_b), 1e-6))

    ablation_rows.append({
        "component": "calibration_off_top1",
        "fine_accuracy": accuracy_score(test_out["yf"], probs_f_raw.argmax(axis=1)),
        "fine_macro_f1": f1_score(test_out["yf"], probs_f_raw.argmax(axis=1), average="macro"),
        "coarse_accuracy": accuracy_score(test_out["yc"], probs_c_raw.argmax(axis=1)),
        "coarse_macro_f1": f1_score(test_out["yc"], probs_c_raw.argmax(axis=1), average="macro"),
        "bin_f1_malicious": f1_score(test_out["yb"], probs_b_raw.argmax(axis=1), average="binary", pos_label=1),
    })

    ablation_rows.append({
        "component": "calibration_on_top1",
        "fine_accuracy": accuracy_score(test_out["yf"], probs_f_temp.argmax(axis=1)),
        "fine_macro_f1": f1_score(test_out["yf"], probs_f_temp.argmax(axis=1), average="macro"),
        "coarse_accuracy": accuracy_score(test_out["yc"], probs_c_temp.argmax(axis=1)),
        "coarse_macro_f1": f1_score(test_out["yc"], probs_c_temp.argmax(axis=1), average="macro"),
        "bin_f1_malicious": f1_score(test_out["yb"], probs_b_temp.argmax(axis=1), average="binary", pos_label=1),
    })

# zero-cost conformal ablation
if "raps_calib" in globals() and "test_probs_f_cal" in globals() and "test_probs_c_cal" in globals():
    alpha0 = FAST_CFG["RAPS_ALPHA"]
    if str(alpha0) in raps_calib["q_by_pred_coarse"]:
        q_by = np.array(raps_calib["q_by_pred_coarse"][str(alpha0)], dtype=np.float32)
        g_hat = test_probs_c_cal.argmax(axis=1)
        q_row = q_by[g_hat]
        k_star = raps_predict_k(test_probs_f_cal, q_row, cfg.RAPS_KREG, cfg.RAPS_LAMBDA)
        single = (k_star == 1)
        y_pred = test_probs_f_cal.argmax(axis=1)

        if single.any():
            sing_acc = float((y_pred[single] == test_out["yf"][single]).mean())
            sing_mf1 = float(f1_score(test_out["yf"][single], y_pred[single], average="macro"))
        else:
            sing_acc = float("nan")
            sing_mf1 = float("nan")

        ablation_rows.append({
            "component": f"conformal_RAPS_alpha_{alpha0}",
            "singleton_coverage": float(single.mean()),
            "singleton_risk_1_minus_acc": float(1.0 - sing_acc),
            "singleton_macro_f1": sing_mf1,
            "avg_set_size": float(k_star.mean()),
        })

# quick multitask ablation
mt_rows = []
if FAST_CFG["RUN_MULTITASK_ABLATION"]:
    print("Running quick multitask ablation on capped subset...")
    for seed in FAST_CFG["ABLATION_SEEDS"]:
        res_mt_on = train_quick_model(seed=seed, use_multitask=True)
        res_mt_off = train_quick_model(seed=seed, use_multitask=False)

        mt_rows.append({
            "seed": seed,
            "variant": "multitask_on",
            "fine_accuracy": res_mt_on["test_fine"]["accuracy"],
            "fine_macro_f1": res_mt_on["test_fine"]["macro_f1"],
            "coarse_macro_f1": res_mt_on["test_coarse"]["macro_f1"],
            "bin_f1_malicious": res_mt_on["test_binary"]["f1_malicious"],
        })
        mt_rows.append({
            "seed": seed,
            "variant": "multitask_off",
            "fine_accuracy": res_mt_off["test_fine"]["accuracy"],
            "fine_macro_f1": res_mt_off["test_fine"]["macro_f1"],
            "coarse_macro_f1": res_mt_off["test_coarse"]["macro_f1"],
            "bin_f1_malicious": res_mt_off["test_binary"]["f1_malicious"],
        })

        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

mt_df = pd.DataFrame(mt_rows) if len(mt_rows) > 0 else pd.DataFrame()
ablation_df = pd.DataFrame(ablation_rows) if len(ablation_rows) > 0 else pd.DataFrame()

if len(ablation_df) > 0:
    save_df(ablation_df, ablation_dir / "inference_ablation_fast", index=False)

if len(mt_df) > 0:
    save_df(mt_df, ablation_dir / "multitask_ablation_fast", index=False)

# ============================================================
# D) External / cross-dataset validation
# ============================================================
external_summary = {"ran": False, "reason": "not_requested"}

def build_eval_model_for_external():
    # use current in-memory model from main run
    m = build_fresh_model_local()
    m.load_state_dict(model.state_dict(), strict=True)
    m.eval()
    return m

if FAST_CFG["RUN_EXTERNAL"] and FAST_CFG["EXT_DATA_ROOT"] is not None:
    print("\n=== EXTERNAL VALIDATION ===")
    ext_dir = FAST_PUB_DIR / "external_validation"
    ext_dir.mkdir(parents=True, exist_ok=True)

    try:
        ext_root = Path(FAST_CFG["EXT_DATA_ROOT"]).expanduser().resolve()
        ext_files = discover_csv_files(ext_root, FAST_CFG["EXT_MAX_FILES"])
        ext_dfs = []

        for p in ext_files:
            d = read_one_csv(p, cfg)
            d["__src_file"] = p.name
            ext_dfs.append(d)

        ext_df = pd.concat(ext_dfs, ignore_index=True)
        del ext_dfs
        gc.collect()

        ext_df = sanitize_frame(ext_df, cfg.CLIP_INF)
        ext_df = downcast_numeric(ext_df)

        ext_label_col = detect_col(ext_df.columns.tolist(), cfg.LABEL_COL_CANDIDATES)
        ext_time_col = detect_col(ext_df.columns.tolist(), cfg.TIME_COL_CANDIDATES)
        ext_group_col = detect_col(ext_df.columns.tolist(), cfg.GROUP_COL_CANDIDATES)

        if ext_label_col is None:
            raise ValueError("Could not detect label column in external dataset.")

        ext_df[ext_label_col] = ext_df[ext_label_col].astype(str).str.strip()
        known_mask = ext_df[ext_label_col].isin(DICT_34_CLASSES.keys())
        ext_df = ext_df.loc[known_mask].copy()

        ext_df["y_fine"] = ext_df[ext_label_col].map(DICT_34_CLASSES).astype(np.int64)
        ext_df["y_coarse"] = ext_df["y_fine"].map(lambda i: int(fine_to_coarse[i])).astype(np.int64)
        ext_df["y_bin"] = (ext_df["y_fine"] != benign_fine_id).astype(np.int64)

        # keep only training-time feature columns that exist
        ext_feature_cols = [c for c in FEATURE_COLS if c in ext_df.columns]
        if len(ext_feature_cols) != len(FEATURE_COLS):
            missing = sorted(set(FEATURE_COLS) - set(ext_feature_cols))
            print("External data missing feature columns:", missing[:20], "...")

        X_ext = transform_df(ext_df, ext_feature_cols, num_imp, scaler, meta, cfg)
        y_ext_f = ext_df["y_fine"].values.astype(np.int64)
        y_ext_c = ext_df["y_coarse"].values.astype(np.int64)
        y_ext_b = ext_df["y_bin"].values.astype(np.int64)

        ds_ext = FlowDataset(X_ext, y_ext_f, y_ext_c, y_ext_b)
        ext_loader = DataLoader(
            ds_ext,
            batch_size=FAST_CFG["QUICK_BATCH_SIZE"],
            shuffle=False,
            num_workers=0,
            pin_memory=(DEVICE == "cuda"),
        )

        ext_model = build_eval_model_for_external()
        ext_out = predict_logits(ext_model, ext_loader)

        ext_pf = softmax_np(ext_out["fine"] / max(float(temp_f), 1e-6))
        ext_pc = softmax_np(ext_out["coarse"] / max(float(temp_c), 1e-6))
        ext_pb = softmax_np(ext_out["bin"] / max(float(temp_b), 1e-6))

        external_summary = {
            "ran": True,
            "data_root": str(ext_root),
            "num_samples": int(len(y_ext_f)),
            "fine": quick_metrics_mc(ext_out["yf"], ext_pf),
            "coarse": quick_metrics_mc(ext_out["yc"], ext_pc),
            "binary": quick_metrics_bin(ext_out["yb"], ext_pb),
            "note": "External validation used training-time preprocessor without refit.",
        }

        save_json(ext_dir / "external_validation_summary.json", external_summary)
        print("External validation summary:", external_summary)

    except Exception as e:
        external_summary = {"ran": False, "reason": str(e)}
        print("External validation skipped/failed:", e)

# ============================================================
# E) Final combined summary
# ============================================================
final_fast_summary = {
    "config": FAST_CFG,
    "repeatability_summary": repeat_summary,
    "baseline_results_available": None if baseline_df is None else baseline_df.to_dict(orient="records"),
    "inference_ablation_available": None if len(ablation_df) == 0 else ablation_df.to_dict(orient="records"),
    "multitask_ablation_available": None if len(mt_df) == 0 else mt_df.to_dict(orient="records"),
    "external_validation": external_summary,
    "notes": [
        "This package is the fast publication version.",
        "3-seed repeatability and multitask ablation are done on capped stratified subsets, not full-data reruns.",
        "Calibration and conformal ablations reuse cached logits and are effectively zero-cost.",
        "External validation only runs if a second dataset folder is provided.",
    ],
}

save_json(FAST_PUB_DIR / "fast_extra_publication_summary.json", final_fast_summary)

md_text = f"""
# Fast Extra Publication Results

## What this adds
- 3-seed repeatability (fast subset)
- 2–4 strong tabular baselines
- calibration/conformal ablations
- quick multitask ablation
- optional external validation

## Important note
These are **fast publication extras** designed to avoid multi-day runs.
The repeatability and multitask ablation results are on capped stratified subsets.

## Files
- Summary JSON: `{FAST_PUB_DIR / "fast_extra_publication_summary.json"}`
- Repeatability dir: `{FAST_PUB_DIR / "repeatability"}`
- Baselines dir: `{FAST_PUB_DIR / "baselines"}`
- Ablations dir: `{FAST_PUB_DIR / "ablations"}`
- External validation dir: `{FAST_PUB_DIR / "external_validation"}`
"""
(FAST_PUB_DIR / "README_fast_extra_publication.md").write_text(md_text, encoding="utf-8")

print("\nSaved:", FAST_PUB_DIR / "fast_extra_publication_summary.json")
print("All fast extra publication outputs are in:", FAST_PUB_DIR)

FAST publication dir: results_camelot_ids_v2_pc\fast_extra_publication
{
  "TRAIN_CAP": 250000,
  "VAL_CAP": 60000,
  "TEST_CAP": 100000,
  "REPEAT_SEEDS": [
    11,
    42,
    123
  ],
  "QUICK_EPOCHS": 8,
  "QUICK_PATIENCE": 3,
  "QUICK_BATCH_SIZE": 2048,
  "QUICK_LR": 0.001,
  "QUICK_WD": 0.005,
  "QUICK_GRAD_CLIP": 1.0,
  "BASELINE_TRAIN_CAP": 200000,
  "BASELINE_TEST_CAP": 100000,
  "RUN_REPEATABILITY": true,
  "RUN_BASELINES": true,
  "RUN_MULTITASK_ABLATION": true,
  "ABLATION_SEEDS": [
    42
  ],
  "RUN_EXTERNAL": false,
  "EXT_DATA_ROOT": null,
  "EXT_MAX_FILES": 25,
  "RAPS_ALPHA": 0.1
}

=== FAST 3-SEED REPEATABILITY (subset) ===
[seed=11 | multitask=True] epoch=1 train_loss=4.4975 val_macroF1=0.0014
[seed=11 | multitask=True] epoch=2 train_loss=4.4333 val_macroF1=0.0000
[seed=11 | multitask=True] epoch=3 train_loss=4.4218 val_macroF1=0.0000
[seed=11 | multitask=True] epoch=4 train_loss=4.3809 val_macroF1=0.0001
[seed=42 | multitask=True] epoch=1 train_loss=4.5045 val_macr

In [9]:
print("RUN_DIR exists:", "RUN_DIR" in globals())
print("X_train exists:", "X_train" in globals())
print("FlowDataset exists:", "FlowDataset" in globals())
print("CamelotIDSv2 exists:", "CamelotIDSv2" in globals())
print("predict_logits exists:", "predict_logits" in globals())
print("softmax_np exists:", "softmax_np" in globals())
print("set_seed exists:", "set_seed" in globals())
print("hierarchical_consistency_loss exists:", "hierarchical_consistency_loss" in globals())

RUN_DIR exists: True
X_train exists: True
FlowDataset exists: True
CamelotIDSv2 exists: True
predict_logits exists: True
softmax_np exists: True
set_seed exists: True
hierarchical_consistency_loss exists: True
